# Healthcare Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 2: ED Cost Prediction**.

**Goal**: Predict `ed_cost_next3y_usd` for each patient
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular ED cost data with a simple Random Forest regressor.


In [1]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="adsb_AJp9fgtjuTRZ4Tk0mNnYDWxG_1759039055",        # Get from your team dashboard
    team_name="clai"     # Your exact team name
)

# Iteration 1 
- 464.6911

In [2]:
import os, sys, re, math, json, warnings
from pathlib import Path
from typing import Tuple, Dict, Optional, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# -----------------------------
# Global config
# -----------------------------
SEED = 42
np.random.seed(SEED)

DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"

TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"

PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"

ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"

RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR     = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"

SAMPLE_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"  # sample reference only
OUT_SUB_PATH    = r"D:\AgentDs\agent_ds_healthcare\submission.csv"  # overwrite ok

print("Plan: patient+admissions aggregates + receipt code/spend features -> CatBoost MAE ensemble (+ log-target variant) + foldwise calibration")

# -----------------------------
# Helpers
# -----------------------------
def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def safe_read_csv(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(p)

def ensure_cat_as_str(df: pd.DataFrame, cols: List[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")

def median_fill_numeric(train_df: pd.DataFrame, test_df: pd.DataFrame, numeric_cols: List[str]) -> None:
    for c in numeric_cols:
        med = pd.to_numeric(train_df[c], errors="coerce").median()
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(med)
        test_df[c]  = pd.to_numeric(test_df[c],  errors="coerce").fillna(med)

def detect_gpu_catboost() -> bool:
    try:
        from catboost.utils import get_gpu_device_count
        return get_gpu_device_count() > 0
    except Exception:
        return False

def fill_prefix_numeric_zero(df: pd.DataFrame, prefixes: List[str], exclude: Optional[List[str]] = None) -> None:
    exclude = set(exclude or [])
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes) and (c not in exclude)]
    for c in cols:
        if c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]) or df[c].dtype == "object":
                df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

# -----------------------------
# Receipt parsing (last resort) + feature builder
# -----------------------------
def extract_text_from_pdf(path: Path) -> str:
    # Try pdfplumber, fallback to PyMuPDF. If neither works, return "".
    try:
        import pdfplumber
        with pdfplumber.open(str(path)) as pdf:
            return "\n".join((p.extract_text() or "") for p in pdf.pages)
    except Exception:
        try:
            import fitz  # PyMuPDF
            doc = fitz.open(str(path))
            text = "\n".join(page.get_text("text") for page in doc)
            doc.close()
            return text
        except Exception:
            return ""

def parse_receipt_text(text: str) -> Tuple[Dict, pd.DataFrame]:
    pid_re = re.compile(r"Patient ID:\s*(\d+)")
    zip_ins_re = re.compile(r"ZIP3:\s*([0-9A-Za-z]+)\s+Insurance:\s*([A-Za-z_]+)")
    line_re = re.compile(r"^(?P<code>\S+)\s+(?P<desc>.*?)\s+(?P<qty>\d+)\s+(?P<unit>[\d,]+\.\d{2})\s+(?P<line_total>[\d,]+\.\d{2})$")
    total_re = re.compile(r"^TOTAL\s+(?P<total>[\d,]+\.\d{2})$")

    pid = None
    zip3 = None
    insurance = None
    total = np.nan

    m = pid_re.search(text)
    if m:
        pid = int(m.group(1))
    m = zip_ins_re.search(text)
    if m:
        zip3 = standardize_zip3(m.group(1))
        insurance = m.group(2)

    rows = []
    for raw in text.splitlines():
        raw = raw.strip()
        if not raw:
            continue
        mt = total_re.match(raw)
        if mt:
            total = float(mt.group("total").replace(",", ""))
            continue
        ml = line_re.match(raw)
        if ml:
            d = ml.groupdict()
            rows.append({
                "patient_id": pid,
                "code": d["code"],
                "desc": d["desc"],
                "qty": int(d["qty"]),
                "unit": float(d["unit"].replace(",", "")),
                "line_total": float(d["line_total"].replace(",", "")),
            })

    header = {"patient_id": pid, "zip3_pdf": zip3, "insurance_pdf": insurance, "pdf_total": total}
    return header, pd.DataFrame(rows)

def parse_all_receipts_from_pdfs(pdf_dir: Path, patient_ids: List[int]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    headers, items = [], []
    missing = 0
    for pid in patient_ids:
        pdf_path = pdf_dir / f"receipt_{pid}.pdf"
        if not pdf_path.exists():
            missing += 1
            continue
        txt = extract_text_from_pdf(pdf_path)
        h, df = parse_receipt_text(txt)
        headers.append(h)
        if df is not None and len(df):
            items.append(df)
    headers_df = pd.DataFrame(headers)
    lineitems_df = pd.concat(items, ignore_index=True) if items else pd.DataFrame(columns=["patient_id","code","desc","qty","unit","line_total"])
    print(f"[receipts] Parsed PDFs (fallback): headers={headers_df.shape} lineitems={lineitems_df.shape} missing_pdfs={missing}")
    return headers_df, lineitems_df

def extract_receipt_dfs(obj) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    headers_df, lineitems_df = None, None
    if isinstance(obj, dict):
        for hk in ["headers_df","headers","header_df","header"]:
            if hk in obj and isinstance(obj[hk], pd.DataFrame):
                headers_df = obj[hk]
                break
        for lk in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if lk in obj and isinstance(obj[lk], pd.DataFrame):
                lineitems_df = obj[lk]
                break
        if lineitems_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and {"patient_id","code"}.issubset(v.columns):
                    lineitems_df = v
                    break
        if headers_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and ("patient_id" in v.columns) and (v is not lineitems_df):
                    headers_df = v
                    break
    elif isinstance(obj, (tuple, list)):
        dfs = [x for x in obj if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if {"patient_id","code"}.issubset(df.columns):
                lineitems_df = df
            elif "patient_id" in df.columns:
                headers_df = df
        if headers_df is None and len(dfs) == 1:
            headers_df = dfs[0]
    elif isinstance(obj, pd.DataFrame):
        if {"patient_id","code"}.issubset(obj.columns):
            lineitems_df = obj
        elif "patient_id" in obj.columns:
            headers_df = obj
    return headers_df, lineitems_df

def build_receipt_features(headers_df: Optional[pd.DataFrame],
                          lineitems_df: Optional[pd.DataFrame],
                          top_n_codes: int = 70) -> pd.DataFrame:
    # If we only have patient-level features already, return them
    if (lineitems_df is None) or (not isinstance(lineitems_df, pd.DataFrame)) or (len(lineitems_df) == 0):
        if headers_df is not None and isinstance(headers_df, pd.DataFrame) and ("patient_id" in headers_df.columns):
            df = headers_df.copy()
            keep_cols = ["patient_id"] + [c for c in df.columns if c != "patient_id" and (pd.api.types.is_numeric_dtype(df[c]) or c in ["pdf_total","sum_items","zip3_pdf","insurance_pdf"])]
            return df[keep_cols].copy()
        return pd.DataFrame(columns=["patient_id"])

    li = lineitems_df.copy()

    # normalize column names
    if "code" not in li.columns:
        for alt in ["cpt","cpt_code","hcpcs","proc_code"]:
            if alt in li.columns:
                li["code"] = li[alt]
                break
    if "line_total" not in li.columns:
        for alt in ["line_total_usd","total","amount","line_cost","line_total_$"]:
            if alt in li.columns:
                li["line_total"] = li[alt]
                break
    if "qty" not in li.columns:
        for alt in ["quantity","q"]:
            if alt in li.columns:
                li["qty"] = li[alt]
                break
    if "unit" not in li.columns:
        if "line_total" in li.columns and "qty" in li.columns:
            li["unit"] = pd.to_numeric(li["line_total"], errors="coerce") / pd.to_numeric(li["qty"], errors="coerce").replace(0, np.nan)
        else:
            li["unit"] = np.nan

    if "patient_id" not in li.columns or "code" not in li.columns:
        return pd.DataFrame(columns=["patient_id"])

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li["code"].astype(str).str.strip().str.upper()
    li["qty"] = pd.to_numeric(li["qty"], errors="coerce").fillna(1).astype(float)
    li["unit"] = pd.to_numeric(li["unit"], errors="coerce")
    li["line_total"] = pd.to_numeric(li["line_total"], errors="coerce").fillna(0.0).astype(float)

    # limit code dimensionality for safety
    must_codes = {
        "31500","36556","36620","92950",
        "70450","74177","84484","G0378",
        "99281","99282","99283","99284","99285",
        "99291","99292",
        "85025","87070","71045"
    }
    code_cost = li.groupby("code")["line_total"].sum().sort_values(ascending=False)
    top_cost = code_cost.head(top_n_codes).index.tolist()
    top_cnt = li["code"].value_counts().head(top_n_codes).index.tolist()
    codes = sorted(set(top_cost + top_cnt + list(must_codes)))

    # deflated price index (separates volume from price)
    med_unit_by_code = li.groupby("code")["unit"].median()
    li["unit_med_code"] = li["code"].map(med_unit_by_code).astype(float)
    li["deflated_line_total"] = li["qty"] * li["unit_med_code"]

    g = li.groupby("patient_id")
    agg = g.agg(
        pdf_total_line_cost=("line_total","sum"),
        pdf_deflated_total=("deflated_line_total","sum"),
        pdf_n_line_items=("code","size"),
        pdf_total_qty=("qty","sum"),
        pdf_n_unique_codes=("code","nunique"),
        pdf_unit_mean=("unit","mean"),
        pdf_unit_median=("unit","median"),
        pdf_line_max=("line_total","max"),
        pdf_line_mean=("line_total","mean"),
        pdf_line_std=("line_total", lambda x: float(np.std(x, ddof=0))),
        pdf_line_median=("line_total","median"),
    ).reset_index()

    agg["pdf_price_index"] = agg["pdf_total_line_cost"] / agg["pdf_deflated_total"].replace(0, np.nan)
    agg["pdf_price_index"] = agg["pdf_price_index"].replace([np.inf, -np.inf], np.nan).fillna(1.0)

    # Wide per-code features (counts + costs)
    li_sel = li[li["code"].isin(codes)].copy()
    piv_cnt  = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="size", fill_value=0)
    piv_cost = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="sum", fill_value=0.0)
    piv_cnt.columns  = [f"pdf_cnt_{c}"  for c in piv_cnt.columns.astype(str)]
    piv_cost.columns = [f"pdf_cost_{c}" for c in piv_cost.columns.astype(str)]
    wide = piv_cnt.join(piv_cost, how="outer").reset_index()

    out = agg.merge(wide, on="patient_id", how="left")
    fill_prefix_numeric_zero(out, prefixes=["pdf_"])

    # Other bucket (codes not in selected)
    sel_cost_cols = [c for c in out.columns if c.startswith("pdf_cost_") and (c.replace("pdf_cost_","") in codes)]
    sel_cnt_cols  = [c for c in out.columns if c.startswith("pdf_cnt_")  and (c.replace("pdf_cnt_","")  in codes)]
    out["pdf_cost_selected_sum"] = out[sel_cost_cols].sum(axis=1) if sel_cost_cols else 0.0
    out["pdf_cost_other"] = np.maximum(0.0, out["pdf_total_line_cost"] - out["pdf_cost_selected_sum"])
    out["pdf_cnt_selected_sum"] = out[sel_cnt_cols].sum(axis=1) if sel_cnt_cols else 0.0
    out["pdf_cnt_other"] = np.maximum(0.0, out["pdf_n_line_items"] - out["pdf_cnt_selected_sum"])

    denom = out["pdf_total_line_cost"].replace(0, np.nan)

    # Shares per code (selected only)
    for c in codes:
        col = f"pdf_cost_{c}"
        out[f"pdf_share_{c}"] = (out[col] / denom).fillna(0.0) if col in out.columns else 0.0

    # Clinical-ish category groupings
    ED_EM = {"99281","99282","99283","99284","99285"}
    CRIT = {"99291","99292"}
    LAB  = {"85025","87070","84484"}
    IMG  = {"71045","70450","74177"}
    OBS  = {"G0378"}
    SEVERE = {"31500","36556","36620","92950"}

    def sum_cols(prefix: str, code_set: set) -> pd.Series:
        cols = [f"{prefix}_{c}" for c in code_set if f"{prefix}_{c}" in out.columns]
        return out[cols].sum(axis=1) if cols else pd.Series(np.zeros(len(out)), index=out.index)

    # Counts
    out["pdf_cnt_ed_em"]       = sum_cols("pdf_cnt", ED_EM)
    out["pdf_cnt_crit_care"]   = sum_cols("pdf_cnt", CRIT)
    out["pdf_cnt_lab"]         = sum_cols("pdf_cnt", LAB)
    out["pdf_cnt_imaging"]     = sum_cols("pdf_cnt", IMG)
    out["pdf_cnt_obs"]         = sum_cols("pdf_cnt", OBS)
    out["pdf_cnt_severe_proc"] = sum_cols("pdf_cnt", SEVERE)

    # Costs
    out["pdf_cost_ed_em"]       = sum_cols("pdf_cost", ED_EM)
    out["pdf_cost_crit_care"]   = sum_cols("pdf_cost", CRIT)
    out["pdf_cost_lab"]         = sum_cols("pdf_cost", LAB)
    out["pdf_cost_imaging"]     = sum_cols("pdf_cost", IMG)
    out["pdf_cost_obs"]         = sum_cols("pdf_cost", OBS)
    out["pdf_cost_severe_proc"] = sum_cols("pdf_cost", SEVERE)

    for k in ["ed_em","crit_care","lab","imaging","obs","severe_proc"]:
        out[f"pdf_share_{k}"] = (out[f"pdf_cost_{k}"] / denom).fillna(0.0)
    out["pdf_share_other"] = (out["pdf_cost_other"] / denom).fillna(0.0)

    # High-acuity flags (from prompt)
    code_flags = {
        "has_intub_31500": "31500",
        "has_cvc_36556": "36556",
        "has_artline_36620": "36620",
        "has_cpr_92950": "92950",
        "has_ct_head_70450": "70450",
        "has_ct_abdpel_74177": "74177",
        "has_troponin_84484": "84484",
        "has_obs_G0378": "G0378",
        "has_99285": "99285",
    }
    for fname, code in code_flags.items():
        cnt_col = f"pdf_cnt_{code}"
        out[fname] = (out[cnt_col] > 0).astype(int) if cnt_col in out.columns else 0
    out["has_critical_care"] = (out["pdf_cnt_crit_care"] > 0).astype(int)

    out["severe_proc_score"] = (
        out["has_intub_31500"]
        + out["has_cvc_36556"]
        + out["has_artline_36620"]
        + out["has_cpr_92950"]
        + out["has_critical_care"]
    ).astype(int)

    # E/M level stats
    em_level_map = {"99281":1, "99282":2, "99283":3, "99284":4, "99285":5}
    em = li[li["code"].isin(ED_EM)].copy()
    if len(em):
        em["em_level"] = em["code"].map(em_level_map).astype(float)
        em_agg = em.groupby("patient_id").agg(
            pdf_em_level_mean=("em_level","mean"),
            pdf_em_level_max=("em_level","max"),
            pdf_em_lines=("em_level","size"),
            pdf_em_high_frac=("em_level", lambda x: float(np.mean(np.asarray(x) >= 4))),
        ).reset_index()
        out = out.merge(em_agg, on="patient_id", how="left")
    for c in ["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]:
        if c not in out.columns:
            out[c] = 0.0
    out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]] = out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]].fillna(0.0)

    # Cost entropy over codes
    cost_code_cols = [c for c in out.columns if c.startswith("pdf_cost_") and (c.replace("pdf_cost_","") in codes)]
    if cost_code_cols:
        mat = out[cost_code_cols].to_numpy(dtype=float)
        row_sum = mat.sum(axis=1, keepdims=True)
        p = np.divide(mat, row_sum, out=np.zeros_like(mat), where=row_sum > 0)
        ent = -(p * np.log(np.clip(p, 1e-12, 1.0))).sum(axis=1)
        ent_norm = ent / np.log(max(2, mat.shape[1]))
        out["pdf_code_cost_entropy"] = ent
        out["pdf_code_cost_entropy_norm"] = ent_norm
    else:
        out["pdf_code_cost_entropy"] = 0.0
        out["pdf_code_cost_entropy_norm"] = 0.0

    # Merge header totals if present
    if headers_df is not None and isinstance(headers_df, pd.DataFrame) and ("patient_id" in headers_df.columns):
        cols = ["patient_id"] + [c for c in ["pdf_total","zip3_pdf","insurance_pdf"] if c in headers_df.columns]
        hd = headers_df[cols].copy()
        out = out.merge(hd, on="patient_id", how="left")
        if "pdf_total" in out.columns:
            out["pdf_total_missing"] = out["pdf_total"].isna().astype(int)
            out["pdf_total"] = pd.to_numeric(out["pdf_total"], errors="coerce").fillna(0.0)
            out["pdf_total_minus_sum"] = out["pdf_total"] - out["pdf_total_line_cost"]
        if "zip3_pdf" in out.columns:
            out["zip3_pdf"] = out["zip3_pdf"].apply(standardize_zip3)
        if "insurance_pdf" in out.columns:
            out["insurance_pdf"] = out["insurance_pdf"].astype("string").fillna("Unknown")

    return out

# -----------------------------
# Admissions aggregates
# -----------------------------
def build_admissions_features(adm_tr: pd.DataFrame, adm_te: pd.DataFrame) -> pd.DataFrame:
    adm_tr = adm_tr.drop(columns=["readmit_30d"], errors="ignore")
    adm_all = pd.concat([adm_tr, adm_te], ignore_index=True)

    if "patient_id" not in adm_all.columns:
        return pd.DataFrame(columns=["patient_id"])

    adm_all["patient_id"] = pd.to_numeric(adm_all["patient_id"], errors="coerce").astype("Int64")
    adm_all = adm_all.dropna(subset=["patient_id"]).copy()
    adm_all["patient_id"] = adm_all["patient_id"].astype(int)

    num_cols = ["los_days","acuity_emergent","charlson_band","ed_visits_6m","discharge_weekday"]
    for c in num_cols:
        if c in adm_all.columns:
            adm_all[c] = pd.to_numeric(adm_all[c], errors="coerce")

    g = adm_all.groupby("patient_id")
    out = pd.DataFrame({"patient_id": g.size().index, "adm_n": g.size().values})

    for c in num_cols:
        if c in adm_all.columns:
            out[f"adm_{c}_mean"] = g[c].mean().values
            out[f"adm_{c}_max"]  = g[c].max().values
            out[f"adm_{c}_sum"]  = g[c].sum().values
            out[f"adm_{c}_std"]  = g[c].std(ddof=0).values

    if "primary_dx" in adm_all.columns:
        dx_ct = pd.crosstab(adm_all["patient_id"], adm_all["primary_dx"])
        dx_ct.columns = [f"adm_dx_cnt_{str(c)}" for c in dx_ct.columns]
        dx_ct = dx_ct.reset_index()
        out = out.merge(dx_ct, on="patient_id", how="left")

        dx_mode = g["primary_dx"].agg(lambda x: x.value_counts().index[0] if len(x) else "Unknown").reset_index()
        dx_mode.columns = ["patient_id", "adm_primary_dx_mode"]
        out = out.merge(dx_mode, on="patient_id", how="left")

    for c in out.columns:
        if c in ["patient_id", "adm_primary_dx_mode"]:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    if "adm_primary_dx_mode" in out.columns:
        out["adm_primary_dx_mode"] = out["adm_primary_dx_mode"].astype("string").fillna("Unknown")
    return out

# -----------------------------
# Load data
# -----------------------------
train = safe_read_csv(TRAIN_PATH)
test  = safe_read_csv(TEST_PATH)
patients = safe_read_csv(PATIENTS_PATH)
adm_tr = safe_read_csv(ADM_TRAIN_PATH)
adm_te = safe_read_csv(ADM_TEST_PATH)

for df in [train, test, patients, adm_tr, adm_te]:
    if "patient_id" in df.columns:
        df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

if "zip3" in patients.columns:
    patients["zip3"] = patients["zip3"].apply(standardize_zip3)

# -----------------------------
# Build admissions + receipt features
# -----------------------------
adm_feat = build_admissions_features(adm_tr, adm_te)
print(f"[admissions] patient-level features shape: {adm_feat.shape}")

receipt_feat = pd.DataFrame(columns=["patient_id"])
try:
    from joblib import load as joblib_load
except Exception:
    joblib_load = None

joblib_path = Path(RECEIPTS_JOBLIB_PATH)
pdf_dir = Path(RECEIPTS_PDF_DIR)
all_patient_ids = pd.concat([train["patient_id"], test["patient_id"]], ignore_index=True).astype(int).unique().tolist()

if joblib_path.exists() and (joblib_load is not None):
    try:
        obj = joblib_load(joblib_path)
        headers_df, lineitems_df = extract_receipt_dfs(obj)

        if isinstance(obj, pd.DataFrame) and ("patient_id" in obj.columns) and ("code" not in obj.columns) and (obj["patient_id"].nunique() == len(obj)):
            receipt_feat = obj.copy()
        else:
            receipt_feat = build_receipt_features(headers_df, lineitems_df, top_n_codes=70)

        print(f"[receipts] Loaded receipts_parsed.joblib -> receipt_feat shape: {receipt_feat.shape}")
    except Exception as e:
        print(f"[receipts] Failed loading joblib ({e}); attempting PDF parse fallback...")
        if pdf_dir.exists():
            headers_df, lineitems_df = parse_all_receipts_from_pdfs(pdf_dir, all_patient_ids)
            receipt_feat = build_receipt_features(headers_df, lineitems_df, top_n_codes=70)
        else:
            print("[receipts] PDF dir not found; skipping receipt features.")
else:
    print("[receipts] receipts_parsed.joblib not found; attempting PDF parse fallback...")
    if pdf_dir.exists():
        headers_df, lineitems_df = parse_all_receipts_from_pdfs(pdf_dir, all_patient_ids)
        receipt_feat = build_receipt_features(headers_df, lineitems_df, top_n_codes=70)
    else:
        print("[receipts] PDF dir not found; skipping receipt features.")

# -----------------------------
# Merge into patient-level modeling tables
# -----------------------------
def build_model_df(ed_df: pd.DataFrame, patients: pd.DataFrame,
                   adm_feat: pd.DataFrame, receipt_feat: pd.DataFrame) -> pd.DataFrame:
    df = ed_df.merge(patients, on="patient_id", how="left")
    if adm_feat is not None and len(adm_feat):
        df = df.merge(adm_feat, on="patient_id", how="left")
    if receipt_feat is not None and len(receipt_feat):
        df = df.merge(receipt_feat, on="patient_id", how="left", suffixes=("", "_rcpt"))

    fill_prefix_numeric_zero(df, prefixes=["adm_"], exclude=["adm_primary_dx_mode"])
    fill_prefix_numeric_zero(df, prefixes=["pdf_"])
    fill_prefix_numeric_zero(df, prefixes=["has_"])
    fill_prefix_numeric_zero(df, prefixes=["severe_"])

    if "adm_primary_dx_mode" in df.columns:
        df["adm_primary_dx_mode"] = df["adm_primary_dx_mode"].astype("string").fillna("Unknown")
    if "zip3_pdf" in df.columns:
        df["zip3_pdf"] = df["zip3_pdf"].apply(standardize_zip3)
    if "insurance_pdf" in df.columns:
        df["insurance_pdf"] = df["insurance_pdf"].astype("string").fillna("Unknown")
    return df

train_df = build_model_df(train, patients, adm_feat, receipt_feat)
test_df  = build_model_df(test,  patients, adm_feat, receipt_feat)

# Derived features
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["prior_cost"] = pd.to_numeric(df.get("prior_ed_cost_5y_usd", np.nan), errors="coerce").fillna(0.0)
    df["prior_visits"] = pd.to_numeric(df.get("prior_ed_visits_5y", np.nan), errors="coerce").fillna(0.0)

    df["log_prior_cost"] = np.log1p(df["prior_cost"])
    df["log_prior_visits"] = np.log1p(df["prior_visits"])
    df["cost_per_visit"] = df["prior_cost"] / np.maximum(1.0, df["prior_visits"])
    df["log_cost_per_visit"] = np.log1p(df["cost_per_visit"])
    df["visits_per_year"] = df["prior_visits"] / 5.0
    df["cost_per_year"] = df["prior_cost"] / 5.0
    df["baseline_next3y"] = df["prior_cost"] * (3.0 / 5.0)
    df["baseline_next3y_log"] = np.log1p(df["baseline_next3y"])

    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        age_med = float(df["age"].median()) if df["age"].notna().any() else 0.0
        df["age"] = df["age"].fillna(age_med)
        df["age_logpriorcost"] = df["age"] * df["log_prior_cost"]
        df["age_logpriorvisits"] = df["age"] * df["log_prior_visits"]

    if "adm_charlson_band_mean" in df.columns:
        df["priorcost_x_charlson"] = df["log_prior_cost"] * pd.to_numeric(df["adm_charlson_band_mean"], errors="coerce").fillna(0.0)
    if "severe_proc_score" in df.columns:
        df["priorcost_x_severeproc"] = df["log_prior_cost"] * pd.to_numeric(df["severe_proc_score"], errors="coerce").fillna(0.0)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

train_df = add_derived_features(train_df)
test_df  = add_derived_features(test_df)

print(f"[data] train_df={train_df.shape} test_df={test_df.shape}")

# -----------------------------
# Prepare X/y and categorical handling
# -----------------------------
TARGET_COL = "ed_cost_next3y_usd"
y = pd.to_numeric(train_df[TARGET_COL], errors="coerce").values.astype(float)

X = train_df.drop(columns=[TARGET_COL])
X_test = test_df.copy()

for df in [X, X_test]:
    df.drop(columns=["patient_id"], inplace=True, errors="ignore")

known_cats = ["primary_chronic","sex","insurance","zip3","zip3_pdf","insurance_pdf","adm_primary_dx_mode"]
cat_cols = [c for c in known_cats if (c in X.columns and c in X_test.columns)]

ensure_cat_as_str(X, cat_cols)
ensure_cat_as_str(X_test, cat_cols)

num_cols = [c for c in X.columns if c not in cat_cols]
median_fill_numeric(X, X_test, num_cols)

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
median_fill_numeric(X, X_test, num_cols)

# -----------------------------
# Modeling: CatBoost ensemble + CV
# -----------------------------
try:
    from catboost import CatBoostRegressor, Pool
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
    from catboost import CatBoostRegressor, Pool

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error

use_gpu = detect_gpu_catboost()
print(f"[catboost] GPU available: {use_gpu}")

strat = train_df["primary_chronic"].astype(str) + "_" + pd.qcut(train_df["prior_ed_cost_5y_usd"], q=8, duplicates="drop").astype(str)

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

common_params = dict(
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    task_type="GPU" if use_gpu else "CPU",
)
if not use_gpu:
    common_params["thread_count"] = -1

model_defs = [
    dict(
        name="cb_mae_deep",
        target_transform=None,
        params=dict(common_params, loss_function="MAE", eval_metric="MAE",
                    iterations=3500 if use_gpu else 1400, learning_rate=0.06, depth=8,
                    l2_leaf_reg=6, bagging_temperature=0.6, random_strength=1.0),
    ),
    dict(
        name="cb_mae_shallow",
        target_transform=None,
        params=dict(common_params, loss_function="MAE", eval_metric="MAE",
                    iterations=3000 if use_gpu else 1200, learning_rate=0.08, depth=6,
                    l2_leaf_reg=4, bagging_temperature=1.0, random_strength=0.7),
    ),
    dict(
        name="cb_rmse_log",
        target_transform="log1p",
        params=dict(common_params, loss_function="RMSE", eval_metric="RMSE",
                    iterations=5000 if use_gpu else 1400, learning_rate=0.06, depth=8,
                    l2_leaf_reg=3, bagging_temperature=0.5, random_strength=1.0),
    ),
]

n = len(X)
fold_id = np.full(n, -1, dtype=int)
oof_preds = {m["name"]: np.zeros(n, dtype=float) for m in model_defs}
best_iters = {m["name"]: [] for m in model_defs}
fold_mae_by_model = {m["name"]: [] for m in model_defs}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, strat), 1):
    fold_id[va_idx] = fold
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    for m in model_defs:
        name = m["name"]
        params = m["params"].copy()
        params["random_seed"] = SEED + fold

        if m["target_transform"] == "log1p":
            y_tr_fit = np.log1p(y_tr)
            y_va_fit = np.log1p(y_va)
            train_pool = Pool(X_tr, y_tr_fit, cat_features=cat_cols) if cat_cols else Pool(X_tr, y_tr_fit)
            valid_pool = Pool(X_va, y_va_fit, cat_features=cat_cols) if cat_cols else Pool(X_va, y_va_fit)
            model = CatBoostRegressor(**params)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=200)
            pred_va = np.expm1(model.predict(X_va))
        else:
            train_pool = Pool(X_tr, y_tr, cat_features=cat_cols) if cat_cols else Pool(X_tr, y_tr)
            valid_pool = Pool(X_va, y_va, cat_features=cat_cols) if cat_cols else Pool(X_va, y_va)
            model = CatBoostRegressor(**params)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=200)
            pred_va = model.predict(X_va)

        pred_va = np.clip(pred_va, 0.0, None)
        oof_preds[name][va_idx] = pred_va

        try:
            best_iters[name].append(int(model.get_best_iteration()))
        except Exception:
            pass

        mae = mean_absolute_error(y_va, pred_va)
        fold_mae_by_model[name].append(mae)

    fold_summary = " | ".join([f"{m['name']}={fold_mae_by_model[m['name']][-1]:.2f}" for m in model_defs])
    print(f"[CV] fold {fold}/{N_SPLITS}: {fold_summary}")

model_mae = {name: float(mean_absolute_error(y, oof)) for name, oof in oof_preds.items()}
print("\n[OOF] MAE per model:")
for name, mae in sorted(model_mae.items(), key=lambda x: x[1]):
    bi = int(np.median(best_iters[name])) if best_iters[name] else "NA"
    print(f"  {name}: {mae:.4f} | best_iter_median={bi}")

# -----------------------------
# Ensemble search + foldwise calibration
# -----------------------------
oof_mat = np.vstack([oof_preds[m["name"]] for m in model_defs]).T

def mae_of(pred):
    return float(mean_absolute_error(y, pred))

ens_candidates = []
ens_candidates.append(("mean", oof_mat.mean(axis=1), None))
ens_candidates.append(("median", np.median(oof_mat, axis=1), None))

best_w = None
best_pred = None
best_mae = 1e18
step = 0.05
grid = np.arange(0.0, 1.0 + 1e-9, step)

for w1 in grid:
    for w2 in grid:
        w3 = 1.0 - w1 - w2
        if w3 < -1e-9:
            continue
        if w3 < 0:
            w3 = 0.0
        s = w1 + w2 + w3
        if s <= 0:
            continue
        w1n, w2n, w3n = w1/s, w2/s, w3/s
        pred = w1n*oof_mat[:,0] + w2n*oof_mat[:,1] + w3n*oof_mat[:,2]
        m = mae_of(pred)
        if m < best_mae:
            best_mae, best_w, best_pred = m, (w1n, w2n, w3n), pred

ens_candidates.append(("weighted_grid", best_pred, best_w))

group_cols = [c for c in ["primary_chronic", "insurance"] if c in train_df.columns and c in test_df.columns]
calib_shrink = 0.7

def foldwise_group_calibrate(oof_pred: np.ndarray, df_train: pd.DataFrame, y_true: np.ndarray,
                             fold_id: np.ndarray, group_cols: List[str]) -> Tuple[np.ndarray, Dict[str, float], float]:
    if not group_cols:
        calibrated = np.zeros_like(oof_pred)
        for f in np.unique(fold_id):
            val_mask = fold_id == f
            tr_mask = ~val_mask
            shift = float(np.median(y_true[tr_mask] - oof_pred[tr_mask]))
            calibrated[val_mask] = oof_pred[val_mask] + calib_shrink * shift
        full_shift = float(np.median(y_true - oof_pred))
        return calibrated, {}, full_shift

    keys = df_train[group_cols].astype(str).agg("|".join, axis=1).values
    calibrated = np.zeros_like(oof_pred)

    for f in np.unique(fold_id):
        val_mask = fold_id == f
        tr_mask = ~val_mask

        resid_tr = y_true[tr_mask] - oof_pred[tr_mask]
        keys_tr = keys[tr_mask]

        tmp = pd.DataFrame({"k": keys_tr, "r": resid_tr})
        grp = tmp.groupby("k")["r"].median().to_dict()
        global_med = float(np.median(resid_tr))

        keys_val = keys[val_mask]
        adj = np.array([grp.get(k, global_med) for k in keys_val], dtype=float)
        calibrated[val_mask] = oof_pred[val_mask] + calib_shrink * adj

    full_resid = y_true - oof_pred
    full_map = pd.DataFrame({"k": keys, "r": full_resid}).groupby("k")["r"].median().to_dict()
    full_global = float(np.median(full_resid))
    return calibrated, full_map, full_global

best_ens_name = None
best_ens_w = None
best_ens_pred = None
best_final_mae = 1e18
best_cal_map = {}
best_cal_global = 0.0
use_calibration = False

CALIB_MIN_GAIN = 0.5  # require at least this MAE gain to enable calibration

for name, pred, w in ens_candidates:
    pred = np.clip(pred, 0.0, None)
    mae_raw = mae_of(pred)

    pred_cal, cal_map, cal_global = foldwise_group_calibrate(pred, train_df, y, fold_id, group_cols)
    pred_cal = np.clip(pred_cal, 0.0, None)
    mae_cal = mae_of(pred_cal)

    print(f"[ENS] {name:>12s} | raw_MAE={mae_raw:.4f} | cal_MAE={mae_cal:.4f}")

    gain = mae_raw - mae_cal
    use_cal = gain >= CALIB_MIN_GAIN
    effective_mae = mae_cal if use_cal else mae_raw

    if effective_mae < best_final_mae:
        best_final_mae = effective_mae
        best_ens_name, best_ens_w, best_ens_pred = name, w, pred
        best_cal_map, best_cal_global = cal_map, cal_global
        use_calibration = use_cal

if use_calibration:
    final_oof_cal, _, _ = foldwise_group_calibrate(best_ens_pred, train_df, y, fold_id, group_cols)
    final_cv_mae = mae_of(np.clip(final_oof_cal, 0.0, None))
else:
    final_cv_mae = mae_of(best_ens_pred)

print("\n==============================")
print(f"Chosen ensemble: {best_ens_name} | weights={best_ens_w} | calibration={use_calibration} | CV_MAE={final_cv_mae:.4f}")
print("==============================\n")

# -----------------------------
# Train final models on full train + predict test
# -----------------------------
test_preds = []
trained_models = {}

for m in model_defs:
    name = m["name"]
    params = m["params"].copy()
    if best_iters[name]:
        it_med = int(np.median(best_iters[name]))
        params["iterations"] = int(max(200, it_med + (60 if use_gpu else 40)))
    params["random_seed"] = SEED

    if m["target_transform"] == "log1p":
        y_fit = np.log1p(y)
        train_pool = Pool(X, y_fit, cat_features=cat_cols) if cat_cols else Pool(X, y_fit)
        model = CatBoostRegressor(**params)
        model.fit(train_pool, verbose=False)
        pred_test = np.expm1(model.predict(X_test))
    else:
        train_pool = Pool(X, y, cat_features=cat_cols) if cat_cols else Pool(X, y)
        model = CatBoostRegressor(**params)
        model.fit(train_pool, verbose=False)
        pred_test = model.predict(X_test)

    pred_test = np.clip(pred_test, 0.0, None)
    test_preds.append(pred_test)
    trained_models[name] = model

test_mat = np.vstack(test_preds).T

if best_ens_name == "mean":
    pred_test_ens = test_mat.mean(axis=1)
elif best_ens_name == "median":
    pred_test_ens = np.median(test_mat, axis=1)
elif best_ens_name == "weighted_grid" and best_ens_w is not None:
    w = np.array(best_ens_w, dtype=float)
    w = w / w.sum()
    pred_test_ens = (test_mat * w.reshape(1, -1)).sum(axis=1)
else:
    pred_test_ens = test_mat.mean(axis=1)

pred_test_ens = np.clip(pred_test_ens, 0.0, None)

if use_calibration:
    if group_cols:
        keys_test = test_df[group_cols].astype(str).agg("|".join, axis=1).values
        adj = np.array([best_cal_map.get(k, best_cal_global) for k in keys_test], dtype=float)
        pred_test_ens = np.clip(pred_test_ens + calib_shrink * adj, 0.0, None)
    else:
        pred_test_ens = np.clip(pred_test_ens + calib_shrink * best_cal_global, 0.0, None)

# -----------------------------
# Feature importance (top 10) from main model
# -----------------------------
try:
    main_model_name = "cb_mae_deep"
    main_model = trained_models.get(main_model_name)
    if main_model is not None:
        importances = main_model.get_feature_importance()
        feat_names = X.columns.tolist()
        imp_df = pd.DataFrame({"feature": feat_names, "importance": importances}).sort_values("importance", ascending=False).head(10)
        print("[Top 10 feature importances] (from cb_mae_deep)")
        print(imp_df.to_string(index=False))
except Exception as e:
    print(f"[warn] Could not compute feature importances: {e}")

# -----------------------------
# Write submission
# -----------------------------
sub = pd.DataFrame({
    "patient_id": test["patient_id"].astype(int).values,
    "ed_cost_next3y_usd": pred_test_ens.astype(float),
})[["patient_id", "ed_cost_next3y_usd"]]

print("\n[Sanity checks]")
print("overall CV MAE:", f"{final_cv_mae:.4f}")
print("ensemble:", best_ens_name, "| weights:", best_ens_w, "| calibration:", use_calibration)
print("submission shape:", sub.shape)
print("submission columns:", list(sub.columns))
print("any NaN preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(np.min(sub["ed_cost_next3y_usd"])), float(np.median(sub["ed_cost_next3y_usd"])), float(np.max(sub["ed_cost_next3y_usd"])))

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print(f"\nSaved submission to: {out_path}")
print("\nPaste back CV MAE and these logs for next iteration.")


Plan: patient+admissions aggregates + receipt code/spend features -> CatBoost MAE ensemble (+ log-target variant) + foldwise calibration
[admissions] patient-level features shape: (4000, 26)
[receipts] Loaded receipts_parsed.joblib -> receipt_feat shape: (4000, 107)
[data] train_df=(2000, 154) test_df=(2000, 153)
[catboost] GPU available: True


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


[CV] fold 1/5: cb_mae_deep=1389.42 | cb_mae_shallow=1378.73 | cb_rmse_log=461.24


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


[CV] fold 2/5: cb_mae_deep=1356.27 | cb_mae_shallow=1346.17 | cb_rmse_log=459.13


...

[CV] fold 4/5: cb_mae_deep=1298.54 | cb_mae_shallow=1287.90 | cb_rmse_log=463.85


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


[CV] fold 5/5: cb_mae_deep=1384.90 | cb_mae_shallow=1374.43 | cb_rmse_log=453.93

[OOF] MAE per model:
  cb_rmse_log: 456.4115 | best_iter_median=200
  cb_mae_shallow: 1356.3319 | best_iter_median=2999
  cb_mae_deep: 1366.7337 | best_iter_median=3499
[ENS]         mean | raw_MAE=964.3954 | cal_MAE=893.3756
[ENS]       median | raw_MAE=1356.2527 | cal_MAE=1246.1392
[ENS] weighted_grid | raw_MAE=456.4115 | cal_MAE=453.5191

Chosen ensemble: weighted_grid | weights=(np.float64(0.0), np.float64(0.0), np.float64(1.0)) | calibration=True | CV_MAE=453.5191



Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


[Top 10 feature importances] (from cb_mae_deep)
               feature  importance
  prior_ed_cost_5y_usd   39.393905
       primary_chronic   35.962745
    pdf_deflated_total   11.338643
priorcost_x_severeproc    3.464032
         adm_dx_cnt_HF    2.170143
   pdf_total_line_cost    1.996929
         cost_per_year    1.434362
       baseline_next3y    1.143127
            prior_cost    0.936425
 pdf_cost_selected_sum    0.690212

[Sanity checks]
overall CV MAE: 453.5191
ensemble: weighted_grid | weights: (np.float64(0.0), np.float64(0.0), np.float64(1.0)) | calibration: True
submission shape: (2000, 2)
submission columns: ['patient_id', 'ed_cost_next3y_usd']
any NaN preds: False
pred min/median/max: 822.6893698810146 3558.7302934074014 10421.858638504582

Saved submission to: D:\AgentDs\agent_ds_healthcare\submission.csv

Paste back CV MAE and these logs for next iteration.


# Iteration 2
- 457.2229

In [4]:
import os, sys, re, math, json, warnings, gc
from pathlib import Path
from typing import Tuple, Dict, Optional, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# GLOBAL CONFIG
# ============================================================
SEED = 42
np.random.seed(SEED)

DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"

TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"

PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"

ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"

RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR     = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"

SAMPLE_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"  # sample reference only
OUT_SUB_PATH    = r"D:\AgentDs\agent_ds_healthcare\submission.csv"  # overwrite ok

print("=== NEXT ITERATION PIPELINE (more logs + more robust) ===")
print("Key changes vs last run:")
print("  1) Remove noisy group calibration (likely raised min pred + hurt LB).")
print("  2) Add robust receipt distribution features (top-share, gini, prefixes, top-code categorical).")
print("  3) Add unsupervised CPT embedding (TF-IDF + SVD) from line items (if available).")
print("  4) Train a small diverse model set: CatBoost Quantile(0.5), CatBoost MAE, CatBoost log-RMSE, plus a simple CatBoost baseline.")
print("  5) Ensemble via Dirichlet random weight search on OOF MAE + bin diagnostics; optional global bias shift only if it helps OOF.")
print("=========================================================\n")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def seed_everything(seed: int = 42):
    np.random.seed(seed)
seed_everything(SEED)

def exists(path: str) -> bool:
    return Path(path).exists()

def safe_read_csv(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(p)

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | columns={df.shape[1]} | mem={mem:.2f} MB")

def log_missing(name: str, df: pd.DataFrame, topk: int = 12):
    na = df.isna().mean().sort_values(ascending=False)
    na = na[na > 0]
    if len(na) == 0:
        print(f"[{name}] missing: none")
        return
    print(f"[{name}] missing top {min(topk, len(na))}:")
    print(na.head(topk).to_string())

def ensure_cat_as_str(df: pd.DataFrame, cols: List[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")

def median_fill_numeric(train_df: pd.DataFrame, test_df: pd.DataFrame, numeric_cols: List[str]) -> None:
    for c in numeric_cols:
        med = pd.to_numeric(train_df[c], errors="coerce").median()
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(med)
        test_df[c]  = pd.to_numeric(test_df[c],  errors="coerce").fillna(med)

def fill_prefix_numeric_zero(df: pd.DataFrame, prefixes: List[str], exclude: Optional[List[str]] = None) -> None:
    exclude = set(exclude or [])
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes) and (c not in exclude)]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

def mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def quantiles(x, qs=(0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def print_bin_mae(y_true, y_pred, n_bins=6, name="model"):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    qs = np.quantile(y_true, np.linspace(0, 1, n_bins + 1))
    bins = pd.cut(y_true, bins=qs, include_lowest=True, duplicates="drop")
    df = pd.DataFrame({"bin": bins.astype(str), "mae": np.abs(y_true - y_pred)})
    g = df.groupby("bin")["mae"].mean().sort_index()
    print(f"[{name}] MAE by target-quantile bin:")
    print(g.to_string())

def detect_gpu_catboost() -> bool:
    try:
        from catboost.utils import get_gpu_device_count
        return get_gpu_device_count() > 0
    except Exception:
        return False

# ------------------------------------------------------------
# Receipt parsing fallback (only if joblib missing or incomplete)
# ------------------------------------------------------------
def extract_text_from_pdf(path: Path) -> str:
    # Prefer pdfplumber; fallback to PyMuPDF.
    try:
        import pdfplumber
        with pdfplumber.open(str(path)) as pdf:
            return "\n".join((p.extract_text() or "") for p in pdf.pages)
    except Exception:
        try:
            import fitz  # PyMuPDF
            doc = fitz.open(str(path))
            text = "\n".join(page.get_text("text") for page in doc)
            doc.close()
            return text
        except Exception:
            return ""

def parse_receipt_text(text: str) -> Tuple[Dict, pd.DataFrame]:
    pid_re = re.compile(r"Patient ID:\s*(\d+)")
    zip_ins_re = re.compile(r"ZIP3:\s*([0-9A-Za-z]+)\s+Insurance:\s*([A-Za-z_]+)")
    line_re = re.compile(r"^(?P<code>\S+)\s+(?P<desc>.*?)\s+(?P<qty>\d+)\s+(?P<unit>[\d,]+\.\d{2})\s+(?P<line_total>[\d,]+\.\d{2})$")
    total_re = re.compile(r"^TOTAL\s+(?P<total>[\d,]+\.\d{2})$")

    pid = None
    zip3 = None
    insurance = None
    total = np.nan

    m = pid_re.search(text)
    if m:
        pid = int(m.group(1))
    m = zip_ins_re.search(text)
    if m:
        zip3 = standardize_zip3(m.group(1))
        insurance = m.group(2)

    rows = []
    for raw in text.splitlines():
        raw = raw.strip()
        if not raw:
            continue
        mt = total_re.match(raw)
        if mt:
            total = float(mt.group("total").replace(",", ""))
            continue
        ml = line_re.match(raw)
        if ml:
            d = ml.groupdict()
            rows.append({
                "patient_id": pid,
                "code": d["code"],
                "desc": d["desc"],
                "qty": int(d["qty"]),
                "unit": float(d["unit"].replace(",", "")),
                "line_total": float(d["line_total"].replace(",", "")),
            })

    header = {"patient_id": pid, "zip3_pdf": zip3, "insurance_pdf": insurance, "pdf_total": total}
    return header, pd.DataFrame(rows)

def parse_some_receipts_from_pdfs(pdf_dir: Path, patient_ids: List[int], max_to_parse: int = 5000) -> Tuple[pd.DataFrame, pd.DataFrame]:
    headers, items = [], []
    parsed = 0
    missing = 0
    for pid in patient_ids:
        pdf_path = pdf_dir / f"receipt_{pid}.pdf"
        if not pdf_path.exists():
            missing += 1
            continue
        txt = extract_text_from_pdf(pdf_path)
        h, df = parse_receipt_text(txt)
        headers.append(h)
        if df is not None and len(df):
            items.append(df)
        parsed += 1
        if parsed >= max_to_parse:
            break
    headers_df = pd.DataFrame(headers)
    lineitems_df = pd.concat(items, ignore_index=True) if items else pd.DataFrame(columns=["patient_id","code","desc","qty","unit","line_total"])
    print(f"[receipts:fallback] Parsed PDFs: parsed={parsed} headers={headers_df.shape} lineitems={lineitems_df.shape} missing_pdfs={missing}")
    return headers_df, lineitems_df

def extract_receipt_dfs(obj) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    headers_df, lineitems_df = None, None
    if isinstance(obj, dict):
        for hk in ["headers_df","headers","header_df","header"]:
            if hk in obj and isinstance(obj[hk], pd.DataFrame):
                headers_df = obj[hk]
                break
        for lk in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if lk in obj and isinstance(obj[lk], pd.DataFrame):
                lineitems_df = obj[lk]
                break
        if lineitems_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and {"patient_id","code"}.issubset(set(v.columns)):
                    lineitems_df = v
                    break
        if headers_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and ("patient_id" in v.columns) and (v is not lineitems_df):
                    headers_df = v
                    break
    elif isinstance(obj, (tuple, list)):
        dfs = [x for x in obj if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if {"patient_id","code"}.issubset(set(df.columns)):
                lineitems_df = df
            elif "patient_id" in df.columns:
                headers_df = df
    elif isinstance(obj, pd.DataFrame):
        if {"patient_id","code"}.issubset(set(obj.columns)):
            lineitems_df = obj
        elif "patient_id" in obj.columns:
            headers_df = obj
    return headers_df, lineitems_df

# ------------------------------------------------------------
# Receipt feature builder (patient-level)
# ------------------------------------------------------------
def build_receipt_features(headers_df: Optional[pd.DataFrame],
                          lineitems_df: Optional[pd.DataFrame],
                          max_codes_dense: int = 220,
                          svd_dim: int = 16) -> pd.DataFrame:
    """
    Build robust receipt-derived features:
      - per-code cost/cnt/share (dense for <= max_codes_dense codes, else top by cost+count + must codes)
      - distribution stats (entropy, gini, top shares, prefix aggregates)
      - top-code categorical features
      - optional unsupervised CPT embedding (TF-IDF+SVD) from code sequences
    """

    # If we only have patient-level features already, return minimal safe subset
    if (lineitems_df is None) or (not isinstance(lineitems_df, pd.DataFrame)) or (len(lineitems_df) == 0):
        if headers_df is not None and isinstance(headers_df, pd.DataFrame) and ("patient_id" in headers_df.columns):
            df = headers_df.copy()
            if "patient_id" not in df.columns:
                return pd.DataFrame(columns=["patient_id"])
            keep = ["patient_id"] + [c for c in df.columns if c != "patient_id"]
            return df[keep].copy()
        return pd.DataFrame(columns=["patient_id"])

    li = lineitems_df.copy()

    # normalize column names
    if "code" not in li.columns:
        for alt in ["cpt","cpt_code","hcpcs","proc_code"]:
            if alt in li.columns:
                li["code"] = li[alt]
                break
    if "line_total" not in li.columns:
        for alt in ["line_total_usd","total","amount","line_cost","line_total_$","sum_items"]:
            if alt in li.columns:
                li["line_total"] = li[alt]
                break
    if "qty" not in li.columns:
        for alt in ["quantity","q"]:
            if alt in li.columns:
                li["qty"] = li[alt]
                break
    if "unit" not in li.columns:
        if "line_total" in li.columns and "qty" in li.columns:
            li["unit"] = pd.to_numeric(li["line_total"], errors="coerce") / pd.to_numeric(li["qty"], errors="coerce").replace(0, np.nan)
        else:
            li["unit"] = np.nan

    if "patient_id" not in li.columns or "code" not in li.columns:
        return pd.DataFrame(columns=["patient_id"])

    # clean types
    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li["code"].astype(str).str.strip().str.upper()
    li["qty"] = pd.to_numeric(li["qty"], errors="coerce").fillna(1).astype(float)
    li["unit"] = pd.to_numeric(li["unit"], errors="coerce")
    li["line_total"] = pd.to_numeric(li["line_total"], errors="coerce").fillna(0.0).astype(float)

    # must-include high acuity codes
    must_codes = {
        "31500","36556","36620","92950",
        "70450","74177","84484","G0378",
        "99281","99282","99283","99284","99285",
        "99291","99292",
        "85025","87070","71045"
    }

    uniq_codes = li["code"].nunique()
    if uniq_codes <= max_codes_dense:
        codes = sorted(set(li["code"].unique().tolist()) | must_codes)
    else:
        code_cost = li.groupby("code")["line_total"].sum().sort_values(ascending=False)
        top_cost = code_cost.head(max_codes_dense).index.tolist()
        top_cnt = li["code"].value_counts().head(max_codes_dense).index.tolist()
        codes = sorted(set(top_cost + top_cnt + list(must_codes)))

    # deflated total to separate mix vs pricing (unit median per code)
    med_unit_by_code = li.groupby("code")["unit"].median()
    li["unit_med_code"] = li["code"].map(med_unit_by_code).astype(float)
    li["deflated_line_total"] = li["qty"] * li["unit_med_code"]

    # patient-level aggregates
    g = li.groupby("patient_id")
    agg = g.agg(
        pdf_total_line_cost=("line_total","sum"),
        pdf_deflated_total=("deflated_line_total","sum"),
        pdf_n_line_items=("code","size"),
        pdf_total_qty=("qty","sum"),
        pdf_n_unique_codes=("code","nunique"),
        pdf_unit_mean=("unit","mean"),
        pdf_unit_median=("unit","median"),
        pdf_line_max=("line_total","max"),
        pdf_line_mean=("line_total","mean"),
        pdf_line_std=("line_total", lambda x: float(np.std(x, ddof=0))),
        pdf_line_median=("line_total","median"),
    ).reset_index()

    agg["pdf_price_index"] = agg["pdf_total_line_cost"] / agg["pdf_deflated_total"].replace(0, np.nan)
    agg["pdf_price_index"] = agg["pdf_price_index"].replace([np.inf, -np.inf], np.nan).fillna(1.0)

    # Wide per-code features (counts + costs)
    li_sel = li[li["code"].isin(codes)].copy()
    piv_cnt  = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="size", fill_value=0)
    piv_cost = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="sum", fill_value=0.0)
    piv_cnt.columns  = [f"pdf_cnt_{c}"  for c in piv_cnt.columns.astype(str)]
    piv_cost.columns = [f"pdf_cost_{c}" for c in piv_cost.columns.astype(str)]
    wide = piv_cnt.join(piv_cost, how="outer").reset_index()

    out = agg.merge(wide, on="patient_id", how="left")
    fill_prefix_numeric_zero(out, prefixes=["pdf_"])

    # selected sums and other
    sel_cost_cols = [c for c in out.columns if c.startswith("pdf_cost_")]
    sel_cnt_cols  = [c for c in out.columns if c.startswith("pdf_cnt_")]
    out["pdf_cost_selected_sum"] = out[sel_cost_cols].sum(axis=1) if sel_cost_cols else 0.0
    out["pdf_cost_other"] = np.maximum(0.0, out["pdf_total_line_cost"] - out["pdf_cost_selected_sum"])
    out["pdf_cnt_selected_sum"] = out[sel_cnt_cols].sum(axis=1) if sel_cnt_cols else 0.0
    out["pdf_cnt_other"] = np.maximum(0.0, out["pdf_n_line_items"] - out["pdf_cnt_selected_sum"])

    denom = out["pdf_total_line_cost"].replace(0, np.nan)

    # Shares per selected code
    for c in codes:
        col = f"pdf_cost_{c}"
        out[f"pdf_share_{c}"] = (out[col] / denom).fillna(0.0) if col in out.columns else 0.0

    # High-acuity flags (from prompt)
    code_flags = {
        "has_intub_31500": "31500",
        "has_cvc_36556": "36556",
        "has_artline_36620": "36620",
        "has_cpr_92950": "92950",
        "has_ct_head_70450": "70450",
        "has_ct_abdpel_74177": "74177",
        "has_troponin_84484": "84484",
        "has_obs_G0378": "G0378",
        "has_99285": "99285",
    }
    for fname, code in code_flags.items():
        cnt_col = f"pdf_cnt_{code}"
        out[fname] = (out[cnt_col] > 0).astype(int) if cnt_col in out.columns else 0

    # critical care flag (99291/99292)
    crit_cols = [c for c in ["99291","99292"] if f"pdf_cnt_{c}" in out.columns]
    if crit_cols:
        out["has_critical_care"] = (out[[f"pdf_cnt_{c}" for c in crit_cols]].sum(axis=1) > 0).astype(int)
    else:
        out["has_critical_care"] = 0

    out["severe_proc_score"] = (
        out.get("has_intub_31500", 0)
        + out.get("has_cvc_36556", 0)
        + out.get("has_artline_36620", 0)
        + out.get("has_cpr_92950", 0)
        + out.get("has_critical_care", 0)
    ).astype(int)

    # E/M severity recurrence stats
    em_codes = ["99281","99282","99283","99284","99285"]
    em_level_map = {"99281":1, "99282":2, "99283":3, "99284":4, "99285":5}
    em = li[li["code"].isin(em_codes)].copy()
    if len(em):
        em["em_level"] = em["code"].map(em_level_map).astype(float)
        em_agg = em.groupby("patient_id").agg(
            pdf_em_level_mean=("em_level","mean"),
            pdf_em_level_max=("em_level","max"),
            pdf_em_lines=("em_level","size"),
            pdf_em_high_frac=("em_level", lambda x: float(np.mean(np.asarray(x) >= 4))),
        ).reset_index()
        out = out.merge(em_agg, on="patient_id", how="left")
    for c in ["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]:
        if c not in out.columns:
            out[c] = 0.0
    out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]] = out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]].fillna(0.0)

    # Cost distribution metrics across codes (entropy, gini, top shares)
    cost_code_cols = [c for c in out.columns if c.startswith("pdf_cost_")]
    if cost_code_cols:
        mat = out[cost_code_cols].to_numpy(dtype=float)
        row_sum = mat.sum(axis=1, keepdims=True)
        # entropy
        p = np.divide(mat, row_sum, out=np.zeros_like(mat), where=row_sum > 0)
        ent = -(p * np.log(np.clip(p, 1e-12, 1.0))).sum(axis=1)
        ent_norm = ent / np.log(max(2, mat.shape[1]))
        out["pdf_code_cost_entropy"] = ent
        out["pdf_code_cost_entropy_norm"] = ent_norm

        # top-k shares
        sorted_mat = np.sort(mat, axis=1)
        sum_x = sorted_mat.sum(axis=1)
        top1 = sorted_mat[:, -1]
        top2 = top1 + sorted_mat[:, -2] if sorted_mat.shape[1] >= 2 else top1
        top3 = top2 + sorted_mat[:, -3] if sorted_mat.shape[1] >= 3 else top2
        out["pdf_cost_top1_share"] = np.divide(top1, sum_x, out=np.zeros_like(top1), where=sum_x > 0)
        out["pdf_cost_top2_share"] = np.divide(top2, sum_x, out=np.zeros_like(top2), where=sum_x > 0)
        out["pdf_cost_top3_share"] = np.divide(top3, sum_x, out=np.zeros_like(top3), where=sum_x > 0)
        out["pdf_cost_max"] = top1
        out["pdf_cost_top1"] = top1
        out["pdf_cost_top2"] = top2
        out["pdf_cost_top3"] = top3

        # gini
        n = sorted_mat.shape[1]
        i = np.arange(1, n + 1, dtype=float)
        weighted = (sorted_mat * i.reshape(1, -1)).sum(axis=1)
        gini = np.zeros_like(sum_x, dtype=float)
        mask = sum_x > 0
        gini[mask] = (2.0 * weighted[mask]) / (n * sum_x[mask]) - (n + 1) / n
        out["pdf_cost_gini"] = gini

        # number of "dominant" codes
        shares = np.divide(mat, row_sum, out=np.zeros_like(mat), where=row_sum > 0)
        out["pdf_n_codes_share_gt_20pct"] = (shares > 0.20).sum(axis=1).astype(float)
        out["pdf_n_codes_share_gt_10pct"] = (shares > 0.10).sum(axis=1).astype(float)
    else:
        out["pdf_code_cost_entropy"] = 0.0
        out["pdf_code_cost_entropy_norm"] = 0.0
        out["pdf_cost_top1_share"] = 0.0
        out["pdf_cost_top2_share"] = 0.0
        out["pdf_cost_top3_share"] = 0.0
        out["pdf_cost_gini"] = 0.0
        out["pdf_n_codes_share_gt_20pct"] = 0.0
        out["pdf_n_codes_share_gt_10pct"] = 0.0
        out["pdf_cost_max"] = 0.0
        out["pdf_cost_top1"] = 0.0
        out["pdf_cost_top2"] = 0.0
        out["pdf_cost_top3"] = 0.0

    # Prefix aggregates (first char and first two chars)
    li_sel["prefix1"] = li_sel["code"].str[:1]
    li_sel["prefix2"] = li_sel["code"].str[:2]

    pref1_cost = li_sel.pivot_table(index="patient_id", columns="prefix1", values="line_total", aggfunc="sum", fill_value=0.0)
    pref1_cnt  = li_sel.pivot_table(index="patient_id", columns="prefix1", values="line_total", aggfunc="size", fill_value=0)
    pref1_cost.columns = [f"pdf_pref1_cost_{c}" for c in pref1_cost.columns.astype(str)]
    pref1_cnt.columns  = [f"pdf_pref1_cnt_{c}"  for c in pref1_cnt.columns.astype(str)]
    pref1 = pref1_cnt.join(pref1_cost, how="outer").reset_index()

    pref2_cost = li_sel.pivot_table(index="patient_id", columns="prefix2", values="line_total", aggfunc="sum", fill_value=0.0)
    pref2_cnt  = li_sel.pivot_table(index="patient_id", columns="prefix2", values="line_total", aggfunc="size", fill_value=0)
    # keep only top prefix2 by total cost to avoid too many columns
    pref2_tot = pref2_cost.sum(axis=0).sort_values(ascending=False)
    keep_p2 = pref2_tot.head(25).index.tolist()
    pref2_cost = pref2_cost[keep_p2]
    pref2_cnt  = pref2_cnt[keep_p2]
    pref2_cost.columns = [f"pdf_pref2_cost_{c}" for c in pref2_cost.columns.astype(str)]
    pref2_cnt.columns  = [f"pdf_pref2_cnt_{c}"  for c in pref2_cnt.columns.astype(str)]
    pref2 = pref2_cnt.join(pref2_cost, how="outer").reset_index()

    out = out.merge(pref1, on="patient_id", how="left")
    out = out.merge(pref2, on="patient_id", how="left")
    fill_prefix_numeric_zero(out, prefixes=["pdf_pref1_","pdf_pref2_"])

    # Top-code categorical features (by cost and by count)
    code_cost_patient = li.groupby(["patient_id","code"])["line_total"].sum().reset_index()
    code_cost_patient = code_cost_patient.sort_values(["patient_id","line_total"], ascending=[True, False])
    top1 = code_cost_patient.groupby("patient_id").head(1).rename(columns={"code":"pdf_topcode_cost1","line_total":"pdf_topcode_cost1_amt"})
    top2 = code_cost_patient.groupby("patient_id").head(2).groupby("patient_id").tail(1).rename(columns={"code":"pdf_topcode_cost2","line_total":"pdf_topcode_cost2_amt"})
    topc = top1[["patient_id","pdf_topcode_cost1","pdf_topcode_cost1_amt"]].merge(
        top2[["patient_id","pdf_topcode_cost2","pdf_topcode_cost2_amt"]], on="patient_id", how="left"
    )
    out = out.merge(topc, on="patient_id", how="left")
    for c in ["pdf_topcode_cost1","pdf_topcode_cost2"]:
        if c in out.columns:
            out[c] = out[c].astype("string").fillna("Unknown")

    denom2 = out["pdf_total_line_cost"].replace(0, np.nan)
    out["pdf_topcode_cost1_share"] = (pd.to_numeric(out.get("pdf_topcode_cost1_amt", 0.0), errors="coerce") / denom2).fillna(0.0)
    out["pdf_topcode_cost2_share"] = (pd.to_numeric(out.get("pdf_topcode_cost2_amt", 0.0), errors="coerce") / denom2).fillna(0.0)
    out["pdf_topcode_cost1_amt"] = pd.to_numeric(out.get("pdf_topcode_cost1_amt", 0.0), errors="coerce").fillna(0.0)
    out["pdf_topcode_cost2_amt"] = pd.to_numeric(out.get("pdf_topcode_cost2_amt", 0.0), errors="coerce").fillna(0.0)

    # Optional unsupervised CPT embedding (TF-IDF + SVD) using code sequences
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD

        code_str = (
            li.groupby(["patient_id","code"])
              .size()
              .reset_index(name="cnt")
              .sort_values(["patient_id","cnt"], ascending=[True, False])
        )
        # repeat tokens up to 3 to reflect repetition without exploding length
        code_str["rep"] = code_str["cnt"].clip(upper=3)
        code_str["token"] = code_str["code"].astype(str) + " "
        code_str["tok_rep"] = code_str.apply(lambda r: r["token"] * int(r["rep"]), axis=1)
        seq = code_str.groupby("patient_id")["tok_rep"].sum().reset_index()
        seq.columns = ["patient_id", "pdf_code_seq"]

        # fit on all sequences provided in li
        vec = TfidfVectorizer(token_pattern=r"\S+", min_df=2, max_features=400)
        X_tfidf = vec.fit_transform(seq["pdf_code_seq"].fillna(""))

        svd_dim_eff = min(svd_dim, max(2, X_tfidf.shape[1] - 1))
        svd = TruncatedSVD(n_components=svd_dim_eff, random_state=SEED)
        X_svd = svd.fit_transform(X_tfidf)

        svd_cols = [f"pdf_code_svd_{i}" for i in range(svd_dim_eff)]
        seq_svd = pd.DataFrame(X_svd, columns=svd_cols)
        seq_svd.insert(0, "patient_id", seq["patient_id"].values)

        out = out.merge(seq_svd, on="patient_id", how="left")
        fill_prefix_numeric_zero(out, prefixes=["pdf_code_svd_"])
        print(f"[receipts] Added CPT TF-IDF+SVD embeddings: dim={svd_dim_eff} | vocab={X_tfidf.shape[1]}")
    except Exception as e:
        print(f"[receipts] Skipped CPT embedding (TF-IDF+SVD) due to: {e}")

    # Merge header features if present (pdf_total may be missing; sum_items is reliable per prompt)
    if headers_df is not None and isinstance(headers_df, pd.DataFrame) and ("patient_id" in headers_df.columns):
        cols = ["patient_id"] + [c for c in ["pdf_total","sum_items","zip3_pdf","insurance_pdf"] if c in headers_df.columns]
        hd = headers_df[cols].copy()
        out = out.merge(hd, on="patient_id", how="left")

        if "pdf_total" in out.columns:
            out["pdf_total_missing"] = out["pdf_total"].isna().astype(int)
            out["pdf_total"] = pd.to_numeric(out["pdf_total"], errors="coerce").fillna(0.0)

        if "sum_items" in out.columns:
            out["sum_items"] = pd.to_numeric(out["sum_items"], errors="coerce").fillna(0.0)
            out["sum_items_minus_linecost"] = out["sum_items"] - out["pdf_total_line_cost"]
            out["sum_items_missing"] = out["sum_items"].isna().astype(int)

        if "zip3_pdf" in out.columns:
            out["zip3_pdf"] = out["zip3_pdf"].apply(standardize_zip3)

        if "insurance_pdf" in out.columns:
            out["insurance_pdf"] = out["insurance_pdf"].astype("string").fillna("Unknown")

    return out

# ------------------------------------------------------------
# Admissions aggregates (patient-level)
# ------------------------------------------------------------
def build_admissions_features(adm_tr: pd.DataFrame, adm_te: pd.DataFrame) -> pd.DataFrame:
    adm_tr = adm_tr.drop(columns=["readmit_30d"], errors="ignore")  # DO NOT USE label
    adm_all = pd.concat([adm_tr, adm_te], ignore_index=True)

    if "patient_id" not in adm_all.columns:
        return pd.DataFrame(columns=["patient_id"])

    adm_all["patient_id"] = pd.to_numeric(adm_all["patient_id"], errors="coerce").astype("Int64")
    adm_all = adm_all.dropna(subset=["patient_id"]).copy()
    adm_all["patient_id"] = adm_all["patient_id"].astype(int)

    num_cols = ["los_days","acuity_emergent","charlson_band","ed_visits_6m","discharge_weekday"]
    for c in num_cols:
        if c in adm_all.columns:
            adm_all[c] = pd.to_numeric(adm_all[c], errors="coerce")

    g = adm_all.groupby("patient_id")
    out = pd.DataFrame({"patient_id": g.size().index, "adm_n": g.size().values})

    for c in num_cols:
        if c in adm_all.columns:
            out[f"adm_{c}_mean"] = g[c].mean().values
            out[f"adm_{c}_max"]  = g[c].max().values
            out[f"adm_{c}_sum"]  = g[c].sum().values
            out[f"adm_{c}_std"]  = g[c].std(ddof=0).values

    # extra robust stats
    if "acuity_emergent" in adm_all.columns:
        out["adm_emergent_frac"] = (out.get("adm_acuity_emergent_sum", 0.0) / np.maximum(1.0, out["adm_n"])).astype(float)

    if "primary_dx" in adm_all.columns:
        dx_ct = pd.crosstab(adm_all["patient_id"], adm_all["primary_dx"])
        dx_ct.columns = [f"adm_dx_cnt_{str(c)}" for c in dx_ct.columns]
        dx_ct = dx_ct.reset_index()
        out = out.merge(dx_ct, on="patient_id", how="left")

        dx_mode = g["primary_dx"].agg(lambda x: x.value_counts().index[0] if len(x) else "Unknown").reset_index()
        dx_mode.columns = ["patient_id", "adm_primary_dx_mode"]
        out = out.merge(dx_mode, on="patient_id", how="left")

        dx_nuniq = g["primary_dx"].nunique().reset_index()
        dx_nuniq.columns = ["patient_id", "adm_dx_nunique"]
        out = out.merge(dx_nuniq, on="patient_id", how="left")

    for c in out.columns:
        if c in ["patient_id", "adm_primary_dx_mode"]:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    if "adm_primary_dx_mode" in out.columns:
        out["adm_primary_dx_mode"] = out["adm_primary_dx_mode"].astype("string").fillna("Unknown")

    return out

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
train = safe_read_csv(TRAIN_PATH)
test  = safe_read_csv(TEST_PATH)
patients = safe_read_csv(PATIENTS_PATH)
adm_tr = safe_read_csv(ADM_TRAIN_PATH)
adm_te = safe_read_csv(ADM_TEST_PATH)

for df in [train, test, patients, adm_tr, adm_te]:
    if "patient_id" in df.columns:
        df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

if "zip3" in patients.columns:
    patients["zip3"] = patients["zip3"].apply(standardize_zip3)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)
log_shape("admissions_train", adm_tr)
log_shape("admissions_test", adm_te)

print("\n[target stats]")
print(train["ed_cost_next3y_usd"].describe().to_string())
print("\n[prior cost stats]")
print(train["prior_ed_cost_5y_usd"].describe().to_string())

# ------------------------------------------------------------
# Build admissions + receipt features
# ------------------------------------------------------------
adm_feat = build_admissions_features(adm_tr, adm_te)
print(f"\n[admissions] patient-level features shape: {adm_feat.shape}")
if len(adm_feat):
    log_missing("adm_feat", adm_feat)

receipt_feat = pd.DataFrame(columns=["patient_id"])

# joblib load
joblib_path = Path(RECEIPTS_JOBLIB_PATH)
pdf_dir = Path(RECEIPTS_PDF_DIR)
all_patient_ids = pd.concat([train["patient_id"], test["patient_id"]], ignore_index=True).astype(int).unique().tolist()

joblib_load = None
try:
    from joblib import load as joblib_load
except Exception:
    joblib_load = None

headers_df = None
lineitems_df = None

if joblib_path.exists() and (joblib_load is not None):
    try:
        obj = joblib_load(joblib_path)
        headers_df, lineitems_df = extract_receipt_dfs(obj)

        if isinstance(obj, pd.DataFrame) and ("patient_id" in obj.columns) and ("code" not in obj.columns) and (obj["patient_id"].nunique() == len(obj)):
            receipt_feat = obj.copy()
            print(f"\n[receipts] Loaded receipts_parsed.joblib as patient-level DF: {receipt_feat.shape}")
        else:
            receipt_feat = build_receipt_features(headers_df, lineitems_df, max_codes_dense=220, svd_dim=16)
            print(f"\n[receipts] Loaded receipts_parsed.joblib -> built receipt_feat: {receipt_feat.shape}")

    except Exception as e:
        print(f"\n[receipts] Failed loading joblib ({e}); will try PDF fallback if available.")
        headers_df, lineitems_df = None, None

else:
    print("\n[receipts] receipts_parsed.joblib not found; will try PDF fallback if available.")

# Coverage check + fallback for missing receipt patients (if we have PDFs)
if receipt_feat is not None and len(receipt_feat) and ("patient_id" in receipt_feat.columns):
    receipt_feat["patient_id"] = pd.to_numeric(receipt_feat["patient_id"], errors="coerce").astype("Int64")
    receipt_feat = receipt_feat.dropna(subset=["patient_id"]).copy()
    receipt_feat["patient_id"] = receipt_feat["patient_id"].astype(int)

    cov = receipt_feat["patient_id"].nunique() / len(all_patient_ids)
    missing_pids = sorted(set(all_patient_ids) - set(receipt_feat["patient_id"].unique().tolist()))
    print(f"[receipts] coverage: {receipt_feat['patient_id'].nunique()}/{len(all_patient_ids)} = {cov:.3f} | missing={len(missing_pids)}")

    if (len(missing_pids) > 0) and pdf_dir.exists():
        print("[receipts] Attempting PDF fallback for missing patients (only missing ones)...")
        h2, li2 = parse_some_receipts_from_pdfs(pdf_dir, missing_pids, max_to_parse=len(missing_pids))
        if len(li2):
            r2 = build_receipt_features(h2, li2, max_codes_dense=220, svd_dim=16)
            receipt_feat = pd.concat([receipt_feat, r2], ignore_index=True)
            receipt_feat = receipt_feat.drop_duplicates(subset=["patient_id"], keep="last")
            cov2 = receipt_feat["patient_id"].nunique() / len(all_patient_ids)
            print(f"[receipts] coverage after fallback: {receipt_feat['patient_id'].nunique()}/{len(all_patient_ids)} = {cov2:.3f}")
else:
    if pdf_dir.exists():
        print("\n[receipts] No receipt_feat from joblib; doing full PDF parse fallback (may be slower)...")
        h2, li2 = parse_some_receipts_from_pdfs(pdf_dir, all_patient_ids, max_to_parse=len(all_patient_ids))
        receipt_feat = build_receipt_features(h2, li2, max_codes_dense=220, svd_dim=16)
        print(f"[receipts] Built receipt_feat from PDFs: {receipt_feat.shape}")
    else:
        print("\n[receipts] No receipts features available (joblib missing and receipts_pdf folder missing). Proceeding without.")

if receipt_feat is not None and len(receipt_feat):
    log_missing("receipt_feat", receipt_feat)

# ------------------------------------------------------------
# Merge into patient-level modeling tables
# ------------------------------------------------------------
def build_model_df(ed_df: pd.DataFrame, patients: pd.DataFrame,
                   adm_feat: pd.DataFrame, receipt_feat: pd.DataFrame) -> pd.DataFrame:
    df = ed_df.merge(patients, on="patient_id", how="left")
    if adm_feat is not None and len(adm_feat):
        df = df.merge(adm_feat, on="patient_id", how="left")
    if receipt_feat is not None and len(receipt_feat):
        df = df.merge(receipt_feat, on="patient_id", how="left", suffixes=("", "_rcpt"))

    fill_prefix_numeric_zero(df, prefixes=["adm_"], exclude=["adm_primary_dx_mode"])
    fill_prefix_numeric_zero(df, prefixes=["pdf_"])
    fill_prefix_numeric_zero(df, prefixes=["has_"])
    fill_prefix_numeric_zero(df, prefixes=["severe_"])

    # categorical receipt-derived
    for c in ["zip3_pdf","insurance_pdf","pdf_topcode_cost1","pdf_topcode_cost2"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")
            if c == "zip3_pdf":
                df[c] = df[c].apply(standardize_zip3)

    if "adm_primary_dx_mode" in df.columns:
        df["adm_primary_dx_mode"] = df["adm_primary_dx_mode"].astype("string").fillna("Unknown")

    return df

train_df = build_model_df(train, patients, adm_feat, receipt_feat)
test_df  = build_model_df(test,  patients, adm_feat, receipt_feat)

print("\n[merged data]")
log_shape("train_df", train_df)
log_shape("test_df", test_df)
log_missing("train_df", train_df, topk=15)

# ------------------------------------------------------------
# Derived features
# ------------------------------------------------------------
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["prior_cost"] = pd.to_numeric(df.get("prior_ed_cost_5y_usd", np.nan), errors="coerce").fillna(0.0)
    df["prior_visits"] = pd.to_numeric(df.get("prior_ed_visits_5y", np.nan), errors="coerce").fillna(0.0)

    df["log_prior_cost"] = np.log1p(df["prior_cost"])
    df["log_prior_visits"] = np.log1p(df["prior_visits"])
    df["cost_per_visit"] = df["prior_cost"] / np.maximum(1.0, df["prior_visits"])
    df["log_cost_per_visit"] = np.log1p(df["cost_per_visit"])

    # mild caps to stabilize extremes (tree-friendly)
    df["prior_cost_cap20k"] = np.minimum(df["prior_cost"], 20000.0)
    df["log_prior_cost_cap20k"] = np.log1p(df["prior_cost_cap20k"])

    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        df["age"] = df["age"].fillna(df["age"].median())
        df["age_x_logpriorcost"] = df["age"] * df["log_prior_cost"]

    if "severe_proc_score" in df.columns:
        df["priorcost_x_severeproc"] = df["log_prior_cost"] * pd.to_numeric(df["severe_proc_score"], errors="coerce").fillna(0.0)

    # receipts: concentrate/complexity interactions
    if "pdf_cost_gini" in df.columns:
        df["logpriorcost_x_gini"] = df["log_prior_cost"] * pd.to_numeric(df["pdf_cost_gini"], errors="coerce").fillna(0.0)
    if "pdf_cost_top1_share" in df.columns:
        df["logpriorcost_x_top1share"] = df["log_prior_cost"] * pd.to_numeric(df["pdf_cost_top1_share"], errors="coerce").fillna(0.0)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

train_df = add_derived_features(train_df)
test_df  = add_derived_features(test_df)

# ------------------------------------------------------------
# Prepare X/y and categorical handling
# ------------------------------------------------------------
TARGET_COL = "ed_cost_next3y_usd"
y = pd.to_numeric(train_df[TARGET_COL], errors="coerce").values.astype(float)

X = train_df.drop(columns=[TARGET_COL])
X_test = test_df.copy()

# keep patient_id for submission
test_patient_id = test_df["patient_id"].astype(int).values

# drop id from features
X = X.drop(columns=["patient_id"], errors="ignore")
X_test = X_test.drop(columns=["patient_id"], errors="ignore")

cat_candidates = [
    "primary_chronic","sex","insurance","zip3",
    "zip3_pdf","insurance_pdf",
    "adm_primary_dx_mode",
    "pdf_topcode_cost1","pdf_topcode_cost2",
]
cat_cols = [c for c in cat_candidates if c in X.columns and c in X_test.columns]

ensure_cat_as_str(X, cat_cols)
ensure_cat_as_str(X_test, cat_cols)

num_cols = [c for c in X.columns if c not in cat_cols]
median_fill_numeric(X, X_test, num_cols)

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
median_fill_numeric(X, X_test, num_cols)

print("\n[features]")
print(f"Total features: {X.shape[1]} | Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}")
print("Categorical columns:", cat_cols)

# ------------------------------------------------------------
# Modeling: CatBoost (Quantile/MAE/log-RMSE) + CV + Ensemble
# ------------------------------------------------------------
try:
    from catboost import CatBoostRegressor, Pool
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
    from catboost import CatBoostRegressor, Pool

from sklearn.model_selection import StratifiedKFold

use_gpu = detect_gpu_catboost()
print(f"\n[catboost] GPU available: {use_gpu}")

# Notes (sources):
# - CatBoost supports MAE / Quantile losses with alpha parameter; MAE-aligned objectives are key for MAE metric. :contentReference[oaicite:0]{index=0}

# Stratification for stable folds
try:
    strat = train_df["primary_chronic"].astype(str) + "_" + pd.qcut(train_df["prior_ed_cost_5y_usd"], q=8, duplicates="drop").astype(str)
except Exception:
    strat = train_df["primary_chronic"].astype(str)

N_SPLITS = 7  # slightly more stable OOF than 5, still cheap at 2k rows
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

common_cb = dict(
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    task_type="GPU" if use_gpu else "CPU",
)
if not use_gpu:
    common_cb["thread_count"] = -1

# Model defs
model_defs = [
    dict(
        name="cb_q50_d6",
        kind="catboost",
        target_transform=None,
        params=dict(common_cb,
                    loss_function="Quantile:alpha=0.5",
                    eval_metric="MAE",
                    iterations=6000 if use_gpu else 2400,
                    learning_rate=0.04,
                    depth=6,
                    l2_leaf_reg=10,
                    bagging_temperature=1.0,
                    random_strength=1.0,
                    rsm=0.85,
                    min_data_in_leaf=24),
        feature_subset="full",
    ),
    dict(
        name="cb_mae_d6",
        kind="catboost",
        target_transform=None,
        params=dict(common_cb,
                    loss_function="MAE",
                    eval_metric="MAE",
                    iterations=6000 if use_gpu else 2400,
                    learning_rate=0.04,
                    depth=6,
                    l2_leaf_reg=12,
                    bagging_temperature=1.2,
                    random_strength=1.0,
                    rsm=0.85,
                    min_data_in_leaf=24),
        feature_subset="full",
    ),
    dict(
        name="cb_logrmse_d7",
        kind="catboost",
        target_transform="log1p",
        params=dict(common_cb,
                    loss_function="RMSE",
                    eval_metric="RMSE",
                    iterations=8000 if use_gpu else 2600,
                    learning_rate=0.04,
                    depth=7,
                    l2_leaf_reg=6,
                    bagging_temperature=0.7,
                    random_strength=1.0,
                    rsm=0.90,
                    min_data_in_leaf=24),
        feature_subset="full",
    ),
    dict(
        name="cb_base_mae_small",
        kind="catboost",
        target_transform=None,
        params=dict(common_cb,
                    loss_function="MAE",
                    eval_metric="MAE",
                    iterations=5000 if use_gpu else 2000,
                    learning_rate=0.06,
                    depth=4,
                    l2_leaf_reg=12,
                    bagging_temperature=1.0,
                    random_strength=1.0,
                    rsm=0.90,
                    min_data_in_leaf=32),
        feature_subset="base",
    ),
]

# Base feature subset (strong priors only + minimal clinical)
base_cols = [c for c in [
    "primary_chronic","prior_ed_visits_5y","prior_ed_cost_5y_usd",
    "age","sex","insurance","zip3",
    "adm_n","adm_charlson_band_mean","adm_charlson_band_max","adm_ed_visits_6m_mean","adm_ed_visits_6m_max",
    "severe_proc_score","has_critical_care","has_intub_31500","has_cvc_36556","has_cpr_92950",
    "pdf_code_cost_entropy_norm","pdf_cost_gini","pdf_cost_top1_share","pdf_n_unique_codes","pdf_n_line_items"
] if c in X.columns]

def get_X_subset(df: pd.DataFrame, subset: str) -> pd.DataFrame:
    if subset == "base":
        return df[base_cols].copy() if base_cols else df.copy()
    return df

# OOF storage
n = len(X)
fold_id = np.full(n, -1, dtype=int)
oof = {m["name"]: np.zeros(n, dtype=float) for m in model_defs}
fold_mae = {m["name"]: [] for m in model_defs}
best_iters = {m["name"]: [] for m in model_defs}

# CV loop
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, strat), 1):
    fold_id[va_idx] = fold

    X_tr_full, X_va_full = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    for m in model_defs:
        name = m["name"]
        subset = m.get("feature_subset", "full")
        X_tr = get_X_subset(X_tr_full, subset)
        X_va = get_X_subset(X_va_full, subset)

        # adapt cat cols for subset
        cat_cols_sub = [c for c in cat_cols if c in X_tr.columns]

        params = m["params"].copy()
        params["random_seed"] = SEED + fold

        # Train with safe GPU fallback (some losses/configs might fail on GPU on certain setups)
        def fit_predict_catboost(params_local, y_tr_local, y_va_local, transform):
            train_pool = Pool(X_tr, y_tr_local, cat_features=cat_cols_sub) if cat_cols_sub else Pool(X_tr, y_tr_local)
            valid_pool = Pool(X_va, y_va_local, cat_features=cat_cols_sub) if cat_cols_sub else Pool(X_va, y_va_local)
            model = CatBoostRegressor(**params_local)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=250)
            pred = model.predict(X_va)
            return model, pred

        try:
            if m["target_transform"] == "log1p":
                y_tr_fit = np.log1p(y_tr)
                y_va_fit = np.log1p(y_va)
                model, pred_va = fit_predict_catboost(params, y_tr_fit, y_va_fit, "log1p")
                pred_va = np.expm1(pred_va)
            else:
                model, pred_va = fit_predict_catboost(params, y_tr, y_va, None)

        except Exception as e:
            # fallback to CPU for this fold/model if GPU fails
            if params.get("task_type") == "GPU":
                params_cpu = params.copy()
                params_cpu["task_type"] = "CPU"
                params_cpu["thread_count"] = -1
                if m["target_transform"] == "log1p":
                    y_tr_fit = np.log1p(y_tr)
                    y_va_fit = np.log1p(y_va)
                    model, pred_va = fit_predict_catboost(params_cpu, y_tr_fit, y_va_fit, "log1p")
                    pred_va = np.expm1(pred_va)
                else:
                    model, pred_va = fit_predict_catboost(params_cpu, y_tr, y_va, None)
                print(f"[warn] GPU failed for {name} fold {fold} ({e}); retrained on CPU.")
            else:
                raise

        pred_va = np.clip(pred_va, 0.0, None)
        oof[name][va_idx] = pred_va

        try:
            best_iters[name].append(int(model.get_best_iteration()))
        except Exception:
            pass

        fold_mae[name].append(mae(y_va, pred_va))

    fold_line = " | ".join([f"{m['name']}={fold_mae[m['name']][-1]:.2f}" for m in model_defs])
    print(f"[CV] fold {fold}/{N_SPLITS}: {fold_line}")

# OOF summary
print("\n[OOF summary]")
for m in model_defs:
    name = m["name"]
    overall = mae(y, oof[name])
    q = quantiles(oof[name], qs=(0,0.05,0.25,0.5,0.75,0.95,1.0))
    bias_med = float(np.median(y - oof[name]))
    bi = int(np.median(best_iters[name])) if best_iters[name] else None
    print(f"  {name:16s} MAE={overall:.4f} | median_resid(y-pred)={bias_med:+.2f} | best_iter_med={bi}")
    print(f"    pred quantiles: {q}")

# Bin diagnostics per model (helps catch 'min pred too high' pathology)
print("\n[OOF bin diagnostics]")
for m in model_defs:
    print_bin_mae(y, oof[m["name"]], n_bins=6, name=m["name"])
    print("")

# ------------------------------------------------------------
# Ensemble selection (Dirichlet random weights)
# ------------------------------------------------------------
model_names = [m["name"] for m in model_defs]
oof_mat = np.vstack([oof[nm] for nm in model_names]).T

# Add simple ensemble baselines
candidates = []
candidates.append(("mean", np.mean(oof_mat, axis=1), None))
candidates.append(("median", np.median(oof_mat, axis=1), None))
for i, nm in enumerate(model_names):
    candidates.append((f"single_{nm}", oof_mat[:, i], (np.eye(len(model_names))[i]).tolist()))

# Dirichlet random search
rng = np.random.default_rng(SEED)
n_tries = 2500  # cheap at 2k rows
best = {"name": None, "mae": 1e18, "pred": None, "w": None}
for t in range(n_tries):
    w = rng.dirichlet(np.ones(len(model_names)))
    pred = oof_mat.dot(w)
    m = mae(y, pred)
    if m < best["mae"]:
        best = {"name": "dirichlet", "mae": m, "pred": pred, "w": w}

candidates.append((best["name"], best["pred"], best["w"]))

# choose best by MAE (report also bins)
best_name, best_pred, best_w = None, None, None
best_mae_val = 1e18
for nm, pred, w in candidates:
    pred = np.clip(pred, 0.0, None)
    m = mae(y, pred)
    if m < best_mae_val:
        best_mae_val = m
        best_name, best_pred, best_w = nm, pred, w

print("=========================================================")
print(f"[ENS] best ensemble = {best_name} | OOF_MAE={best_mae_val:.4f}")
if best_w is not None:
    wdict = {model_names[i]: float(best_w[i]) for i in range(len(model_names))}
    wdict = dict(sorted(wdict.items(), key=lambda x: -x[1]))
    print("[ENS] weights:")
    for k,v in wdict.items():
        if v >= 0.02:
            print(f"  {k:16s}: {v:.3f}")
        else:
            pass
else:
    print("[ENS] weights: n/a")
print("=========================================================\n")

print_bin_mae(y, best_pred, n_bins=6, name=f"ENS({best_name})")

# Optional global bias shift (median residual) ONLY if it helps OOF MAE
best_pred_shift = best_pred.copy()
shift = float(np.median(y - best_pred_shift))
pred_shifted = np.clip(best_pred_shift + shift, 0.0, None)
mae_shifted = mae(y, pred_shifted)
use_shift = (mae_shifted + 0.15) < best_mae_val  # require real improvement margin
if use_shift:
    print(f"\n[BIAS] Applying global median-residual shift: {shift:+.2f} improves OOF MAE {best_mae_val:.4f} -> {mae_shifted:.4f}")
    best_pred = pred_shifted
    best_mae_val = mae_shifted
else:
    print(f"\n[BIAS] Skipping global shift (shift={shift:+.2f}) | OOF MAE would be {mae_shifted:.4f} vs {best_mae_val:.4f}")

# ------------------------------------------------------------
# Train final models on full train + predict test
# ------------------------------------------------------------
test_preds = []
trained_models = {}
test_mat_cols = []

for m in model_defs:
    name = m["name"]
    subset = m.get("feature_subset", "full")
    X_full = get_X_subset(X, subset)
    X_te   = get_X_subset(X_test, subset)
    cat_cols_sub = [c for c in cat_cols if c in X_full.columns]

    params = m["params"].copy()
    if best_iters[name]:
        it_med = int(np.median(best_iters[name]))
        params["iterations"] = int(max(400, it_med + (120 if use_gpu else 80)))
    params["random_seed"] = SEED

    def fit_full(params_local, y_fit):
        train_pool = Pool(X_full, y_fit, cat_features=cat_cols_sub) if cat_cols_sub else Pool(X_full, y_fit)
        model = CatBoostRegressor(**params_local)
        model.fit(train_pool, verbose=False)
        return model

    try:
        if m["target_transform"] == "log1p":
            model = fit_full(params, np.log1p(y))
            pred_te = np.expm1(model.predict(X_te))
        else:
            model = fit_full(params, y)
            pred_te = model.predict(X_te)
    except Exception as e:
        if params.get("task_type") == "GPU":
            params_cpu = params.copy()
            params_cpu["task_type"] = "CPU"
            params_cpu["thread_count"] = -1
            if m["target_transform"] == "log1p":
                model = fit_full(params_cpu, np.log1p(y))
                pred_te = np.expm1(model.predict(X_te))
            else:
                model = fit_full(params_cpu, y)
                pred_te = model.predict(X_te)
            print(f"[warn] GPU failed in final fit for {name} ({e}); retrained on CPU.")
        else:
            raise

    pred_te = np.clip(pred_te, 0.0, None)
    test_preds.append(pred_te)
    test_mat_cols.append(name)
    trained_models[name] = model

test_mat = np.vstack(test_preds).T

if best_name == "mean":
    pred_test = np.mean(test_mat, axis=1)
elif best_name == "median":
    pred_test = np.median(test_mat, axis=1)
elif best_name == "dirichlet" and best_w is not None:
    pred_test = test_mat.dot(best_w)
elif best_name.startswith("single_"):
    pick = best_name.replace("single_","")
    pred_test = test_mat[:, test_mat_cols.index(pick)]
else:
    # fallback: mean
    pred_test = np.mean(test_mat, axis=1)

pred_test = np.clip(pred_test, 0.0, None)

# Apply bias shift if chosen (use same shift computed from OOF)
if use_shift:
    pred_test = np.clip(pred_test + shift, 0.0, None)

# ------------------------------------------------------------
# Feature importance (top 10) from strongest single model (cb_q50_d6)
# ------------------------------------------------------------
try:
    main_name = "cb_q50_d6" if "cb_q50_d6" in trained_models else model_names[0]
    main_model = trained_models.get(main_name)
    if main_model is not None:
        # importance on the full feature set only
        if main_name == "cb_base_mae_small":
            X_imp = get_X_subset(X, "base")
            fn = X_imp.columns.tolist()
        else:
            fn = X.columns.tolist()
        imp = main_model.get_feature_importance()
        imp_df = pd.DataFrame({"feature": fn, "importance": imp}).sort_values("importance", ascending=False).head(15)
        print("\n[Top feature importances] (from", main_name, ")")
        print(imp_df.to_string(index=False))
except Exception as e:
    print(f"\n[warn] Could not compute feature importances: {e}")

# ------------------------------------------------------------
# Write submission
# ------------------------------------------------------------
sub = pd.DataFrame({
    "patient_id": test_patient_id,
    "ed_cost_next3y_usd": pred_test.astype(float),
})[["patient_id", "ed_cost_next3y_usd"]]

print("\n[FINAL sanity checks]")
print("overall CV MAE (best ensemble OOF):", f"{best_mae_val:.4f}")
print("ensemble:", best_name, "| bias_shift_used:", bool(use_shift), ("| shift=" + f"{shift:+.2f}" if use_shift else ""))
print("submission shape:", sub.shape)
print("submission columns:", list(sub.columns))
print("any NaN preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(np.min(sub["ed_cost_next3y_usd"])), float(np.median(sub["ed_cost_next3y_usd"])), float(np.max(sub["ed_cost_next3y_usd"])))
print("pred quantiles:", quantiles(sub["ed_cost_next3y_usd"].values, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)))

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print(f"\nSaved submission to: {out_path}")
print("\nPaste back: (1) CV MAE, (2) chosen ensemble + weights, (3) pred quantiles, (4) leaderboard score.")


=== NEXT ITERATION PIPELINE (more logs + more robust) ===
Key changes vs last run:
  1) Remove noisy group calibration (likely raised min pred + hurt LB).
  2) Add robust receipt distribution features (top-share, gini, prefixes, top-code categorical).
  3) Add unsupervised CPT embedding (TF-IDF + SVD) from line items (if available).
  4) Train a small diverse model set: CatBoost Quantile(0.5), CatBoost MAE, CatBoost log-RMSE, plus a simple CatBoost baseline.
  5) Ensemble via Dirichlet random weight search on OOF MAE + bin diagnostics; optional global bias shift only if it helps OOF.

[ed_cost_train] shape=(2000, 5) | columns=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | columns=4 | mem=0.15 MB
[patients] shape=(4000, 5) | columns=5 | mem=0.66 MB
[admissions_train] shape=(5000, 9) | columns=9 | mem=0.57 MB
[admissions_test] shape=(5000, 8) | columns=8 | mem=0.53 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.7675

# Iteration 3
- 489.3183

In [6]:
import os, sys, re, math, json, warnings, gc
from pathlib import Path
from typing import Tuple, Dict, Optional, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
SEED = 42
np.random.seed(SEED)

DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"

TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR     = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"
OUT_SUB_PATH    = r"D:\AgentDs\agent_ds_healthcare\submission.csv"

print("=== ITERATION 3: Baseline Quantile + Residual CatBoost (+ shift weighting) ===")
print("Main goals vs last run:")
print("  - Fix CatBoost GPU errors (remove rsm); enable stable GPU training.")
print("  - Add a robust linear/quantile baseline and train CatBoost on residuals (anchors low tail).")
print("  - Add domain-shift sample weights via train-vs-test classifier (covariate shift).")
print("  - Use fold-bagging predictions (often improves LB vs single full-train fit).")
print("  - Ensemble with constraints to avoid 'min pred too high' pathologies.\n")
# Refs (light web research; short):
#  - CatBoost regression losses incl. MAE/Quantile: https://catboost.ai/docs/en/concepts/loss-functions-regression
#  - CatBoost common params + GPU notes: https://catboost.ai/docs/en/references/training-parameters/common
#  - Health expenditure heavy tails motivate robust losses/log: https://www.sciencedirect.com/science/article/pii/S0167629624000572
#  - XGBoost docs mention reg:quantileerror etc. (not used here): https://xgboost.readthedocs.io/en/stable/parameter.html

# =========================
# Helpers
# =========================
def safe_read_csv(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(p)

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def quantiles(x, qs=(0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def ensure_cat_as_str(df: pd.DataFrame, cols: List[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")

def fill_prefix_numeric_zero(df: pd.DataFrame, prefixes: List[str], exclude: Optional[List[str]] = None) -> None:
    exclude = set(exclude or [])
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes) and (c not in exclude)]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

def median_fill_numeric(train_df: pd.DataFrame, test_df: pd.DataFrame, numeric_cols: List[str]) -> None:
    for c in numeric_cols:
        med = pd.to_numeric(train_df[c], errors="coerce").median()
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(med)
        test_df[c]  = pd.to_numeric(test_df[c],  errors="coerce").fillna(med)

def detect_gpu_catboost() -> bool:
    try:
        from catboost.utils import get_gpu_device_count
        return get_gpu_device_count() > 0
    except Exception:
        return False

# =========================
# Receipts parsing helpers (fallback only)
# =========================
def extract_text_from_pdf(path: Path) -> str:
    try:
        import pdfplumber
        with pdfplumber.open(str(path)) as pdf:
            return "\n".join((p.extract_text() or "") for p in pdf.pages)
    except Exception:
        try:
            import fitz
            doc = fitz.open(str(path))
            text = "\n".join(page.get_text("text") for page in doc)
            doc.close()
            return text
        except Exception:
            return ""

def parse_receipt_text(text: str) -> Tuple[Dict, pd.DataFrame]:
    pid_re = re.compile(r"Patient ID:\s*(\d+)")
    zip_ins_re = re.compile(r"ZIP3:\s*([0-9A-Za-z]+)\s+Insurance:\s*([A-Za-z_]+)")
    line_re = re.compile(r"^(?P<code>\S+)\s+(?P<desc>.*?)\s+(?P<qty>\d+)\s+(?P<unit>[\d,]+\.\d{2})\s+(?P<line_total>[\d,]+\.\d{2})$")
    total_re = re.compile(r"^TOTAL\s+(?P<total>[\d,]+\.\d{2})$")

    pid = None
    zip3 = None
    insurance = None
    total = np.nan

    m = pid_re.search(text)
    if m:
        pid = int(m.group(1))
    m = zip_ins_re.search(text)
    if m:
        zip3 = standardize_zip3(m.group(1))
        insurance = m.group(2)

    rows = []
    for raw in text.splitlines():
        raw = raw.strip()
        if not raw:
            continue
        mt = total_re.match(raw)
        if mt:
            total = float(mt.group("total").replace(",", ""))
            continue
        ml = line_re.match(raw)
        if ml:
            d = ml.groupdict()
            rows.append({
                "patient_id": pid,
                "code": d["code"],
                "desc": d["desc"],
                "qty": int(d["qty"]),
                "unit": float(d["unit"].replace(",", "")),
                "line_total": float(d["line_total"].replace(",", "")),
            })

    header = {"patient_id": pid, "zip3_pdf": zip3, "insurance_pdf": insurance, "pdf_total": total}
    return header, pd.DataFrame(rows)

def parse_some_receipts_from_pdfs(pdf_dir: Path, patient_ids: List[int], max_to_parse: int = 5000) -> Tuple[pd.DataFrame, pd.DataFrame]:
    headers, items = [], []
    parsed = 0
    missing = 0
    for pid in patient_ids:
        pdf_path = pdf_dir / f"receipt_{pid}.pdf"
        if not pdf_path.exists():
            missing += 1
            continue
        txt = extract_text_from_pdf(pdf_path)
        h, df = parse_receipt_text(txt)
        headers.append(h)
        if df is not None and len(df):
            items.append(df)
        parsed += 1
        if parsed >= max_to_parse:
            break
    headers_df = pd.DataFrame(headers)
    lineitems_df = pd.concat(items, ignore_index=True) if items else pd.DataFrame(columns=["patient_id","code","desc","qty","unit","line_total"])
    print(f"[receipts:fallback] parsed={parsed} headers={headers_df.shape} lineitems={lineitems_df.shape} missing_pdfs={missing}")
    return headers_df, lineitems_df

def extract_receipt_dfs(obj) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    headers_df, lineitems_df = None, None
    if isinstance(obj, dict):
        for hk in ["headers_df","headers","header_df","header"]:
            if hk in obj and isinstance(obj[hk], pd.DataFrame):
                headers_df = obj[hk]
                break
        for lk in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if lk in obj and isinstance(obj[lk], pd.DataFrame):
                lineitems_df = obj[lk]
                break
        if lineitems_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and {"patient_id","code"}.issubset(set(v.columns)):
                    lineitems_df = v
                    break
        if headers_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and ("patient_id" in v.columns) and (v is not lineitems_df):
                    headers_df = v
                    break
    elif isinstance(obj, (tuple, list)):
        dfs = [x for x in obj if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if {"patient_id","code"}.issubset(set(df.columns)):
                lineitems_df = df
            elif "patient_id" in df.columns:
                headers_df = df
    elif isinstance(obj, pd.DataFrame):
        if {"patient_id","code"}.issubset(set(obj.columns)):
            lineitems_df = obj
        elif "patient_id" in obj.columns:
            headers_df = obj
    return headers_df, lineitems_df

# =========================
# Receipt features (patient-level)
# =========================
def build_receipt_features(headers_df: Optional[pd.DataFrame],
                          lineitems_df: Optional[pd.DataFrame],
                          max_codes_dense: int = 120) -> pd.DataFrame:
    if (lineitems_df is None) or (not isinstance(lineitems_df, pd.DataFrame)) or (len(lineitems_df) == 0):
        if headers_df is not None and isinstance(headers_df, pd.DataFrame) and ("patient_id" in headers_df.columns):
            df = headers_df.copy()
            return df.copy()
        return pd.DataFrame(columns=["patient_id"])

    li = lineitems_df.copy()

    if "code" not in li.columns:
        for alt in ["cpt","cpt_code","hcpcs","proc_code"]:
            if alt in li.columns:
                li["code"] = li[alt]
                break
    if "line_total" not in li.columns:
        for alt in ["line_total_usd","total","amount","line_cost","line_total_$","sum_items"]:
            if alt in li.columns:
                li["line_total"] = li[alt]
                break
    if "qty" not in li.columns:
        for alt in ["quantity","q"]:
            if alt in li.columns:
                li["qty"] = li[alt]
                break
    if "unit" not in li.columns:
        if "line_total" in li.columns and "qty" in li.columns:
            li["unit"] = pd.to_numeric(li["line_total"], errors="coerce") / pd.to_numeric(li["qty"], errors="coerce").replace(0, np.nan)
        else:
            li["unit"] = np.nan

    if "patient_id" not in li.columns or "code" not in li.columns:
        return pd.DataFrame(columns=["patient_id"])

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li["code"].astype(str).str.strip().str.upper()
    li["qty"] = pd.to_numeric(li["qty"], errors="coerce").fillna(1).astype(float)
    li["unit"] = pd.to_numeric(li["unit"], errors="coerce")
    li["line_total"] = pd.to_numeric(li["line_total"], errors="coerce").fillna(0.0).astype(float)

    must_codes = {
        "31500","36556","36620","92950",
        "70450","74177","84484","G0378",
        "99281","99282","99283","99284","99285",
        "99291","99292",
        "85025","87070","71045"
    }

    uniq_codes = li["code"].nunique()
    if uniq_codes <= max_codes_dense:
        codes = sorted(set(li["code"].unique().tolist()) | must_codes)
    else:
        code_cost = li.groupby("code")["line_total"].sum().sort_values(ascending=False)
        top_cost = code_cost.head(max_codes_dense).index.tolist()
        top_cnt = li["code"].value_counts().head(max_codes_dense).index.tolist()
        codes = sorted(set(top_cost + top_cnt + list(must_codes)))

    med_unit_by_code = li.groupby("code")["unit"].median()
    li["unit_med_code"] = li["code"].map(med_unit_by_code).astype(float)
    li["deflated_line_total"] = li["qty"] * li["unit_med_code"]

    g = li.groupby("patient_id")
    agg = g.agg(
        pdf_total_line_cost=("line_total","sum"),
        pdf_deflated_total=("deflated_line_total","sum"),
        pdf_n_line_items=("code","size"),
        pdf_total_qty=("qty","sum"),
        pdf_n_unique_codes=("code","nunique"),
        pdf_line_max=("line_total","max"),
        pdf_line_mean=("line_total","mean"),
        pdf_line_std=("line_total", lambda x: float(np.std(x, ddof=0))),
        pdf_line_median=("line_total","median"),
    ).reset_index()

    denom = agg["pdf_total_line_cost"].replace(0, np.nan)

    li_sel = li[li["code"].isin(codes)].copy()
    piv_cnt  = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="size", fill_value=0)
    piv_cost = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="sum", fill_value=0.0)
    piv_cnt.columns  = [f"pdf_cnt_{c}"  for c in piv_cnt.columns.astype(str)]
    piv_cost.columns = [f"pdf_cost_{c}" for c in piv_cost.columns.astype(str)]
    wide = piv_cnt.join(piv_cost, how="outer").reset_index()

    out = agg.merge(wide, on="patient_id", how="left")
    fill_prefix_numeric_zero(out, prefixes=["pdf_"])

    # per-code shares (composition)
    denom2 = out["pdf_total_line_cost"].replace(0, np.nan)
    for c in codes:
        col = f"pdf_cost_{c}"
        out[f"pdf_share_{c}"] = (out[col] / denom2).fillna(0.0) if col in out.columns else 0.0

    # high-acuity flags (per prompt)
    code_flags = {
        "has_intub_31500": "31500",
        "has_cvc_36556": "36556",
        "has_artline_36620": "36620",
        "has_cpr_92950": "92950",
        "has_ct_head_70450": "70450",
        "has_ct_abdpel_74177": "74177",
        "has_troponin_84484": "84484",
        "has_obs_G0378": "G0378",
        "has_99285": "99285",
    }
    for fname, code in code_flags.items():
        cnt_col = f"pdf_cnt_{code}"
        out[fname] = (out[cnt_col] > 0).astype(int) if cnt_col in out.columns else 0

    crit_cols = [c for c in ["99291","99292"] if f"pdf_cnt_{c}" in out.columns]
    if crit_cols:
        out["has_critical_care"] = (out[[f"pdf_cnt_{c}" for c in crit_cols]].sum(axis=1) > 0).astype(int)
    else:
        out["has_critical_care"] = 0

    out["severe_proc_score"] = (
        out.get("has_intub_31500", 0)
        + out.get("has_cvc_36556", 0)
        + out.get("has_artline_36620", 0)
        + out.get("has_cpr_92950", 0)
        + out.get("has_critical_care", 0)
    ).astype(int)

    # E/M level features
    em_codes = ["99281","99282","99283","99284","99285"]
    em_level_map = {"99281":1, "99282":2, "99283":3, "99284":4, "99285":5}
    em = li[li["code"].isin(em_codes)].copy()
    if len(em):
        em["em_level"] = em["code"].map(em_level_map).astype(float)
        em_agg = em.groupby("patient_id").agg(
            pdf_em_level_mean=("em_level","mean"),
            pdf_em_level_max=("em_level","max"),
            pdf_em_high_frac=("em_level", lambda x: float(np.mean(np.asarray(x) >= 4))),
        ).reset_index()
        out = out.merge(em_agg, on="patient_id", how="left")
    for c in ["pdf_em_level_mean","pdf_em_level_max","pdf_em_high_frac"]:
        if c not in out.columns:
            out[c] = 0.0
    out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_high_frac"]] = out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_high_frac"]].fillna(0.0)

    # distribution stats on costs by code (entropy, gini, top1 share)
    cost_code_cols = [c for c in out.columns if c.startswith("pdf_cost_")]
    if cost_code_cols:
        mat = out[cost_code_cols].to_numpy(dtype=float)
        row_sum = mat.sum(axis=1, keepdims=True)
        p = np.divide(mat, row_sum, out=np.zeros_like(mat), where=row_sum > 0)
        ent = -(p * np.log(np.clip(p, 1e-12, 1.0))).sum(axis=1)
        ent_norm = ent / np.log(max(2, mat.shape[1]))
        out["pdf_code_cost_entropy_norm"] = ent_norm

        sorted_mat = np.sort(mat, axis=1)
        sum_x = sorted_mat.sum(axis=1)
        top1 = sorted_mat[:, -1]
        out["pdf_cost_top1_share"] = np.divide(top1, sum_x, out=np.zeros_like(top1), where=sum_x > 0)

        n = sorted_mat.shape[1]
        i = np.arange(1, n + 1, dtype=float)
        weighted = (sorted_mat * i.reshape(1, -1)).sum(axis=1)
        gini = np.zeros_like(sum_x, dtype=float)
        mask = sum_x > 0
        gini[mask] = (2.0 * weighted[mask]) / (n * sum_x[mask]) - (n + 1) / n
        out["pdf_cost_gini"] = gini
    else:
        out["pdf_code_cost_entropy_norm"] = 0.0
        out["pdf_cost_top1_share"] = 0.0
        out["pdf_cost_gini"] = 0.0

    # Merge header totals if present
    if headers_df is not None and isinstance(headers_df, pd.DataFrame) and ("patient_id" in headers_df.columns):
        cols = ["patient_id"] + [c for c in ["pdf_total","sum_items","zip3_pdf","insurance_pdf"] if c in headers_df.columns]
        hd = headers_df[cols].copy()
        out = out.merge(hd, on="patient_id", how="left")
        if "pdf_total" in out.columns:
            out["pdf_total_missing"] = out["pdf_total"].isna().astype(int)
            out["pdf_total"] = pd.to_numeric(out["pdf_total"], errors="coerce").fillna(0.0)
        if "sum_items" in out.columns:
            out["sum_items"] = pd.to_numeric(out["sum_items"], errors="coerce").fillna(0.0)
        if "zip3_pdf" in out.columns:
            out["zip3_pdf"] = out["zip3_pdf"].apply(standardize_zip3)
        if "insurance_pdf" in out.columns:
            out["insurance_pdf"] = out["insurance_pdf"].astype("string").fillna("Unknown")

    return out

# =========================
# Admissions features (patient-level)
# =========================
def build_admissions_features(adm_tr: pd.DataFrame, adm_te: pd.DataFrame) -> pd.DataFrame:
    adm_tr = adm_tr.drop(columns=["readmit_30d"], errors="ignore")
    adm_all = pd.concat([adm_tr, adm_te], ignore_index=True)

    if "patient_id" not in adm_all.columns:
        return pd.DataFrame(columns=["patient_id"])

    adm_all["patient_id"] = pd.to_numeric(adm_all["patient_id"], errors="coerce").astype("Int64")
    adm_all = adm_all.dropna(subset=["patient_id"]).copy()
    adm_all["patient_id"] = adm_all["patient_id"].astype(int)

    num_cols = ["los_days","acuity_emergent","charlson_band","ed_visits_6m","discharge_weekday"]
    for c in num_cols:
        if c in adm_all.columns:
            adm_all[c] = pd.to_numeric(adm_all[c], errors="coerce")

    g = adm_all.groupby("patient_id")
    out = pd.DataFrame({"patient_id": g.size().index, "adm_n": g.size().values})

    for c in num_cols:
        if c in adm_all.columns:
            out[f"adm_{c}_mean"] = g[c].mean().values
            out[f"adm_{c}_max"]  = g[c].max().values
            out[f"adm_{c}_sum"]  = g[c].sum().values
            out[f"adm_{c}_std"]  = g[c].std(ddof=0).values

    if "primary_dx" in adm_all.columns:
        dx_ct = pd.crosstab(adm_all["patient_id"], adm_all["primary_dx"])
        dx_ct.columns = [f"adm_dx_cnt_{str(c)}" for c in dx_ct.columns]
        dx_ct = dx_ct.reset_index()
        out = out.merge(dx_ct, on="patient_id", how="left")

        dx_mode = g["primary_dx"].agg(lambda x: x.value_counts().index[0] if len(x) else "Unknown").reset_index()
        dx_mode.columns = ["patient_id", "adm_primary_dx_mode"]
        out = out.merge(dx_mode, on="patient_id", how="left")

    for c in out.columns:
        if c in ["patient_id", "adm_primary_dx_mode"]:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    if "adm_primary_dx_mode" in out.columns:
        out["adm_primary_dx_mode"] = out["adm_primary_dx_mode"].astype("string").fillna("Unknown")
    return out

# =========================
# Load data
# =========================
train = safe_read_csv(TRAIN_PATH)
test  = safe_read_csv(TEST_PATH)
patients = safe_read_csv(PATIENTS_PATH)
adm_tr = safe_read_csv(ADM_TRAIN_PATH)
adm_te = safe_read_csv(ADM_TEST_PATH)

for df in [train, test, patients, adm_tr, adm_te]:
    if "patient_id" in df.columns:
        df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

if "zip3" in patients.columns:
    patients["zip3"] = patients["zip3"].apply(standardize_zip3)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)

print("\n[target summary]")
print(train["ed_cost_next3y_usd"].describe().to_string())
print("\n[prior cost summary]")
print(train["prior_ed_cost_5y_usd"].describe().to_string())

# =========================
# Build admissions + receipts
# =========================
adm_feat = build_admissions_features(adm_tr, adm_te)
print(f"\n[admissions] features: {adm_feat.shape}")

receipt_feat = pd.DataFrame(columns=["patient_id"])
joblib_path = Path(RECEIPTS_JOBLIB_PATH)
pdf_dir = Path(RECEIPTS_PDF_DIR)
all_patient_ids = pd.concat([train["patient_id"], test["patient_id"]], ignore_index=True).astype(int).unique().tolist()

joblib_load = None
try:
    from joblib import load as joblib_load
except Exception:
    joblib_load = None

headers_df = None
lineitems_df = None

if joblib_path.exists() and (joblib_load is not None):
    try:
        obj = joblib_load(joblib_path)
        headers_df, lineitems_df = extract_receipt_dfs(obj)

        if isinstance(obj, pd.DataFrame) and ("patient_id" in obj.columns) and ("code" not in obj.columns) and (obj["patient_id"].nunique() == len(obj)):
            receipt_feat = obj.copy()
            print(f"\n[receipts] joblib is patient-level DF: {receipt_feat.shape}")
        else:
            receipt_feat = build_receipt_features(headers_df, lineitems_df, max_codes_dense=120)
            print(f"\n[receipts] built from joblib lineitems: {receipt_feat.shape}")
    except Exception as e:
        print(f"\n[receipts] joblib load failed: {e}")
        headers_df, lineitems_df = None, None

if (receipt_feat is None) or (len(receipt_feat) == 0):
    if pdf_dir.exists():
        print("\n[receipts] fallback: parsing PDFs (slow)...")
        h2, li2 = parse_some_receipts_from_pdfs(pdf_dir, all_patient_ids, max_to_parse=len(all_patient_ids))
        receipt_feat = build_receipt_features(h2, li2, max_codes_dense=120)
        print(f"[receipts] built from PDFs: {receipt_feat.shape}")
    else:
        print("\n[receipts] WARNING: no receipts features available. Proceeding without receipts.")
        receipt_feat = pd.DataFrame({"patient_id": all_patient_ids})

# ensure unique patient_id
if "patient_id" in receipt_feat.columns:
    receipt_feat["patient_id"] = pd.to_numeric(receipt_feat["patient_id"], errors="coerce").astype("Int64")
    receipt_feat = receipt_feat.dropna(subset=["patient_id"]).copy()
    receipt_feat["patient_id"] = receipt_feat["patient_id"].astype(int)
    receipt_feat = receipt_feat.drop_duplicates(subset=["patient_id"], keep="last")

cov = receipt_feat["patient_id"].nunique() / len(all_patient_ids) if "patient_id" in receipt_feat.columns else 0.0
print(f"[receipts] coverage: {receipt_feat['patient_id'].nunique()}/{len(all_patient_ids)} = {cov:.3f}")

# =========================
# Merge into modeling DF
# =========================
def build_model_df(ed_df: pd.DataFrame, patients: pd.DataFrame,
                   adm_feat: pd.DataFrame, receipt_feat: pd.DataFrame) -> pd.DataFrame:
    df = ed_df.merge(patients, on="patient_id", how="left")
    if adm_feat is not None and len(adm_feat):
        df = df.merge(adm_feat, on="patient_id", how="left")
    if receipt_feat is not None and len(receipt_feat):
        df = df.merge(receipt_feat, on="patient_id", how="left", suffixes=("", "_rcpt"))

    fill_prefix_numeric_zero(df, prefixes=["adm_"], exclude=["adm_primary_dx_mode"])
    fill_prefix_numeric_zero(df, prefixes=["pdf_"])
    fill_prefix_numeric_zero(df, prefixes=["has_"])
    fill_prefix_numeric_zero(df, prefixes=["severe_"])

    # standardize receipt cats if present
    for c in ["zip3_pdf","insurance_pdf"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")
            if c == "zip3_pdf":
                df[c] = df[c].apply(standardize_zip3)

    if "adm_primary_dx_mode" in df.columns:
        df["adm_primary_dx_mode"] = df["adm_primary_dx_mode"].astype("string").fillna("Unknown")
    return df

train_df = build_model_df(train, patients, adm_feat, receipt_feat)
test_df  = build_model_df(test,  patients, adm_feat, receipt_feat)

# derived features
def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["prior_cost"] = pd.to_numeric(df.get("prior_ed_cost_5y_usd", np.nan), errors="coerce").fillna(0.0)
    df["prior_visits"] = pd.to_numeric(df.get("prior_ed_visits_5y", np.nan), errors="coerce").fillna(0.0)
    df["log_prior_cost"] = np.log1p(df["prior_cost"])
    df["log_prior_visits"] = np.log1p(df["prior_visits"])
    df["cost_per_visit"] = df["prior_cost"] / np.maximum(1.0, df["prior_visits"])
    df["log_cost_per_visit"] = np.log1p(df["cost_per_visit"])
    # modest cap helps prevent huge leverage
    df["prior_cost_cap25k"] = np.minimum(df["prior_cost"], 25000.0)
    df["log_prior_cost_cap25k"] = np.log1p(df["prior_cost_cap25k"])
    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce").fillna(df["age"].median())
        df["age_x_logpriorcost"] = df["age"] * df["log_prior_cost"]
    if "adm_charlson_band_mean" in df.columns:
        df["logpriorcost_x_charlson"] = df["log_prior_cost"] * pd.to_numeric(df["adm_charlson_band_mean"], errors="coerce").fillna(0.0)
    if "severe_proc_score" in df.columns:
        df["logpriorcost_x_severe"] = df["log_prior_cost"] * pd.to_numeric(df["severe_proc_score"], errors="coerce").fillna(0.0)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

train_df = add_derived(train_df)
test_df  = add_derived(test_df)

print("\n[merged]")
log_shape("train_df", train_df)
log_shape("test_df", test_df)

# compare train vs test distributions (key columns)
key_cols = [c for c in [
    "prior_ed_cost_5y_usd","prior_ed_visits_5y","age","adm_n","adm_charlson_band_mean","severe_proc_score",
    "pdf_n_unique_codes","pdf_n_line_items","pdf_cost_gini","pdf_cost_top1_share","pdf_code_cost_entropy_norm"
] if c in train_df.columns and c in test_df.columns]
print("\n[train vs test feature shift quick stats]")
for c in key_cols:
    tr = pd.to_numeric(train_df[c], errors="coerce")
    te = pd.to_numeric(test_df[c], errors="coerce")
    print(f"  {c:28s} train_mean={tr.mean():8.2f}  test_mean={te.mean():8.2f} | train_p95={np.nanpercentile(tr,95):8.2f} test_p95={np.nanpercentile(te,95):8.2f}")

# =========================
# Prepare X/y
# =========================
TARGET_COL = "ed_cost_next3y_usd"
y = pd.to_numeric(train_df[TARGET_COL], errors="coerce").values.astype(float)

# keep IDs for output
test_patient_id = test_df["patient_id"].astype(int).values

X_full = train_df.drop(columns=[TARGET_COL])
X_test_full = test_df.copy()

# categorical columns for tree models
cat_cols = [c for c in [
    "primary_chronic","sex","insurance","zip3",
    "zip3_pdf","insurance_pdf",
    "adm_primary_dx_mode",
] if c in X_full.columns and c in X_test_full.columns]

ensure_cat_as_str(X_full, cat_cols)
ensure_cat_as_str(X_test_full, cat_cols)

# drop patient_id from ML features
X_full = X_full.drop(columns=["patient_id"], errors="ignore")
X_test_full = X_test_full.drop(columns=["patient_id"], errors="ignore")

# numeric fill
num_cols = [c for c in X_full.columns if c not in cat_cols]
median_fill_numeric(X_full, X_test_full, num_cols)

# =========================
# Baseline Quantile Model Features (small, robust)
# =========================
base_cat = [c for c in ["primary_chronic","insurance","sex"] if c in train_df.columns and c in test_df.columns]
base_num = []
# baseline numeric features (keep small & stable)
for c in ["log_prior_cost","log_prior_visits","age","adm_n","adm_charlson_band_mean","severe_proc_score",
          "has_critical_care","has_intub_31500","has_cvc_36556","has_cpr_92950",
          "pdf_em_level_max","pdf_n_unique_codes","pdf_n_line_items","pdf_cost_gini","pdf_cost_top1_share","pdf_code_cost_entropy_norm"]:
    if c in train_df.columns and c in test_df.columns:
        base_num.append(c)

base_train = train_df[base_cat + base_num].copy()
base_test  = test_df[base_cat + base_num].copy()
ensure_cat_as_str(base_train, base_cat)
ensure_cat_as_str(base_test, base_cat)

# numeric fill baseline
for c in base_num:
    base_train[c] = pd.to_numeric(base_train[c], errors="coerce")
    base_test[c]  = pd.to_numeric(base_test[c], errors="coerce")
    med = float(base_train[c].median())
    base_train[c] = base_train[c].fillna(med)
    base_test[c]  = base_test[c].fillna(med)

# fixed one-hot columns using combined (no target leakage)
base_all = pd.concat([base_train, base_test], ignore_index=True)
base_all_enc = pd.get_dummies(base_all, columns=base_cat, dummy_na=True)
base_train_enc = base_all_enc.iloc[:len(base_train)].reset_index(drop=True)
base_test_enc  = base_all_enc.iloc[len(base_train):].reset_index(drop=True)

print("\n[baseline design]")
print(f"  baseline numeric={len(base_num)} cat(onehot)={len(base_all_enc.columns)-len(base_num)} total={base_train_enc.shape[1]}")
print("  baseline_num_cols:", base_num)
print("  baseline_cat_cols:", base_cat)

# =========================
# Domain shift weights (train-vs-test classifier)
# =========================
use_gpu = detect_gpu_catboost()
print(f"\n[catboost] GPU available: {use_gpu}")

try:
    from catboost import CatBoostClassifier, CatBoostRegressor, Pool
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
    from catboost import CatBoostClassifier, CatBoostRegressor, Pool

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X_dom = pd.concat([X_full, X_test_full], ignore_index=True)
y_dom = np.array([0]*len(X_full) + [1]*len(X_test_full), dtype=int)

# ensure categorical types
ensure_cat_as_str(X_dom, cat_cols)
dom_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

dom_oof = np.zeros(len(X_dom), dtype=float)
dom_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=1500 if use_gpu else 900,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=4,
    random_strength=1.0,
    bagging_temperature=1.0,
    allow_writing_files=False,
    verbose=False,
    task_type="GPU" if use_gpu else "CPU",
)
if not use_gpu:
    dom_params["thread_count"] = -1

for f, (tr_i, va_i) in enumerate(dom_skf.split(X_dom, y_dom), 1):
    Xtr, Xva = X_dom.iloc[tr_i], X_dom.iloc[va_i]
    ytr, yva = y_dom[tr_i], y_dom[va_i]
    tr_pool = Pool(Xtr, ytr, cat_features=cat_cols) if cat_cols else Pool(Xtr, ytr)
    va_pool = Pool(Xva, yva, cat_features=cat_cols) if cat_cols else Pool(Xva, yva)
    clf = CatBoostClassifier(**dom_params, random_seed=SEED+f)
    clf.fit(tr_pool, eval_set=va_pool, use_best_model=True, early_stopping_rounds=200)
    dom_oof[va_i] = clf.predict_proba(Xva)[:, 1]

dom_auc = roc_auc_score(y_dom, dom_oof)
print(f"[domain shift] Train-vs-Test AUC (OOF): {dom_auc:.4f} (0.50 means no detectable shift)")

# convert to sample weights for TRAIN rows
p_train = dom_oof[:len(X_full)]
p_train = np.clip(p_train, 0.05, 0.95)
w_raw = p_train / (1.0 - p_train)
w_raw = np.clip(w_raw, 0.25, 4.0)

# shrink weights to avoid extremes (gamma in [0,1])
gamma = 0.5
w = 1.0 + gamma*(w_raw - 1.0)
w = w / np.mean(w)

print(f"[domain weights] mean={w.mean():.3f} std={w.std():.3f} min={w.min():.3f} p5={np.quantile(w,0.05):.3f} p95={np.quantile(w,0.95):.3f} max={w.max():.3f}")

# =========================
# CV setup
# =========================
# stratify to stabilize folds
try:
    strat = train_df["primary_chronic"].astype(str) + "_" + pd.qcut(train_df["prior_ed_cost_5y_usd"], q=8, duplicates="drop").astype(str)
except Exception:
    strat = train_df["primary_chronic"].astype(str)

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# residual feature set: remove strong "total cost" columns (baseline already uses prior-cost)
drop_resid_prefixes = ("pdf_cost_", "pdf_pref1_cost_", "pdf_pref2_cost_")
drop_resid_contains = [
    "prior_ed_cost_5y_usd","prior_cost","prior_cost_cap","log_prior_cost","log_prior_cost_cap",
    "baseline_next3y","baseline_next3y_log","cost_per_year","cost_per_visit","log_cost_per_visit",
    "pdf_total_line_cost","pdf_deflated_total","pdf_total","sum_items"
]
resid_drop_cols = set()
for c in X_full.columns:
    if any(c.startswith(p) for p in drop_resid_prefixes):
        resid_drop_cols.add(c)
    if any(s in c for s in drop_resid_contains):
        resid_drop_cols.add(c)
# do NOT drop composition shares & flags
resid_drop_cols = {c for c in resid_drop_cols if not c.startswith("pdf_share_")}
resid_drop_cols = {c for c in resid_drop_cols if not c.startswith("has_")}

X_resid = X_full.drop(columns=list(resid_drop_cols), errors="ignore").copy()
X_resid_test = X_test_full.drop(columns=list(resid_drop_cols), errors="ignore").copy()

# cat cols for residual set
cat_cols_resid = [c for c in cat_cols if c in X_resid.columns]

print("\n[feature sets]")
print(f"  Full features:  {X_full.shape[1]} (cat={len(cat_cols)})")
print(f"  Resid features: {X_resid.shape[1]} (cat={len(cat_cols_resid)})")
print(f"  Resid dropped:  {len(resid_drop_cols)}")

# =========================
# Models
# =========================
cb_common = dict(
    allow_writing_files=False,
    verbose=False,
    task_type="GPU" if use_gpu else "CPU",
)
if not use_gpu:
    cb_common["thread_count"] = -1

# IMPORTANT: do NOT use rsm on GPU (it caused your previous error). Keep it absent.
models = [
    dict(
        name="cb_resid_mae",
        target="residual",
        params=dict(cb_common,
                    loss_function="MAE",
                    eval_metric="MAE",
                    iterations=6000 if use_gpu else 2500,
                    learning_rate=0.035,
                    depth=6,
                    l2_leaf_reg=18,
                    bagging_temperature=1.0,
                    random_strength=1.0,
                    min_data_in_leaf=24),
    ),
    dict(
        name="cb_resid_q50",
        target="residual",
        params=dict(cb_common,
                    loss_function="Quantile:alpha=0.5",
                    eval_metric="MAE",
                    iterations=7000 if use_gpu else 2800,
                    learning_rate=0.03,
                    depth=6,
                    l2_leaf_reg=20,
                    bagging_temperature=1.2,
                    random_strength=1.0,
                    min_data_in_leaf=24),
    ),
    dict(
        name="cb_full_logrmse",
        target="full_log",
        params=dict(cb_common,
                    loss_function="RMSE",
                    eval_metric="RMSE",
                    iterations=9000 if use_gpu else 3200,
                    learning_rate=0.03,
                    depth=7,
                    l2_leaf_reg=8,
                    bagging_temperature=0.8,
                    random_strength=1.0,
                    min_data_in_leaf=24),
    ),
]

# Baseline model: sklearn QuantileRegressor
try:
    from sklearn.linear_model import QuantileRegressor
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    from sklearn.linear_model import QuantileRegressor

def make_qr():
    # solver='highs' is fast when available; fallback if unsupported
    try:
        return QuantileRegressor(quantile=0.5, alpha=0.001, solver="highs")
    except TypeError:
        return QuantileRegressor(quantile=0.5, alpha=0.001)

# =========================
# CV training + fold bagging preds
# =========================
oof = {m["name"]: np.zeros(len(X_full), dtype=float) for m in models}
test_bag = {m["name"]: np.zeros(len(X_test_full), dtype=float) for m in models}
oof_base = np.zeros(len(X_full), dtype=float)
test_base_bag = np.zeros(len(X_test_full), dtype=float)

fold_logs = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, strat), 1):
    y_tr, y_va = y[tr_idx], y[va_idx]

    # baseline quantile reg (fit on train fold)
    qr = make_qr()
    qr.fit(base_train_enc.iloc[tr_idx], y_tr)
    base_pred_tr = qr.predict(base_train_enc.iloc[tr_idx])
    base_pred_va = qr.predict(base_train_enc.iloc[va_idx])
    base_pred_te = qr.predict(base_test_enc)

    base_pred_tr = np.clip(base_pred_tr, 0.0, None)
    base_pred_va = np.clip(base_pred_va, 0.0, None)
    base_pred_te = np.clip(base_pred_te, 0.0, None)

    oof_base[va_idx] = base_pred_va
    test_base_bag += base_pred_te / N_SPLITS

    base_mae = mae(y_va, base_pred_va)

    # residual targets
    y_tr_res = y_tr - base_pred_tr
    y_va_res = y_va - base_pred_va

    # fold weights for training (domain shift)
    w_tr = w[tr_idx]

    # model loop
    fold_line = {"fold": fold, "baseline_mae": base_mae}
    for m in models:
        name = m["name"]
        params = m["params"].copy()
        params["random_seed"] = SEED + fold

        if m["target"] == "residual":
            Xtr = X_resid.iloc[tr_idx]
            Xva = X_resid.iloc[va_idx]
            Xte = X_resid_test
            cat_use = cat_cols_resid

            tr_pool = Pool(Xtr, y_tr_res, weight=w_tr, cat_features=cat_use) if cat_use else Pool(Xtr, y_tr_res, weight=w_tr)
            va_pool = Pool(Xva, y_va_res, cat_features=cat_use) if cat_use else Pool(Xva, y_va_res)

            reg = CatBoostRegressor(**params)
            reg.fit(tr_pool, eval_set=va_pool, use_best_model=True, early_stopping_rounds=250)

            pred_va_res = reg.predict(Xva)
            pred_te_res = reg.predict(Xte)

            pred_va = base_pred_va + pred_va_res
            pred_te = base_pred_te + pred_te_res

        elif m["target"] == "full_log":
            Xtr = X_full.iloc[tr_idx]
            Xva = X_full.iloc[va_idx]
            Xte = X_test_full
            cat_use = cat_cols

            y_tr_log = np.log1p(y_tr)
            y_va_log = np.log1p(y_va)

            tr_pool = Pool(Xtr, y_tr_log, weight=w_tr, cat_features=cat_use) if cat_use else Pool(Xtr, y_tr_log, weight=w_tr)
            va_pool = Pool(Xva, y_va_log, cat_features=cat_use) if cat_use else Pool(Xva, y_va_log)

            reg = CatBoostRegressor(**params)
            reg.fit(tr_pool, eval_set=va_pool, use_best_model=True, early_stopping_rounds=250)

            pred_va = np.expm1(reg.predict(Xva))
            pred_te = np.expm1(reg.predict(Xte))

        else:
            raise ValueError("Unknown target type")

        pred_va = np.clip(pred_va, 0.0, None)
        pred_te = np.clip(pred_te, 0.0, None)

        oof[name][va_idx] = pred_va
        test_bag[name] += pred_te / N_SPLITS

        fold_mae_val = mae(y_va, pred_va)
        fold_line[name] = fold_mae_val

    fold_logs.append(fold_line)
    msg = " | ".join([f"{k}={fold_line[k]:.2f}" for k in ["baseline_mae"] + [m["name"] for m in models]])
    print(f"[CV] fold {fold}/{N_SPLITS}: {msg}")

# =========================
# OOF summary
# =========================
print("\n[OOF Summary]")
base_oof_mae = mae(y, oof_base)
print(f"  baseline_qr     MAE={base_oof_mae:.4f} | pred_q={quantiles(oof_base, qs=(0,0.01,0.05,0.5,0.95,1.0))}")

for m in models:
    name = m["name"]
    m_mae = mae(y, oof[name])
    print(f"  {name:14s} MAE={m_mae:.4f} | pred_q={quantiles(oof[name], qs=(0,0.01,0.05,0.5,0.95,1.0))}")

# Error diagnostics: top 12 absolute errors for best single model (by OOF MAE)
best_single = min(models, key=lambda mm: mae(y, oof[mm["name"]]))["name"]
abs_err = np.abs(y - oof[best_single])
top_idx = np.argsort(-abs_err)[:12]
diag_cols = [c for c in ["primary_chronic","prior_ed_cost_5y_usd","prior_ed_visits_5y","age","adm_n","adm_charlson_band_mean",
                         "severe_proc_score","has_critical_care","has_intub_31500","has_cvc_36556","has_cpr_92950",
                         "pdf_em_level_max","pdf_n_unique_codes","pdf_cost_gini","pdf_cost_top1_share","pdf_code_cost_entropy_norm"]
             if c in train_df.columns]
diag = train_df.loc[top_idx, ["patient_id", TARGET_COL] + diag_cols].copy()
diag["pred_oof"] = oof[best_single][top_idx]
diag["abs_err"] = abs_err[top_idx]
print(f"\n[Diagnostics] Top 12 abs errors (model={best_single})")
print(diag.sort_values("abs_err", ascending=False).head(12).to_string(index=False))

# =========================
# Ensemble selection (constrained)
# =========================
names = ["baseline_qr"] + [m["name"] for m in models]
oof_mat = np.vstack([oof_base] + [oof[m["name"]] for m in models]).T
test_mat = np.vstack([test_base_bag] + [test_bag[m["name"]] for m in models]).T

# Candidate ensembles
cands = []
cands.append(("mean_all", oof_mat.mean(axis=1), None))
cands.append(("median_all", np.median(oof_mat, axis=1), None))
# simple hand-tuned blends (robust)
# (baseline helps low tail; residuals handle nonlinearity; logrmse adds tail)
w_fixed = np.array([0.25, 0.30, 0.25, 0.20], dtype=float)  # baseline, resid_mae, resid_q50, full_logrmse
w_fixed = w_fixed / w_fixed.sum()
cands.append(("fixed_0.25_0.30_0.25_0.20", oof_mat.dot(w_fixed), w_fixed))

# Random Dirichlet with constraints: baseline weight >=0.18 and predicted 1% quantile <= 900 (avoid too-high floor)
rng = np.random.default_rng(SEED)
best = {"name": None, "mae": 1e18, "pred": None, "w": None, "q01": None}
tries = 6000

y_q20 = np.quantile(y, 0.20)
low_mask = (y <= y_q20)

for _ in range(tries):
    wv = rng.dirichlet(np.ones(oof_mat.shape[1]))
    if wv[0] < 0.18:
        continue
    pred = oof_mat.dot(wv)
    q01 = float(np.quantile(pred, 0.01))
    if q01 > 900.0:
        continue
    # objective emphasizes low-tail a bit (fix "min pred too high")
    obj = mae(y, pred) + 0.20 * mae(y[low_mask], pred[low_mask])
    if obj < best["mae"]:
        best = {"name": "dirichlet_constrained", "mae": obj, "pred": pred, "w": wv, "q01": q01}

if best["pred"] is not None:
    cands.append((best["name"], best["pred"], best["w"]))

# Select best by plain MAE among candidates (but they were constrained already)
best_name, best_pred, best_w = None, None, None
best_mae = 1e18
for nm, pred, wv in cands:
    pred = np.clip(pred, 0.0, None)
    m = mae(y, pred)
    if m < best_mae:
        best_mae, best_name, best_pred, best_w = m, nm, pred, wv

print("\n====================")
print(f"[ENS] chosen={best_name} | OOF_MAE={best_mae:.4f}")
if best_w is not None:
    wdict = {names[i]: float(best_w[i]) for i in range(len(names))}
    wdict = dict(sorted(wdict.items(), key=lambda x: -x[1]))
    print("[ENS] weights:")
    for k, v in wdict.items():
        if v >= 0.02:
            print(f"  {k:16s}: {v:.3f}")
else:
    print("[ENS] weights: n/a")
print("====================\n")
print("[ENS] OOF pred quantiles:", quantiles(best_pred, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))

# Build test ensemble prediction
if best_name == "mean_all":
    pred_test = test_mat.mean(axis=1)
elif best_name == "median_all":
    pred_test = np.median(test_mat, axis=1)
elif best_w is not None:
    pred_test = test_mat.dot(best_w)
else:
    pred_test = test_mat.mean(axis=1)

pred_test = np.clip(pred_test, 0.0, None)

# =========================
# Train final models on full train (requirement) + optional blend
# =========================
# Baseline full
qr_full = make_qr()
qr_full.fit(base_train_enc, y)
base_pred_train_full = np.clip(qr_full.predict(base_train_enc), 0.0, None)
base_pred_test_full  = np.clip(qr_full.predict(base_test_enc), 0.0, None)

# Residual full models
final_preds_full = {"baseline_qr": base_pred_test_full.copy()}
trained_models = {}

# weights for full training
w_full = w

for m in models:
    name = m["name"]
    params = m["params"].copy()
    params["random_seed"] = SEED

    if m["target"] == "residual":
        y_res_full = y - base_pred_train_full
        tr_pool = Pool(X_resid, y_res_full, weight=w_full, cat_features=cat_cols_resid) if cat_cols_resid else Pool(X_resid, y_res_full, weight=w_full)
        reg = CatBoostRegressor(**params)
        reg.fit(tr_pool, verbose=False)
        res_te = reg.predict(X_resid_test)
        pred_te = np.clip(base_pred_test_full + res_te, 0.0, None)
    elif m["target"] == "full_log":
        tr_pool = Pool(X_full, np.log1p(y), weight=w_full, cat_features=cat_cols) if cat_cols else Pool(X_full, np.log1p(y), weight=w_full)
        reg = CatBoostRegressor(**params)
        reg.fit(tr_pool, verbose=False)
        pred_te = np.clip(np.expm1(reg.predict(X_test_full)), 0.0, None)
    else:
        continue

    trained_models[name] = reg
    final_preds_full[name] = pred_te

# Full-train ensemble prediction using same weights (if any)
full_mat = np.vstack([final_preds_full["baseline_qr"]] + [final_preds_full[m["name"]] for m in models]).T
if best_name == "mean_all":
    pred_test_full_ens = full_mat.mean(axis=1)
elif best_name == "median_all":
    pred_test_full_ens = np.median(full_mat, axis=1)
elif best_w is not None:
    pred_test_full_ens = full_mat.dot(best_w)
else:
    pred_test_full_ens = full_mat.mean(axis=1)

pred_test_full_ens = np.clip(pred_test_full_ens, 0.0, None)

# Blend bagged ensemble + full-train ensemble (stability)
blend_alpha = 0.60  # weight on bagged predictions
pred_test_final = np.clip(blend_alpha * pred_test + (1.0 - blend_alpha) * pred_test_full_ens, 0.0, None)

# Feature importance (top 15) from cb_resid_mae if available
try:
    if "cb_resid_mae" in trained_models:
        model_imp = trained_models["cb_resid_mae"]
        imp = model_imp.get_feature_importance()
        fn = X_resid.columns.tolist()
        imp_df = pd.DataFrame({"feature": fn, "importance": imp}).sort_values("importance", ascending=False).head(15)
        print("\n[Top feature importances] (cb_resid_mae on residual features)")
        print(imp_df.to_string(index=False))
except Exception as e:
    print(f"\n[warn] importance failed: {e}")

# =========================
# Write submission
# =========================
sub = pd.DataFrame({
    "patient_id": test_patient_id,
    "ed_cost_next3y_usd": pred_test_final.astype(float),
})[["patient_id", "ed_cost_next3y_usd"]]

print("\n[FINAL sanity checks]")
print("OOF MAE (ensemble):", f"{best_mae:.4f}")
print("ensemble:", best_name, "| bag+full blend alpha:", blend_alpha)
print("submission shape:", sub.shape)
print("submission columns:", list(sub.columns))
print("any NaN preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(np.min(sub["ed_cost_next3y_usd"])), float(np.median(sub["ed_cost_next3y_usd"])), float(np.max(sub["ed_cost_next3y_usd"])))
print("pred quantiles:", quantiles(sub["ed_cost_next3y_usd"].values, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)))
print("Saved to:", OUT_SUB_PATH)

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print("\nPaste back: leaderboard MAE + logs (domain AUC, baseline MAE, chosen ensemble/weights, pred quantiles).")


=== ITERATION 3: Baseline Quantile + Residual CatBoost (+ shift weighting) ===
Main goals vs last run:
  - Fix CatBoost GPU errors (remove rsm); enable stable GPU training.
  - Add a robust linear/quantile baseline and train CatBoost on residuals (anchors low tail).
  - Add domain-shift sample weights via train-vs-test classifier (covariate shift).
  - Use fold-bagging predictions (often improves LB vs single full-train fit).
  - Ensemble with constraints to avoid 'min pred too high' pathologies.

[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB

[target summary]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[prior cost summary]
count     2000.000000
mean      4003.177295
std       4857.839498
min         50.000000
25%        940.982500
50%       2168.710000
75%       5073.502500
max      39746.790000

[ad

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[domain shift] Train-vs-Test AUC (OOF): 0.5215 (0.50 means no detectable shift)
[domain weights] mean=1.000 std=0.111 min=0.654 p5=0.860 p95=1.151 max=2.475

[feature sets]
  Full features:  123 (cat=5)
  Resid features: 93 (cat=5)
  Resid dropped:  30


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


[CV] fold 1/5: baseline_mae=618.43 | cb_resid_mae=602.83 | cb_resid_q50=603.13 | cb_full_logrmse=459.02


...

[CV] fold 4/5: baseline_mae=581.82 | cb_resid_mae=568.54 | cb_resid_q50=568.74 | cb_full_logrmse=466.79


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


[CV] fold 5/5: baseline_mae=619.09 | cb_resid_mae=602.61 | cb_resid_q50=602.87 | cb_full_logrmse=448.31

[OOF Summary]
  baseline_qr     MAE=607.1529 | pred_q={0: 0.0, 0.01: 0.0, 0.05: 1006.6068673729086, 0.5: 3786.3903243205127, 0.95: 6513.728054153332, 1.0: 7817.803157025699}
  cb_resid_mae   MAE=591.8298 | pred_q={0: 48.685044311475735, 0.01: 73.41925374777297, 0.05: 1055.5838565851657, 0.5: 3767.8876686414733, 0.95: 6546.34535399445, 1.0: 7894.509649537379}
  cb_resid_q50   MAE=592.0785 | pred_q={0: 46.63681295972617, 0.01: 71.46052657559764, 0.05: 1054.6038959610723, 0.5: 3767.696748939021, 0.95: 6545.8125566662575, 1.0: 7892.973207692549}
  cb_full_logrmse MAE=456.6403 | pred_q={0: 974.0639203290026, 0.01: 1279.4221547225031, 0.05: 1683.8573441658657, 0.5: 3499.4741390701756, 0.95: 7354.569154508947, 1.0: 9657.310895407469}

[Diagnostics] Top 12 abs errors (model=cb_full_logrmse)
 patient_id  ed_cost_next3y_usd primary_chronic  prior_ed_cost_5y_usd  prior_ed_visits_5y  age  adm_n

Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU



[Top feature importances] (cb_resid_mae on residual features)
              feature  importance
      pdf_line_median   31.012298
logpriorcost_x_severe   26.935266
        pdf_line_mean   18.215351
         pdf_line_max    8.711514
   prior_ed_visits_5y    5.701192
         prior_visits    2.056027
      pdf_share_99285    1.949039
         pdf_line_std    1.255990
     log_prior_visits    0.953061
      pdf_share_99282    0.891226
        adm_dx_cnt_HF    0.575623
      pdf_share_99283    0.493618
      pdf_share_74177    0.454469
      primary_chronic    0.345284
    severe_proc_score    0.193067

[FINAL sanity checks]
OOF MAE (ensemble): 468.5363
ensemble: dirichlet_constrained | bag+full blend alpha: 0.6
submission shape: (2000, 2)
submission columns: ['patient_id', 'ed_cost_next3y_usd']
any NaN preds: False
pred min/median/max: 631.7284047844396 3628.1150499682617 9455.738541679588
pred quantiles: {0: 631.7284047844396, 0.01: 847.8167818921171, 0.05: 1431.0681101367488, 0.1: 1857

# Iteration 4
 - 465.3977

In [9]:
import os, sys, re, math, warnings, gc
from pathlib import Path
from typing import Tuple, Dict, Optional, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================
SEED = 42
np.random.seed(SEED)

DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"

TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"

PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"

ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"

RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR     = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"

OUT_SUB_PATH    = r"D:\AgentDs\agent_ds_healthcare\submission.csv"

print("=== ITERATION 4 (rollback to best idea + safer generalization) ===")
print("Why we got worse last time: residual model + shift weighting hurt generalization.")
print("This run focuses on: (1) strong CatBoost tabular models, (2) repeated CV bagging,")
print("(3) conservative ensemble (few params), (4) optional chronic-only bias correction.")
print("Also fixes GPU error by NOT using rsm on GPU.\n")

# =========================================================
# Utilities
# =========================================================
def safe_read_csv(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(p)

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def quantiles(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def ensure_cat_as_str(df: pd.DataFrame, cols: List[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")

def median_fill_numeric(train_df: pd.DataFrame, test_df: pd.DataFrame, numeric_cols: List[str]) -> None:
    for c in numeric_cols:
        med = pd.to_numeric(train_df[c], errors="coerce").median()
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(med)
        test_df[c]  = pd.to_numeric(test_df[c],  errors="coerce").fillna(med)

def fill_prefix_numeric_zero(df: pd.DataFrame, prefixes: List[str], exclude: Optional[List[str]] = None) -> None:
    exclude = set(exclude or [])
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes) and (c not in exclude)]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

def print_bin_mae(y_true, y_pred, n_bins=6, name="model"):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    edges = np.quantile(y_true, np.linspace(0, 1, n_bins + 1))
    bins = pd.cut(y_true, bins=edges, include_lowest=True, duplicates="drop")
    df = pd.DataFrame({"bin": bins.astype(str), "ae": np.abs(y_true - y_pred), "resid": (y_true - y_pred)})
    g = df.groupby("bin").agg(mae=("ae","mean"), median_resid=("resid","median"), n=("ae","size"))
    print(f"[{name}] MAE by target-quantile bin:")
    print(g.to_string())

def detect_gpu_catboost() -> bool:
    try:
        from catboost.utils import get_gpu_device_count
        return get_gpu_device_count() > 0
    except Exception:
        return False

# =========================================================
# Receipts: joblib extraction + feature builder
# =========================================================
def extract_receipt_dfs(obj) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    headers_df, lineitems_df = None, None
    if isinstance(obj, dict):
        for hk in ["headers_df","headers","header_df","header"]:
            if hk in obj and isinstance(obj[hk], pd.DataFrame):
                headers_df = obj[hk]
                break
        for lk in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if lk in obj and isinstance(obj[lk], pd.DataFrame):
                lineitems_df = obj[lk]
                break
        if lineitems_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and {"patient_id","code"}.issubset(set(v.columns)):
                    lineitems_df = v
                    break
        if headers_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and ("patient_id" in v.columns) and (v is not lineitems_df):
                    headers_df = v
                    break
    elif isinstance(obj, (tuple, list)):
        dfs = [x for x in obj if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if {"patient_id","code"}.issubset(set(df.columns)):
                lineitems_df = df
            elif "patient_id" in df.columns:
                headers_df = df
    elif isinstance(obj, pd.DataFrame):
        if {"patient_id","code"}.issubset(set(obj.columns)):
            lineitems_df = obj
        elif "patient_id" in obj.columns:
            headers_df = obj
    return headers_df, lineitems_df

def build_receipt_features(headers_df: Optional[pd.DataFrame],
                          lineitems_df: Optional[pd.DataFrame],
                          max_codes_dense: int = 180,
                          svd_dim: int = 8) -> pd.DataFrame:
    """
    Conservative-but-strong receipt features:
      - totals, counts, deflated totals, price index
      - per-code cnt/cost/share for selected codes
      - high-acuity flags + severity score
      - distribution (entropy, gini, top shares)
      - prefix cost/cnt (top 25 prefix2)
      - TF-IDF+SVD code embedding (small dim)
      - top-cost code categorical (top1/top2)
    """
    if (lineitems_df is None) or (not isinstance(lineitems_df, pd.DataFrame)) or (len(lineitems_df) == 0):
        if headers_df is not None and isinstance(headers_df, pd.DataFrame) and ("patient_id" in headers_df.columns):
            return headers_df.copy()
        return pd.DataFrame(columns=["patient_id"])

    li = lineitems_df.copy()

    # normalize column names
    if "code" not in li.columns:
        for alt in ["cpt","cpt_code","hcpcs","proc_code"]:
            if alt in li.columns:
                li["code"] = li[alt]
                break
    if "line_total" not in li.columns:
        for alt in ["line_total_usd","total","amount","line_cost","line_total_$","sum_items"]:
            if alt in li.columns:
                li["line_total"] = li[alt]
                break
    if "qty" not in li.columns:
        for alt in ["quantity","q"]:
            if alt in li.columns:
                li["qty"] = li[alt]
                break
    if "unit" not in li.columns:
        if "line_total" in li.columns and "qty" in li.columns:
            li["unit"] = pd.to_numeric(li["line_total"], errors="coerce") / pd.to_numeric(li["qty"], errors="coerce").replace(0, np.nan)
        else:
            li["unit"] = np.nan

    if "patient_id" not in li.columns or "code" not in li.columns:
        return pd.DataFrame(columns=["patient_id"])

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li["code"].astype(str).str.strip().str.upper()
    li["qty"] = pd.to_numeric(li["qty"], errors="coerce").fillna(1).astype(float)
    li["unit"] = pd.to_numeric(li["unit"], errors="coerce")
    li["line_total"] = pd.to_numeric(li["line_total"], errors="coerce").fillna(0.0).astype(float)

    must_codes = {
        "31500","36556","36620","92950",
        "70450","74177","84484","G0378",
        "99281","99282","99283","99284","99285",
        "99291","99292",
        "85025","87070","71045"
    }

    uniq_codes = li["code"].nunique()
    if uniq_codes <= max_codes_dense:
        codes = sorted(set(li["code"].unique().tolist()) | must_codes)
    else:
        code_cost = li.groupby("code")["line_total"].sum().sort_values(ascending=False)
        top_cost = code_cost.head(max_codes_dense).index.tolist()
        top_cnt = li["code"].value_counts().head(max_codes_dense).index.tolist()
        codes = sorted(set(top_cost + top_cnt + list(must_codes)))

    # deflated totals
    med_unit_by_code = li.groupby("code")["unit"].median()
    li["unit_med_code"] = li["code"].map(med_unit_by_code).astype(float)
    li["deflated_line_total"] = li["qty"] * li["unit_med_code"]

    g = li.groupby("patient_id")
    agg = g.agg(
        pdf_total_line_cost=("line_total","sum"),
        pdf_deflated_total=("deflated_line_total","sum"),
        pdf_n_line_items=("code","size"),
        pdf_total_qty=("qty","sum"),
        pdf_n_unique_codes=("code","nunique"),
        pdf_unit_mean=("unit","mean"),
        pdf_unit_median=("unit","median"),
        pdf_line_max=("line_total","max"),
        pdf_line_mean=("line_total","mean"),
        pdf_line_std=("line_total", lambda x: float(np.std(x, ddof=0))),
        pdf_line_median=("line_total","median"),
    ).reset_index()

    agg["pdf_price_index"] = agg["pdf_total_line_cost"] / agg["pdf_deflated_total"].replace(0, np.nan)
    agg["pdf_price_index"] = agg["pdf_price_index"].replace([np.inf, -np.inf], np.nan).fillna(1.0)

    li_sel = li[li["code"].isin(codes)].copy()
    piv_cnt  = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="size", fill_value=0)
    piv_cost = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="sum", fill_value=0.0)
    piv_cnt.columns  = [f"pdf_cnt_{c}"  for c in piv_cnt.columns.astype(str)]
    piv_cost.columns = [f"pdf_cost_{c}" for c in piv_cost.columns.astype(str)]
    wide = piv_cnt.join(piv_cost, how="outer").reset_index()

    out = agg.merge(wide, on="patient_id", how="left")
    fill_prefix_numeric_zero(out, prefixes=["pdf_"])

    denom = out["pdf_total_line_cost"].replace(0, np.nan)

    # shares per code
    for c in codes:
        cc = f"pdf_cost_{c}"
        out[f"pdf_share_{c}"] = (out[cc] / denom).fillna(0.0) if cc in out.columns else 0.0

    # high-acuity flags
    code_flags = {
        "has_intub_31500": "31500",
        "has_cvc_36556": "36556",
        "has_artline_36620": "36620",
        "has_cpr_92950": "92950",
        "has_ct_head_70450": "70450",
        "has_ct_abdpel_74177": "74177",
        "has_troponin_84484": "84484",
        "has_obs_G0378": "G0378",
        "has_99285": "99285",
    }
    for fname, code in code_flags.items():
        cnt_col = f"pdf_cnt_{code}"
        out[fname] = (out[cnt_col] > 0).astype(int) if cnt_col in out.columns else 0

    crit_cols = [c for c in ["99291","99292"] if f"pdf_cnt_{c}" in out.columns]
    if crit_cols:
        out["has_critical_care"] = (out[[f"pdf_cnt_{c}" for c in crit_cols]].sum(axis=1) > 0).astype(int)
    else:
        out["has_critical_care"] = 0

    out["severe_proc_score"] = (
        out.get("has_intub_31500", 0)
        + out.get("has_cvc_36556", 0)
        + out.get("has_artline_36620", 0)
        + out.get("has_cpr_92950", 0)
        + out.get("has_critical_care", 0)
    ).astype(int)

    # E/M level features
    em_level_map = {"99281":1, "99282":2, "99283":3, "99284":4, "99285":5}
    em = li[li["code"].isin(list(em_level_map.keys()))].copy()
    if len(em):
        em["em_level"] = em["code"].map(em_level_map).astype(float)
        em_agg = em.groupby("patient_id").agg(
            pdf_em_level_mean=("em_level","mean"),
            pdf_em_level_max=("em_level","max"),
            pdf_em_lines=("em_level","size"),
            pdf_em_high_frac=("em_level", lambda x: float(np.mean(np.asarray(x) >= 4))),
        ).reset_index()
        out = out.merge(em_agg, on="patient_id", how="left")
    for c in ["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]:
        if c not in out.columns:
            out[c] = 0.0
    out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]] = out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_lines","pdf_em_high_frac"]].fillna(0.0)

    # distribution metrics (entropy, gini, top shares)
    cost_cols = [c for c in out.columns if c.startswith("pdf_cost_")]
    if cost_cols:
        mat = out[cost_cols].to_numpy(dtype=float)
        row_sum = mat.sum(axis=1, keepdims=True)
        p = np.divide(mat, row_sum, out=np.zeros_like(mat), where=row_sum > 0)
        ent = -(p * np.log(np.clip(p, 1e-12, 1.0))).sum(axis=1)
        ent_norm = ent / np.log(max(2, mat.shape[1]))
        out["pdf_code_cost_entropy_norm"] = ent_norm

        sorted_mat = np.sort(mat, axis=1)
        s = sorted_mat.sum(axis=1)
        top1 = sorted_mat[:, -1]
        top2 = top1 + (sorted_mat[:, -2] if sorted_mat.shape[1] >= 2 else 0.0)
        top3 = top2 + (sorted_mat[:, -3] if sorted_mat.shape[1] >= 3 else 0.0)

        out["pdf_cost_top1_share"] = np.divide(top1, s, out=np.zeros_like(top1), where=s > 0)
        out["pdf_cost_top2_share"] = np.divide(top2, s, out=np.zeros_like(top2), where=s > 0)
        out["pdf_cost_top3_share"] = np.divide(top3, s, out=np.zeros_like(top3), where=s > 0)

        n = sorted_mat.shape[1]
        i = np.arange(1, n + 1, dtype=float)
        weighted = (sorted_mat * i.reshape(1, -1)).sum(axis=1)
        gini = np.zeros_like(s, dtype=float)
        mask = s > 0
        gini[mask] = (2.0 * weighted[mask]) / (n * s[mask]) - (n + 1) / n
        out["pdf_cost_gini"] = gini
    else:
        out["pdf_code_cost_entropy_norm"] = 0.0
        out["pdf_cost_top1_share"] = 0.0
        out["pdf_cost_top2_share"] = 0.0
        out["pdf_cost_top3_share"] = 0.0
        out["pdf_cost_gini"] = 0.0

    # Prefix aggregates
    li_sel["prefix1"] = li_sel["code"].str[:1]
    li_sel["prefix2"] = li_sel["code"].str[:2]

    pref2_cost = li_sel.pivot_table(index="patient_id", columns="prefix2", values="line_total", aggfunc="sum", fill_value=0.0)
    pref2_cnt  = li_sel.pivot_table(index="patient_id", columns="prefix2", values="line_total", aggfunc="size", fill_value=0)
    pref2_tot = pref2_cost.sum(axis=0).sort_values(ascending=False)
    keep_p2 = pref2_tot.head(25).index.tolist()
    pref2_cost = pref2_cost[keep_p2]
    pref2_cnt  = pref2_cnt[keep_p2]
    pref2_cost.columns = [f"pdf_pref2_cost_{c}" for c in pref2_cost.columns.astype(str)]
    pref2_cnt.columns  = [f"pdf_pref2_cnt_{c}"  for c in pref2_cnt.columns.astype(str)]
    pref2 = pref2_cnt.join(pref2_cost, how="outer").reset_index()
    out = out.merge(pref2, on="patient_id", how="left")
    fill_prefix_numeric_zero(out, prefixes=["pdf_pref2_"])

    # Top code categorical by cost (top1/top2)
    code_cost_patient = li.groupby(["patient_id","code"])["line_total"].sum().reset_index()
    code_cost_patient = code_cost_patient.sort_values(["patient_id","line_total"], ascending=[True, False])
    top1 = code_cost_patient.groupby("patient_id").head(1).rename(columns={"code":"pdf_topcode_cost1","line_total":"pdf_topcode_cost1_amt"})
    top2 = code_cost_patient.groupby("patient_id").head(2).groupby("patient_id").tail(1).rename(columns={"code":"pdf_topcode_cost2","line_total":"pdf_topcode_cost2_amt"})
    topc = top1[["patient_id","pdf_topcode_cost1","pdf_topcode_cost1_amt"]].merge(
        top2[["patient_id","pdf_topcode_cost2","pdf_topcode_cost2_amt"]], on="patient_id", how="left"
    )
    out = out.merge(topc, on="patient_id", how="left")
    for c in ["pdf_topcode_cost1","pdf_topcode_cost2"]:
        if c in out.columns:
            out[c] = out[c].astype("string").fillna("Unknown")
    out["pdf_topcode_cost1_amt"] = pd.to_numeric(out.get("pdf_topcode_cost1_amt", 0.0), errors="coerce").fillna(0.0)
    out["pdf_topcode_cost2_amt"] = pd.to_numeric(out.get("pdf_topcode_cost2_amt", 0.0), errors="coerce").fillna(0.0)
    out["pdf_topcode_cost1_share"] = (out["pdf_topcode_cost1_amt"] / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pdf_topcode_cost2_share"] = (out["pdf_topcode_cost2_amt"] / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    # Optional TF-IDF + SVD embedding (small)
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD

        code_str = (
            li.groupby(["patient_id","code"])
              .size()
              .reset_index(name="cnt")
              .sort_values(["patient_id","cnt"], ascending=[True, False])
        )
        code_str["rep"] = code_str["cnt"].clip(upper=3)
        code_str["tok_rep"] = (code_str["code"].astype(str) + " ") * code_str["rep"].astype(int)
        seq = code_str.groupby("patient_id")["tok_rep"].sum().reset_index()
        seq.columns = ["patient_id", "pdf_code_seq"]

        vec = TfidfVectorizer(token_pattern=r"\S+", min_df=1, max_features=300)
        X_tfidf = vec.fit_transform(seq["pdf_code_seq"].fillna(""))

        svd_dim_eff = min(svd_dim, max(2, X_tfidf.shape[1] - 1))
        svd = TruncatedSVD(n_components=svd_dim_eff, random_state=SEED)
        X_svd = svd.fit_transform(X_tfidf)

        svd_cols = [f"pdf_code_svd_{i}" for i in range(svd_dim_eff)]
        seq_svd = pd.DataFrame(X_svd, columns=svd_cols)
        seq_svd.insert(0, "patient_id", seq["patient_id"].values)

        out = out.merge(seq_svd, on="patient_id", how="left")
        fill_prefix_numeric_zero(out, prefixes=["pdf_code_svd_"])
        print(f"[receipts] TF-IDF+SVD embeddings: vocab={X_tfidf.shape[1]} dim={svd_dim_eff}")
    except Exception as e:
        print(f"[receipts] TF-IDF+SVD skipped: {e}")

    # Merge header if present
    if headers_df is not None and isinstance(headers_df, pd.DataFrame) and ("patient_id" in headers_df.columns):
        cols = ["patient_id"] + [c for c in ["pdf_total","sum_items","zip3_pdf","insurance_pdf"] if c in headers_df.columns]
        hd = headers_df[cols].copy()
        out = out.merge(hd, on="patient_id", how="left")
        if "pdf_total" in out.columns:
            out["pdf_total_missing"] = out["pdf_total"].isna().astype(int)
            out["pdf_total"] = pd.to_numeric(out["pdf_total"], errors="coerce").fillna(0.0)
        if "sum_items" in out.columns:
            out["sum_items"] = pd.to_numeric(out["sum_items"], errors="coerce").fillna(0.0)
        if "zip3_pdf" in out.columns:
            out["zip3_pdf"] = out["zip3_pdf"].apply(standardize_zip3)
        if "insurance_pdf" in out.columns:
            out["insurance_pdf"] = out["insurance_pdf"].astype("string").fillna("Unknown")

    return out

# =========================================================
# Admissions features
# =========================================================
def build_admissions_features(adm_tr: pd.DataFrame, adm_te: pd.DataFrame) -> pd.DataFrame:
    adm_tr = adm_tr.drop(columns=["readmit_30d"], errors="ignore")
    adm_all = pd.concat([adm_tr, adm_te], ignore_index=True)

    if "patient_id" not in adm_all.columns:
        return pd.DataFrame(columns=["patient_id"])

    adm_all["patient_id"] = pd.to_numeric(adm_all["patient_id"], errors="coerce").astype("Int64")
    adm_all = adm_all.dropna(subset=["patient_id"]).copy()
    adm_all["patient_id"] = adm_all["patient_id"].astype(int)

    num_cols = ["los_days","acuity_emergent","charlson_band","ed_visits_6m","discharge_weekday"]
    for c in num_cols:
        if c in adm_all.columns:
            adm_all[c] = pd.to_numeric(adm_all[c], errors="coerce")

    g = adm_all.groupby("patient_id")
    out = pd.DataFrame({"patient_id": g.size().index, "adm_n": g.size().values})

    for c in num_cols:
        if c in adm_all.columns:
            out[f"adm_{c}_mean"] = g[c].mean().values
            out[f"adm_{c}_max"]  = g[c].max().values
            out[f"adm_{c}_sum"]  = g[c].sum().values
            out[f"adm_{c}_std"]  = g[c].std(ddof=0).values

    if "primary_dx" in adm_all.columns:
        dx_ct = pd.crosstab(adm_all["patient_id"], adm_all["primary_dx"])
        dx_ct.columns = [f"adm_dx_cnt_{str(c)}" for c in dx_ct.columns]
        dx_ct = dx_ct.reset_index()
        out = out.merge(dx_ct, on="patient_id", how="left")

        dx_mode = g["primary_dx"].agg(lambda x: x.value_counts().index[0] if len(x) else "Unknown").reset_index()
        dx_mode.columns = ["patient_id", "adm_primary_dx_mode"]
        out = out.merge(dx_mode, on="patient_id", how="left")

        dx_nuniq = g["primary_dx"].nunique().reset_index()
        dx_nuniq.columns = ["patient_id", "adm_dx_nunique"]
        out = out.merge(dx_nuniq, on="patient_id", how="left")

    for c in out.columns:
        if c in ["patient_id", "adm_primary_dx_mode"]:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    if "adm_primary_dx_mode" in out.columns:
        out["adm_primary_dx_mode"] = out["adm_primary_dx_mode"].astype("string").fillna("Unknown")
    return out

# =========================================================
# Load data
# =========================================================
train = safe_read_csv(TRAIN_PATH)
test  = safe_read_csv(TEST_PATH)
patients = safe_read_csv(PATIENTS_PATH)
adm_tr = safe_read_csv(ADM_TRAIN_PATH)
adm_te = safe_read_csv(ADM_TEST_PATH)

for df in [train, test, patients, adm_tr, adm_te]:
    if "patient_id" in df.columns:
        df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

if "zip3" in patients.columns:
    patients["zip3"] = patients["zip3"].apply(standardize_zip3)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)
log_shape("admissions_train", adm_tr)
log_shape("admissions_test", adm_te)

print("\n[target stats]\n", train["ed_cost_next3y_usd"].describe().to_string())
print("\n[prior cost stats]\n", train["prior_ed_cost_5y_usd"].describe().to_string())

# =========================================================
# Build features: admissions + receipts
# =========================================================
adm_feat = build_admissions_features(adm_tr, adm_te)
print(f"\n[admissions] patient features: {adm_feat.shape}")

receipt_feat = pd.DataFrame(columns=["patient_id"])
joblib_path = Path(RECEIPTS_JOBLIB_PATH)

all_patient_ids = pd.concat([train["patient_id"], test["patient_id"]], ignore_index=True).astype(int).unique().tolist()

joblib_load = None
try:
    from joblib import load as joblib_load
except Exception:
    joblib_load = None

headers_df = None
lineitems_df = None

if joblib_path.exists() and joblib_load is not None:
    try:
        obj = joblib_load(joblib_path)
        headers_df, lineitems_df = extract_receipt_dfs(obj)

        # If already patient-level features, use directly
        if isinstance(obj, pd.DataFrame) and ("patient_id" in obj.columns) and ("code" not in obj.columns) and (obj["patient_id"].nunique() == len(obj)):
            receipt_feat = obj.copy()
            print(f"[receipts] joblib is patient-level features: {receipt_feat.shape}")
        else:
            receipt_feat = build_receipt_features(headers_df, lineitems_df, max_codes_dense=180, svd_dim=8)
            print(f"[receipts] built from joblib lineitems: {receipt_feat.shape}")
    except Exception as e:
        print(f"[receipts] ERROR loading/building from joblib: {e}")
        receipt_feat = pd.DataFrame({"patient_id": all_patient_ids})
else:
    print("[receipts] receipts_parsed.joblib missing; proceeding without extra receipt features.")
    receipt_feat = pd.DataFrame({"patient_id": all_patient_ids})

# ensure unique patient_id
if "patient_id" in receipt_feat.columns:
    receipt_feat["patient_id"] = pd.to_numeric(receipt_feat["patient_id"], errors="coerce").astype("Int64")
    receipt_feat = receipt_feat.dropna(subset=["patient_id"]).copy()
    receipt_feat["patient_id"] = receipt_feat["patient_id"].astype(int)
    receipt_feat = receipt_feat.drop_duplicates(subset=["patient_id"], keep="last")

cov = receipt_feat["patient_id"].nunique() / len(all_patient_ids) if "patient_id" in receipt_feat.columns else 0.0
print(f"[receipts] coverage: {receipt_feat['patient_id'].nunique()}/{len(all_patient_ids)} = {cov:.3f}")

# =========================================================
# Merge into modeling tables
# =========================================================
def build_model_df(ed_df: pd.DataFrame, patients: pd.DataFrame,
                   adm_feat: pd.DataFrame, receipt_feat: pd.DataFrame) -> pd.DataFrame:
    df = ed_df.merge(patients, on="patient_id", how="left")
    if adm_feat is not None and len(adm_feat):
        df = df.merge(adm_feat, on="patient_id", how="left")
    if receipt_feat is not None and len(receipt_feat):
        df = df.merge(receipt_feat, on="patient_id", how="left", suffixes=("", "_rcpt"))

    fill_prefix_numeric_zero(df, prefixes=["adm_"], exclude=["adm_primary_dx_mode"])
    fill_prefix_numeric_zero(df, prefixes=["pdf_"])
    fill_prefix_numeric_zero(df, prefixes=["has_"])
    fill_prefix_numeric_zero(df, prefixes=["severe_"])

    # receipt header cats if present
    for c in ["zip3_pdf","insurance_pdf","pdf_topcode_cost1","pdf_topcode_cost2"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")
            if c == "zip3_pdf":
                df[c] = df[c].apply(standardize_zip3)
    if "adm_primary_dx_mode" in df.columns:
        df["adm_primary_dx_mode"] = df["adm_primary_dx_mode"].astype("string").fillna("Unknown")

    return df

train_df = build_model_df(train, patients, adm_feat, receipt_feat)
test_df  = build_model_df(test,  patients, adm_feat, receipt_feat)

print("\n[merged]")
log_shape("train_df", train_df)
log_shape("test_df", test_df)

# sanity: receipt sum should equal prior cost (if feature exists)
if "pdf_total_line_cost" in train_df.columns and "prior_ed_cost_5y_usd" in train_df.columns:
    diff = np.abs(pd.to_numeric(train_df["pdf_total_line_cost"], errors="coerce").fillna(0.0) - pd.to_numeric(train_df["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0))
    print(f"[sanity] |pdf_total_line_cost - prior_ed_cost_5y_usd|: mean={diff.mean():.6f} max={diff.max():.6f}")
else:
    print("[sanity] pdf_total_line_cost not available to check invariant (ok).")

# =========================================================
# Derived features
# =========================================================
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["prior_cost"] = pd.to_numeric(df.get("prior_ed_cost_5y_usd", np.nan), errors="coerce").fillna(0.0)
    df["prior_visits"] = pd.to_numeric(df.get("prior_ed_visits_5y", np.nan), errors="coerce").fillna(0.0)

    df["log_prior_cost"] = np.log1p(df["prior_cost"])
    df["log_prior_visits"] = np.log1p(df["prior_visits"])
    df["cost_per_visit"] = df["prior_cost"] / np.maximum(1.0, df["prior_visits"])
    df["log_cost_per_visit"] = np.log1p(df["cost_per_visit"])
    df["cost_per_year"] = df["prior_cost"] / 5.0
    df["baseline_next3y"] = df["prior_cost"] * (3.0/5.0)

    # mild cap
    df["prior_cost_cap20k"] = np.minimum(df["prior_cost"], 20000.0)
    df["log_prior_cost_cap20k"] = np.log1p(df["prior_cost_cap20k"])

    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        df["age"] = df["age"].fillna(df["age"].median())
        df["age_x_logpriorcost"] = df["age"] * df["log_prior_cost"]

    if "severe_proc_score" in df.columns:
        df["priorcost_x_severeproc"] = df["log_prior_cost"] * pd.to_numeric(df["severe_proc_score"], errors="coerce").fillna(0.0)

    if "pdf_cost_top1_share" in df.columns:
        df["logpriorcost_x_top1share"] = df["log_prior_cost"] * pd.to_numeric(df["pdf_cost_top1_share"], errors="coerce").fillna(0.0)

    # simple cross-categorical (helps trees)
    if "primary_chronic" in df.columns and "severe_proc_score" in df.columns:
        df["chronic_x_severe"] = df["primary_chronic"].astype(str) + "_sev" + pd.to_numeric(df["severe_proc_score"], errors="coerce").fillna(0).astype(int).astype(str)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

train_df = add_derived_features(train_df)
test_df  = add_derived_features(test_df)

# =========================================================
# Prepare X/y and categorical columns
# =========================================================
TARGET_COL = "ed_cost_next3y_usd"
y = pd.to_numeric(train_df[TARGET_COL], errors="coerce").values.astype(float)

X = train_df.drop(columns=[TARGET_COL])
X_test = test_df.copy()

test_patient_id = test_df["patient_id"].astype(int).values

# remove id
X = X.drop(columns=["patient_id"], errors="ignore")
X_test = X_test.drop(columns=["patient_id"], errors="ignore")

cat_candidates = [
    "primary_chronic","sex","insurance","zip3",
    "zip3_pdf","insurance_pdf",
    "adm_primary_dx_mode",
    "pdf_topcode_cost1","pdf_topcode_cost2",
    "chronic_x_severe",
]
cat_cols = [c for c in cat_candidates if c in X.columns and c in X_test.columns]

ensure_cat_as_str(X, cat_cols)
ensure_cat_as_str(X_test, cat_cols)

num_cols = [c for c in X.columns if c not in cat_cols]
median_fill_numeric(X, X_test, num_cols)

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
median_fill_numeric(X, X_test, num_cols)

print("\n[features]")
print(f"Total features: {X.shape[1]} | numeric={len(num_cols)} | categorical={len(cat_cols)}")
print("Categoricals:", cat_cols)

# =========================================================
# Models + Repeated CV Bagging
# =========================================================
try:
    from catboost import CatBoostRegressor, Pool
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
    from catboost import CatBoostRegressor, Pool

from sklearn.model_selection import StratifiedKFold

use_gpu = detect_gpu_catboost()
print(f"\n[catboost] GPU available: {use_gpu} (we avoid GPU-incompatible params)")

# Stratification: chronic + prior cost bins
try:
    strat = train_df["primary_chronic"].astype(str) + "_" + pd.qcut(train_df["prior_ed_cost_5y_usd"], q=8, duplicates="drop").astype(str)
except Exception:
    strat = train_df["primary_chronic"].astype(str)

# Conservative CatBoost params (no rsm)
common = dict(
    allow_writing_files=False,
    verbose=False,
    task_type="GPU" if use_gpu else "CPU",
    random_seed=SEED,
)
if not use_gpu:
    common["thread_count"] = -1

model_defs = [
    dict(
        name="cb_mae_d6",
        transform=None,
        params=dict(common,
                    loss_function="MAE",
                    eval_metric="MAE",
                    iterations=8000 if use_gpu else 2600,
                    learning_rate=0.045,
                    depth=6,
                    l2_leaf_reg=12,
                    bagging_temperature=1.1,
                    random_strength=1.0,
                    min_data_in_leaf=24),
    ),
    dict(
        name="cb_logrmse_d7",
        transform="log1p",
        params=dict(common,
                    loss_function="RMSE",
                    eval_metric="RMSE",
                    iterations=9000 if use_gpu else 2800,
                    learning_rate=0.040,
                    depth=7,
                    l2_leaf_reg=8,
                    bagging_temperature=0.8,
                    random_strength=1.0,
                    min_data_in_leaf=24),
    ),
    dict(
        name="cb_q80_d6",
        transform=None,
        params=dict(common,
                    loss_function="Quantile:alpha=0.8",
                    eval_metric="MAE",
                    iterations=9000 if use_gpu else 2800,
                    learning_rate=0.040,
                    depth=6,
                    l2_leaf_reg=14,
                    bagging_temperature=1.0,
                    random_strength=1.0,
                    min_data_in_leaf=24),
    ),
]

N_SPLITS = 5
N_REPEATS = 2  # bagging for generalization (cheap at 2k rows)

oof_sum = {m["name"]: np.zeros(len(X), dtype=float) for m in model_defs}
test_sum = {m["name"]: np.zeros(len(X_test), dtype=float) for m in model_defs}
best_iters = {m["name"]: [] for m in model_defs}

fold_log_rows = []

for rep in range(N_REPEATS):
    rs = SEED + rep * 137
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=rs)
    oof_rep = {m["name"]: np.zeros(len(X), dtype=float) for m in model_defs}

    print(f"\n[CV] Repeat {rep+1}/{N_REPEATS} (random_state={rs})")
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, strat), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        row = {"rep": rep+1, "fold": fold}
        for m in model_defs:
            name = m["name"]
            params = m["params"].copy()
            params["random_seed"] = rs + fold  # diversify

            # Build pools
            if m["transform"] == "log1p":
                y_tr_fit = np.log1p(y_tr)
                y_va_fit = np.log1p(y_va)
                train_pool = Pool(X_tr, y_tr_fit, cat_features=cat_cols) if cat_cols else Pool(X_tr, y_tr_fit)
                valid_pool = Pool(X_va, y_va_fit, cat_features=cat_cols) if cat_cols else Pool(X_va, y_va_fit)
            else:
                train_pool = Pool(X_tr, y_tr, cat_features=cat_cols) if cat_cols else Pool(X_tr, y_tr)
                valid_pool = Pool(X_va, y_va, cat_features=cat_cols) if cat_cols else Pool(X_va, y_va)

            reg = CatBoostRegressor(**params)

            # robust fit with early stopping; fallback to CPU if GPU fails (rare)
            try:
                reg.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=300)
            except Exception as e:
                if params.get("task_type") == "GPU":
                    params_cpu = params.copy()
                    params_cpu["task_type"] = "CPU"
                    params_cpu["thread_count"] = -1
                    reg = CatBoostRegressor(**params_cpu)
                    reg.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=300)
                    print(f"[warn] GPU failed for {name} rep{rep+1} fold{fold}: {e} -> retrained on CPU")
                else:
                    raise

            try:
                best_iters[name].append(int(reg.get_best_iteration()))
            except Exception:
                pass

            # predict val/test
            if m["transform"] == "log1p":
                pred_va = np.expm1(reg.predict(X_va))
                pred_te = np.expm1(reg.predict(X_test))
            else:
                pred_va = reg.predict(X_va)
                pred_te = reg.predict(X_test)

            pred_va = np.clip(pred_va, 0.0, None)
            pred_te = np.clip(pred_te, 0.0, None)

            oof_rep[name][va_idx] = pred_va
            test_sum[name] += pred_te

            row[name] = mae(y_va, pred_va)

        fold_log_rows.append(row)
        fold_str = " | ".join([f"{k}={row[k]:.2f}" for k in ["cb_mae_d6","cb_logrmse_d7","cb_q80_d6"]])
        print(f"  fold {fold}/{N_SPLITS}: {fold_str}")

    # add repeat OOF into global OOF sum
    for m in model_defs:
        oof_sum[m["name"]] += oof_rep[m["name"]]

# Average across repeats
oof_avg = {k: v / N_REPEATS for k, v in oof_sum.items()}
test_avg = {k: v / (N_REPEATS * N_SPLITS) for k, v in test_sum.items()}

print("\n[OOF model summary]")
for m in model_defs:
    name = m["name"]
    oof_mae = mae(y, oof_avg[name])
    bias = float(np.median(y - oof_avg[name]))
    print(f"  {name:12s} OOF_MAE={oof_mae:.4f} | median_resid(y-pred)={bias:+.2f} | best_iter_med={int(np.median(best_iters[name])) if best_iters[name] else None}")
    print(f"    pred_q={quantiles(oof_avg[name], qs=(0,0.01,0.05,0.5,0.95,0.99,1.0))}")

# Detailed fold log table (compact)
fold_log_df = pd.DataFrame(fold_log_rows)
print("\n[CV fold MAE stats] (mean±std across all folds/repeats)")
for m in model_defs:
    name = m["name"]
    print(f"  {name:12s} {fold_log_df[name].mean():.3f} ± {fold_log_df[name].std(ddof=0):.3f}")

# =========================================================
# Ensemble: base blend (MAE+logRMSE) + tail uplift using q80 gated by risk
# =========================================================
p_mae = oof_avg["cb_mae_d6"]
p_log = oof_avg["cb_logrmse_d7"]
p_q80 = oof_avg["cb_q80_d6"]

t_mae = test_avg["cb_mae_d6"]
t_log = test_avg["cb_logrmse_d7"]
t_q80 = test_avg["cb_q80_d6"]

# Gate: risk score based on severity/proxy (NO target used)
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def build_gate(df: pd.DataFrame) -> np.ndarray:
    sev = pd.to_numeric(df.get("severe_proc_score", 0), errors="coerce").fillna(0.0).values
    cc  = pd.to_numeric(df.get("has_critical_care", 0), errors="coerce").fillna(0.0).values
    em4 = (pd.to_numeric(df.get("pdf_em_level_max", 0), errors="coerce").fillna(0.0).values >= 4).astype(float)
    lp  = pd.to_numeric(df.get("log_prior_cost", 0), errors="coerce").fillna(0.0).values
    ch  = pd.to_numeric(df.get("adm_charlson_band_mean", 0), errors="coerce").fillna(0.0).values

    # normalize lp and ch roughly
    lp_z = (lp - np.median(lp)) / (np.std(lp) + 1e-9)
    ch_z = (ch - np.median(ch)) / (np.std(ch) + 1e-9)

    score = 0.9*sev + 1.2*cc + 0.6*em4 + 0.35*lp_z + 0.20*ch_z
    g = sigmoid(score - 0.3)  # bias to keep most gates modest
    return g.astype(float)

gate_tr = build_gate(train_df)
gate_te = build_gate(test_df)

best = {"mae": 1e18, "w": None, "alpha": None, "pred": None}
grid_w = np.arange(0.0, 1.0+1e-9, 0.05)
grid_a = np.arange(0.0, 1.0+1e-9, 0.05)

for w in grid_w:
    base = w*p_mae + (1.0-w)*p_log
    for a in grid_a:
        pred = base + a*gate_tr*(p_q80 - base)
        pred = np.clip(pred, 0.0, None)
        m = mae(y, pred)
        if m < best["mae"]:
            best = {"mae": m, "w": float(w), "alpha": float(a), "pred": pred}

# choose simpler if within epsilon: prefer w near 0.5 and alpha near 0.3
eps = 0.15
candidates = []
best_mae_val = best["mae"]

for w in grid_w:
    base = w*p_mae + (1.0-w)*p_log
    for a in grid_a:
        pred = np.clip(base + a*gate_tr*(p_q80 - base), 0.0, None)
        m = mae(y, pred)
        if m <= best_mae_val + eps:
            simplicity = abs(w-0.5) + 0.7*abs(a-0.3)
            candidates.append((m, simplicity, float(w), float(a), pred))
candidates.sort(key=lambda x: (x[0], x[1]))
chosen = candidates[0]
ens_oof = chosen[4]
ens_w, ens_a = chosen[2], chosen[3]
ens_mae = chosen[0]

print("\n[ENS search]")
print(f"  best_raw_MAE={best_mae_val:.4f} at w={best['w']:.2f}, alpha={best['alpha']:.2f}")
print(f"  chosen_MAE  ={ens_mae:.4f} at w={ens_w:.2f}, alpha={ens_a:.2f} (simplicity-tiebreak)")

print_bin_mae(y, ens_oof, n_bins=6, name=f"ENS(w={ens_w:.2f},a={ens_a:.2f})")

# =========================================================
# Optional bias correction (chronic-only) if it improves OOF
# =========================================================
ens_base = ens_oof.copy()
best_corr = {"name": "none", "mae": ens_mae, "pred": ens_base}

# global median shift
global_shift = float(np.median(y - ens_base))
for shrink in [0.25, 0.5, 1.0]:
    p = np.clip(ens_base + shrink*global_shift, 0.0, None)
    m = mae(y, p)
    if m + 0.05 < best_corr["mae"]:
        best_corr = {"name": f"global_shift(shrink={shrink})", "mae": m, "pred": p}

# chronic-only shift
if "primary_chronic" in train_df.columns:
    keys = train_df["primary_chronic"].astype(str).values
    tmp = pd.DataFrame({"k": keys, "r": (y - ens_base)})
    grp = tmp.groupby("k")["r"].median().to_dict()
    global_med = float(np.median(tmp["r"].values))

    for shrink in [0.25, 0.5, 0.75]:
        adj = np.array([grp.get(k, global_med) for k in keys], dtype=float)
        p = np.clip(ens_base + shrink*adj, 0.0, None)
        m = mae(y, p)
        if m + 0.05 < best_corr["mae"]:
            best_corr = {"name": f"chronic_shift(shrink={shrink})", "mae": m, "pred": p}

ens_oof_final = best_corr["pred"]
print(f"\n[Bias correction] chosen: {best_corr['name']} | OOF_MAE={best_corr['mae']:.4f}")

# Apply ensemble to test
base_te = ens_w*t_mae + (1.0-ens_w)*t_log
ens_te = np.clip(base_te + ens_a*gate_te*(t_q80 - base_te), 0.0, None)

# apply same correction to test if selected
if best_corr["name"].startswith("global_shift"):
    shrink = float(best_corr["name"].split("shrink=")[1].split(")")[0])
    ens_te = np.clip(ens_te + shrink*global_shift, 0.0, None)
elif best_corr["name"].startswith("chronic_shift") and "primary_chronic" in test_df.columns:
    shrink = float(best_corr["name"].split("shrink=")[1].split(")")[0])
    keys_te = test_df["primary_chronic"].astype(str).values
    adj_te = np.array([grp.get(k, global_med) for k in keys_te], dtype=float)
    ens_te = np.clip(ens_te + shrink*adj_te, 0.0, None)

# =========================================================
# Train final models on full train (requirement) + feature importance
#   (We keep final prediction from bagged ensemble for generalization.)
# =========================================================
print("\n[Full-train fit] Training one final model for feature importance (cb_mae_d6)...")
main_params = [m for m in model_defs if m["name"] == "cb_mae_d6"][0]["params"].copy()
main_params["random_seed"] = SEED
# optionally set iterations ~ median best_iter + buffer
if best_iters["cb_mae_d6"]:
    it = int(np.median(best_iters["cb_mae_d6"]))
    main_params["iterations"] = int(max(400, it + (150 if use_gpu else 100)))

train_pool_full = Pool(X, y, cat_features=cat_cols) if cat_cols else Pool(X, y)
main_model = CatBoostRegressor(**main_params)
main_model.fit(train_pool_full, verbose=False)

try:
    imp = main_model.get_feature_importance()
    imp_df = pd.DataFrame({"feature": X.columns.tolist(), "importance": imp}).sort_values("importance", ascending=False).head(12)
    print("\n[Top 12 feature importances] (cb_mae_d6 full-train)")
    print(imp_df.to_string(index=False))
except Exception as e:
    print(f"[warn] could not compute importances: {e}")

# =========================================================
# Write submission
# =========================================================
sub = pd.DataFrame({
    "patient_id": test_patient_id,
    "ed_cost_next3y_usd": ens_te.astype(float),
})[["patient_id","ed_cost_next3y_usd"]]

print("\n[FINAL sanity checks]")
print("OOF MAE (ensemble):", f"{best_corr['mae']:.4f}")
print("Ensemble details: base=w*MAE+(1-w)*logRMSE, w=", f"{ens_w:.2f}", " alpha=", f"{ens_a:.2f}", " bias_correction=", best_corr["name"])
print("submission shape:", sub.shape)
print("submission columns:", list(sub.columns))
print("any NaNs in preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(sub["ed_cost_next3y_usd"].min()), float(sub["ed_cost_next3y_usd"].median()), float(sub["ed_cost_next3y_usd"].max()))
print("pred quantiles:", quantiles(sub["ed_cost_next3y_usd"].values, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)))

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print(f"\nSaved submission to: {out_path}")
print("\nPaste back: leaderboard MAE + these logs (OOF MAE, w/alpha, bias_correction, pred quantiles).")


=== ITERATION 4 (rollback to best idea + safer generalization) ===
Why we got worse last time: residual model + shift weighting hurt generalization.
This run focuses on: (1) strong CatBoost tabular models, (2) repeated CV bagging,
(3) conservative ensemble (few params), (4) optional chronic-only bias correction.
Also fixes GPU error by NOT using rsm on GPU.

[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.66 MB
[admissions_train] shape=(5000, 9) | cols=9 | mem=0.57 MB
[admissions_test] shape=(5000, 8) | cols=8 | mem=0.53 MB

[target stats]
 count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[prior cost stats]
 count     2000.000000
mean      4003.177295
std       4857.839498
min         50.000000
25%        940.982500
50%       2168.710000
75%       5073.502500
max  

Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


  fold 1/5: cb_mae_d6=1345.86 | cb_logrmse_d7=460.29 | cb_q80_d6=1947.30


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


  fold 2/5: cb_mae_d6=1314.02 | cb_logrmse_d7=458.80 | cb_q80_d6=1934.68


...

  fold 4/5: cb_mae_d6=1295.99 | cb_logrmse_d7=466.85 | cb_q80_d6=1928.89


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


  fold 5/5: cb_mae_d6=1314.99 | cb_logrmse_d7=465.96 | cb_q80_d6=1939.32

[OOF model summary]
  cb_mae_d6    OOF_MAE=1324.1000 | median_resid(y-pred)=+12.53 | best_iter_med=7999
    pred_q={0: 3372.926566680777, 0.01: 3380.103756876781, 0.05: 3393.4367523422115, 0.5: 3563.792725092713, 0.95: 3740.868271842989, 0.99: 3751.7232270910395, 1.0: 3756.814193610102}
  cb_logrmse_d7 OOF_MAE=449.3361 | median_resid(y-pred)=+54.59 | best_iter_med=335
    pred_q={0: 1065.4898985714092, 0.01: 1266.3798390637642, 0.05: 1670.9400953029728, 0.5: 3513.6296308175774, 0.95: 7377.614076860142, 0.99: 8562.023257238992, 1.0: 9869.263337481047}
  cb_q80_d6    OOF_MAE=1935.2998 | median_resid(y-pred)=-1705.54 | best_iter_med=8999
    pred_q={0: 5225.954616248608, 0.01: 5227.719818824441, 0.05: 5233.888564188494, 0.5: 5276.446599229632, 0.95: 5570.652981572357, 0.99: 5598.616874418645, 1.0: 5620.072494915687}

[CV fold MAE stats] (mean±std across all folds/repeats)
  cb_mae_d6    1324.158 ± 39.197
  cb_logrms

Default metric period is 5 because MAE is/are not implemented for GPU



[Top 12 feature importances] (cb_mae_d6 full-train)
               feature  importance
       primary_chronic   42.964067
  prior_ed_cost_5y_usd   33.576831
priorcost_x_severeproc    6.457605
    pdf_deflated_total    5.026998
     prior_cost_cap20k    4.711885
   pdf_total_line_cost    2.347218
       baseline_next3y    1.348273
            prior_cost    1.114632
 log_prior_cost_cap20k    0.994780
         cost_per_year    0.430580
         adm_dx_cnt_HF    0.271402
 pdf_topcode_cost2_amt    0.101350

[FINAL sanity checks]
OOF MAE (ensemble): 447.2258
Ensemble details: base=w*MAE+(1-w)*logRMSE, w= 0.00  alpha= 0.00  bias_correction= global_shift(shrink=1.0)
submission shape: (2000, 2)
submission columns: ['patient_id', 'ed_cost_next3y_usd']
any NaNs in preds: False
pred min/median/max: 1080.65742165571 3590.1806603262676 9866.849642586076
pred quantiles: {0: 1080.65742165571, 0.01: 1311.1618974310338, 0.05: 1691.8692593973553, 0.1: 2006.919211590394, 0.25: 2602.8594152855067, 0.5: 35

# Iteration 5
 - 463.8476

In [11]:
# === ITERATION 5: Back-to-success pipeline (rich receipts + strong CatBoost bagging + gated q90 uplift) ===
# Goal: recover >=455-level LB by (1) restoring richer receipt features, (2) removing noisy shift weights,
# (3) using fold+seed bagging (more robust), (4) simple coarse ensemble + high-risk uplift using q90 model.

import os, sys, re, math, random, warnings, gc
from pathlib import Path
from typing import Tuple, Dict, Optional, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"

TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR     = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"  # last resort only
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"

TARGET_COL = "ed_cost_next3y_usd"

print("=== ITERATION 5: Back-to-success + Bagging + Gated Q90 uplift ===")
print("Key decisions:")
print("  1) Restore richer receipts features: per-code cnt/cost/share, prefix aggregates, top-code cats, TF-IDF+SVD (small).")
print("  2) Drop covariate-shift weighting (AUC~0.52 = mostly noise).")
print("  3) Use robust CatBoost set: MAE(deep), Q50, Q90, logRMSE; bagged over repeats.")
print("  4) Simple coarse ensemble + severity-gated uplift toward Q90 for high-acuity (reduces underprediction).")
print("  5) No global bias shift (it can overfit and hurt LB).\n")

# -------------------------
# Lightweight dependency checks (install only if missing)
# -------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD

try:
    from catboost import CatBoostRegressor, Pool
    from catboost.utils import get_gpu_device_count
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor, Pool
    from catboost.utils import get_gpu_device_count

# -------------------------
# Helpers
# -------------------------
def safe_read_csv(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(p)

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def quantiles(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def ensure_cat_as_str(df: pd.DataFrame, cols: List[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")

def median_fill_numeric(train_df: pd.DataFrame, test_df: pd.DataFrame, numeric_cols: List[str]) -> None:
    for c in numeric_cols:
        med = pd.to_numeric(train_df[c], errors="coerce").median()
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(med)
        test_df[c]  = pd.to_numeric(test_df[c], errors="coerce").fillna(med)

def fill_prefix_numeric_zero(df: pd.DataFrame, prefixes: List[str], exclude: Optional[List[str]] = None) -> None:
    exclude = set(exclude or [])
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes) and (c not in exclude)]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

def detect_gpu() -> bool:
    try:
        return get_gpu_device_count() > 0
    except Exception:
        return False

def print_bin_mae(y_true, y_pred, n_bins=6, name="model"):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    edges = np.quantile(y_true, np.linspace(0, 1, n_bins + 1))
    bins = pd.cut(y_true, bins=edges, include_lowest=True, duplicates="drop")
    df = pd.DataFrame({"bin": bins.astype(str), "ae": np.abs(y_true - y_pred), "resid": (y_true - y_pred)})
    g = df.groupby("bin").agg(mae=("ae","mean"), median_resid=("resid","median"), n=("ae","size"))
    print(f"[{name}] MAE by target-quantile bin:")
    print(g.to_string())

# -------------------------
# Admissions features
# -------------------------
def build_admissions_features(adm_tr: pd.DataFrame, adm_te: pd.DataFrame) -> pd.DataFrame:
    adm_tr = adm_tr.drop(columns=["readmit_30d"], errors="ignore")
    adm_all = pd.concat([adm_tr, adm_te], ignore_index=True)

    if "patient_id" not in adm_all.columns:
        return pd.DataFrame(columns=["patient_id"])

    adm_all["patient_id"] = pd.to_numeric(adm_all["patient_id"], errors="coerce").astype("Int64")
    adm_all = adm_all.dropna(subset=["patient_id"]).copy()
    adm_all["patient_id"] = adm_all["patient_id"].astype(int)

    # numeric columns
    num_cols = ["los_days","acuity_emergent","charlson_band","ed_visits_6m","discharge_weekday"]
    for c in num_cols:
        if c in adm_all.columns:
            adm_all[c] = pd.to_numeric(adm_all[c], errors="coerce")

    g = adm_all.groupby("patient_id")
    out = pd.DataFrame({"patient_id": g.size().index, "adm_n": g.size().values})

    for c in num_cols:
        if c in adm_all.columns:
            out[f"adm_{c}_mean"] = g[c].mean().values
            out[f"adm_{c}_max"]  = g[c].max().values
            out[f"adm_{c}_sum"]  = g[c].sum().values
            out[f"adm_{c}_std"]  = g[c].std(ddof=0).values
            out[f"adm_{c}_median"] = g[c].median().values

    # dx counts/mode
    if "primary_dx" in adm_all.columns:
        dx_ct = pd.crosstab(adm_all["patient_id"], adm_all["primary_dx"])
        dx_ct.columns = [f"adm_dx_cnt_{str(c)}" for c in dx_ct.columns]
        dx_ct = dx_ct.reset_index()
        out = out.merge(dx_ct, on="patient_id", how="left")

        dx_mode = g["primary_dx"].agg(lambda x: x.value_counts().index[0] if len(x) else "Unknown").reset_index()
        dx_mode.columns = ["patient_id", "adm_primary_dx_mode"]
        out = out.merge(dx_mode, on="patient_id", how="left")

        dx_nuniq = g["primary_dx"].nunique().reset_index()
        dx_nuniq.columns = ["patient_id", "adm_dx_nunique"]
        out = out.merge(dx_nuniq, on="patient_id", how="left")

    # charlson distribution
    if "charlson_band" in adm_all.columns:
        try:
            cb = adm_all.copy()
            cb["charlson_band_int"] = pd.to_numeric(cb["charlson_band"], errors="coerce").fillna(0).astype(int)
            ctab = pd.crosstab(cb["patient_id"], cb["charlson_band_int"])
            ctab.columns = [f"adm_charlson_cnt_{c}" for c in ctab.columns]
            ctab = ctab.reset_index()
            out = out.merge(ctab, on="patient_id", how="left")
        except Exception:
            pass

    # fill numerics
    for c in out.columns:
        if c in ["patient_id", "adm_primary_dx_mode"]:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    if "adm_primary_dx_mode" in out.columns:
        out["adm_primary_dx_mode"] = out["adm_primary_dx_mode"].astype("string").fillna("Unknown")

    return out

# -------------------------
# Receipts: robust joblib extraction
# -------------------------
def extract_receipt_dfs(obj) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    headers_df, lineitems_df = None, None
    if isinstance(obj, dict):
        for hk in ["headers_df","headers","header_df","header"]:
            if hk in obj and isinstance(obj[hk], pd.DataFrame):
                headers_df = obj[hk]
                break
        for lk in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if lk in obj and isinstance(obj[lk], pd.DataFrame):
                lineitems_df = obj[lk]
                break
        if lineitems_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and {"patient_id","code"}.issubset(set(v.columns)):
                    lineitems_df = v
                    break
        if headers_df is None:
            for v in obj.values():
                if isinstance(v, pd.DataFrame) and ("patient_id" in v.columns) and (v is not lineitems_df):
                    headers_df = v
                    break
    elif isinstance(obj, (tuple, list)):
        dfs = [x for x in obj if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if {"patient_id","code"}.issubset(set(df.columns)):
                lineitems_df = df
            elif "patient_id" in df.columns:
                headers_df = df
    elif isinstance(obj, pd.DataFrame):
        if {"patient_id","code"}.issubset(set(obj.columns)):
            lineitems_df = obj
        elif "patient_id" in obj.columns:
            headers_df = obj
    return headers_df, lineitems_df

# -------------------------
# Receipts: feature builder (rich but not crazy)
# -------------------------
def build_receipt_features(headers_df: Optional[pd.DataFrame],
                          lineitems_df: Optional[pd.DataFrame],
                          max_codes: int = 120,
                          min_patient_support: int = 10,
                          prefix2_topk: int = 25,
                          svd_dim: int = 16) -> pd.DataFrame:
    if lineitems_df is None or not isinstance(lineitems_df, pd.DataFrame) or len(lineitems_df) == 0:
        # fallback: just patient_id list
        if headers_df is not None and isinstance(headers_df, pd.DataFrame) and "patient_id" in headers_df.columns:
            return headers_df.drop_duplicates("patient_id").copy()
        return pd.DataFrame(columns=["patient_id"])

    li = lineitems_df.copy()

    # detect columns
    if "code" not in li.columns:
        for alt in ["cpt","cpt_code","hcpcs","proc_code"]:
            if alt in li.columns:
                li["code"] = li[alt]
                break
    if "line_total" not in li.columns:
        for alt in ["line_total_usd","total","amount","line_cost","line_total_$","sum_items"]:
            if alt in li.columns:
                li["line_total"] = li[alt]
                break
    if "qty" not in li.columns:
        for alt in ["quantity","q"]:
            if alt in li.columns:
                li["qty"] = li[alt]
                break
    if "unit" not in li.columns:
        if "line_total" in li.columns and "qty" in li.columns:
            li["unit"] = pd.to_numeric(li["line_total"], errors="coerce") / pd.to_numeric(li["qty"], errors="coerce").replace(0, np.nan)
        else:
            li["unit"] = np.nan

    if "patient_id" not in li.columns or "code" not in li.columns or "line_total" not in li.columns:
        return pd.DataFrame(columns=["patient_id"])

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)
    li["code"] = li["code"].astype(str).str.strip().str.upper()
    li["qty"] = pd.to_numeric(li["qty"], errors="coerce").fillna(1.0).astype(float)
    li["unit"] = pd.to_numeric(li["unit"], errors="coerce")
    li["line_total"] = pd.to_numeric(li["line_total"], errors="coerce").fillna(0.0).astype(float)

    # must-keep codes (per prompt)
    must_codes = set([
        "31500","36556","36620","92950","99291","99292",
        "70450","74177","84484","G0378","99285",
        "99281","99282","99283","99284"
    ])

    # choose codes by patient-support (avoid rare noise) + must
    support = li.groupby("code")["patient_id"].nunique().sort_values(ascending=False)
    codes_keep = support[support >= min_patient_support].index.tolist()
    # also ensure max_codes limit by total cost if too many
    if len(codes_keep) > max_codes:
        cost_sum = li.groupby("code")["line_total"].sum().sort_values(ascending=False)
        codes_keep = [c for c in cost_sum.index.tolist() if c in set(codes_keep)]
        codes_keep = codes_keep[:max_codes]
    codes_keep = sorted(set(codes_keep) | must_codes)

    # median unit per code -> deflated
    med_unit = li.groupby("code")["unit"].median()
    li["unit_med_code"] = li["code"].map(med_unit).astype(float)
    li["deflated_line_total"] = li["qty"] * li["unit_med_code"]

    g = li.groupby("patient_id")
    agg = g.agg(
        pdf_total_line_cost=("line_total","sum"),
        pdf_deflated_total=("deflated_line_total","sum"),
        pdf_n_line_items=("code","size"),
        pdf_total_qty=("qty","sum"),
        pdf_n_unique_codes=("code","nunique"),
        pdf_line_max=("line_total","max"),
        pdf_line_mean=("line_total","mean"),
        pdf_line_std=("line_total", lambda x: float(np.std(x, ddof=0))),
        pdf_line_median=("line_total","median"),
        pdf_unit_mean=("unit","mean"),
        pdf_unit_median=("unit","median"),
    ).reset_index()

    agg["pdf_price_index"] = agg["pdf_total_line_cost"] / agg["pdf_deflated_total"].replace(0, np.nan)
    agg["pdf_price_index"] = agg["pdf_price_index"].replace([np.inf, -np.inf], np.nan).fillna(1.0)

    # per-code pivots
    li_sel = li[li["code"].isin(codes_keep)].copy()
    cnt = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="size", fill_value=0)
    cost = li_sel.pivot_table(index="patient_id", columns="code", values="line_total", aggfunc="sum", fill_value=0.0)
    cnt.columns = [f"pdf_cnt_{c}" for c in cnt.columns.astype(str)]
    cost.columns = [f"pdf_cost_{c}" for c in cost.columns.astype(str)]
    wide = cnt.join(cost, how="outer").reset_index()

    out = agg.merge(wide, on="patient_id", how="left")
    fill_prefix_numeric_zero(out, prefixes=["pdf_"])

    denom = out["pdf_total_line_cost"].replace(0, np.nan)
    for c in codes_keep:
        col = f"pdf_cost_{c}"
        if col in out.columns:
            out[f"pdf_share_{c}"] = (out[col] / denom).fillna(0.0)
        else:
            out[f"pdf_share_{c}"] = 0.0

    # high acuity flags
    code_flags = {
        "has_intub_31500": "31500",
        "has_cvc_36556": "36556",
        "has_artline_36620": "36620",
        "has_cpr_92950": "92950",
        "has_ct_head_70450": "70450",
        "has_ct_abdpel_74177": "74177",
        "has_troponin_84484": "84484",
        "has_obs_G0378": "G0378",
        "has_99285": "99285",
    }
    for fname, code in code_flags.items():
        cc = f"pdf_cnt_{code}"
        out[fname] = (out[cc] > 0).astype(int) if cc in out.columns else 0
    crit_cols = [c for c in ["99291","99292"] if f"pdf_cnt_{c}" in out.columns]
    out["has_critical_care"] = (out[[f"pdf_cnt_{c}" for c in crit_cols]].sum(axis=1) > 0).astype(int) if crit_cols else 0

    out["severe_proc_score"] = (
        out.get("has_intub_31500", 0)
        + out.get("has_cvc_36556", 0)
        + out.get("has_artline_36620", 0)
        + out.get("has_cpr_92950", 0)
        + out.get("has_critical_care", 0)
    ).astype(int)

    # E/M
    em_map = {"99281":1, "99282":2, "99283":3, "99284":4, "99285":5}
    em = li[li["code"].isin(em_map.keys())].copy()
    if len(em):
        em["em_level"] = em["code"].map(em_map).astype(float)
        em_agg = em.groupby("patient_id").agg(
            pdf_em_level_mean=("em_level","mean"),
            pdf_em_level_max=("em_level","max"),
            pdf_em_high_frac=("em_level", lambda x: float(np.mean(np.asarray(x) >= 4))),
        ).reset_index()
        out = out.merge(em_agg, on="patient_id", how="left")
    for c in ["pdf_em_level_mean","pdf_em_level_max","pdf_em_high_frac"]:
        if c not in out.columns:
            out[c] = 0.0
    out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_high_frac"]] = out[["pdf_em_level_mean","pdf_em_level_max","pdf_em_high_frac"]].fillna(0.0)

    # distribution metrics (gini/entropy/topshares) across code costs (only kept codes)
    cost_cols = [c for c in out.columns if c.startswith("pdf_cost_")]
    if cost_cols:
        mat = out[cost_cols].to_numpy(dtype=float)
        row_sum = mat.sum(axis=1, keepdims=True)
        p = np.divide(mat, row_sum, out=np.zeros_like(mat), where=row_sum > 0)
        ent = -(p * np.log(np.clip(p, 1e-12, 1.0))).sum(axis=1)
        out["pdf_code_cost_entropy_norm"] = ent / np.log(max(2, mat.shape[1]))

        sorted_mat = np.sort(mat, axis=1)
        s = sorted_mat.sum(axis=1)
        top1 = sorted_mat[:, -1]
        top2 = top1 + (sorted_mat[:, -2] if sorted_mat.shape[1] >= 2 else 0.0)
        out["pdf_cost_top1_share"] = np.divide(top1, s, out=np.zeros_like(top1), where=s > 0)
        out["pdf_cost_top2_share"] = np.divide(top2, s, out=np.zeros_like(top2), where=s > 0)

        n = sorted_mat.shape[1]
        i = np.arange(1, n + 1, dtype=float)
        weighted = (sorted_mat * i.reshape(1, -1)).sum(axis=1)
        gini = np.zeros_like(s, dtype=float)
        mask = s > 0
        gini[mask] = (2.0 * weighted[mask]) / (n * s[mask]) - (n + 1) / n
        out["pdf_cost_gini"] = gini
    else:
        out["pdf_code_cost_entropy_norm"] = 0.0
        out["pdf_cost_top1_share"] = 0.0
        out["pdf_cost_top2_share"] = 0.0
        out["pdf_cost_gini"] = 0.0

    # selected cost sums (helps like earlier pdf_cost_selected_sum)
    selected_codes = set(["31500","36556","36620","92950","99291","99292","70450","74177","84484","G0378","99285"])
    sel_cols = [f"pdf_cost_{c}" for c in selected_codes if f"pdf_cost_{c}" in out.columns]
    if sel_cols:
        out["pdf_cost_selected_sum"] = out[sel_cols].sum(axis=1)
        out["pdf_cost_selected_share"] = (out["pdf_cost_selected_sum"] / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    else:
        out["pdf_cost_selected_sum"] = 0.0
        out["pdf_cost_selected_share"] = 0.0

    # prefix2 aggregates (top by overall cost)
    li_sel["prefix2"] = li_sel["code"].str[:2]
    pref2_cost = li_sel.pivot_table(index="patient_id", columns="prefix2", values="line_total", aggfunc="sum", fill_value=0.0)
    pref2_cnt  = li_sel.pivot_table(index="patient_id", columns="prefix2", values="line_total", aggfunc="size", fill_value=0)
    pref2_tot = pref2_cost.sum(axis=0).sort_values(ascending=False)
    keep_p2 = pref2_tot.head(prefix2_topk).index.tolist()
    pref2_cost = pref2_cost[keep_p2]
    pref2_cnt  = pref2_cnt[keep_p2]
    pref2_cost.columns = [f"pdf_pref2_cost_{c}" for c in pref2_cost.columns.astype(str)]
    pref2_cnt.columns  = [f"pdf_pref2_cnt_{c}"  for c in pref2_cnt.columns.astype(str)]
    pref2 = pref2_cnt.join(pref2_cost, how="outer").reset_index()
    out = out.merge(pref2, on="patient_id", how="left")
    fill_prefix_numeric_zero(out, prefixes=["pdf_pref2_"])

    # topcode categorical by cost (top1/top2)
    cc = li.groupby(["patient_id","code"])["line_total"].sum().reset_index()
    cc = cc.sort_values(["patient_id","line_total"], ascending=[True, False])
    top1 = cc.groupby("patient_id").head(1).rename(columns={"code":"pdf_topcode_cost1","line_total":"pdf_topcode_cost1_amt"})
    top2 = cc.groupby("patient_id").head(2).groupby("patient_id").tail(1).rename(columns={"code":"pdf_topcode_cost2","line_total":"pdf_topcode_cost2_amt"})
    topc = top1[["patient_id","pdf_topcode_cost1","pdf_topcode_cost1_amt"]].merge(
        top2[["patient_id","pdf_topcode_cost2","pdf_topcode_cost2_amt"]], on="patient_id", how="left"
    )
    out = out.merge(topc, on="patient_id", how="left")
    for c in ["pdf_topcode_cost1","pdf_topcode_cost2"]:
        out[c] = out[c].astype("string").fillna("Unknown")
    out["pdf_topcode_cost1_amt"] = pd.to_numeric(out.get("pdf_topcode_cost1_amt", 0.0), errors="coerce").fillna(0.0)
    out["pdf_topcode_cost2_amt"] = pd.to_numeric(out.get("pdf_topcode_cost2_amt", 0.0), errors="coerce").fillna(0.0)
    out["pdf_topcode_cost1_share"] = (out["pdf_topcode_cost1_amt"] / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pdf_topcode_cost2_share"] = (out["pdf_topcode_cost2_amt"] / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    # TF-IDF+SVD embedding (small)
    try:
        code_ct = li.groupby(["patient_id","code"]).size().reset_index(name="cnt")
        code_ct["rep"] = code_ct["cnt"].clip(upper=3).astype(int)
        code_ct["tok_rep"] = (code_ct["code"].astype(str) + " ") * code_ct["rep"]
        doc = code_ct.groupby("patient_id")["tok_rep"].sum().reset_index()
        doc.columns = ["patient_id", "pdf_code_seq"]
        vec = TfidfVectorizer(token_pattern=r"\S+", min_df=1, max_features=400)
        X_tfidf = vec.fit_transform(doc["pdf_code_seq"].fillna(""))
        dim_eff = min(svd_dim, max(2, X_tfidf.shape[1] - 1))
        svd = TruncatedSVD(n_components=dim_eff, random_state=SEED)
        X_svd = svd.fit_transform(X_tfidf)
        svd_cols = [f"pdf_code_svd_{i}" for i in range(dim_eff)]
        svd_df = pd.DataFrame(X_svd, columns=svd_cols)
        svd_df.insert(0, "patient_id", doc["patient_id"].values)
        out = out.merge(svd_df, on="patient_id", how="left")
        fill_prefix_numeric_zero(out, prefixes=["pdf_code_svd_"])
        print(f"[receipts] TF-IDF+SVD: vocab={X_tfidf.shape[1]} dim={dim_eff}")
    except Exception as e:
        print(f"[receipts] TF-IDF+SVD skipped: {e}")

    # merge header extras if present (do NOT rely on pdf_total; sum_items is reliable via totals)
    if headers_df is not None and isinstance(headers_df, pd.DataFrame) and "patient_id" in headers_df.columns:
        cols = ["patient_id"] + [c for c in ["pdf_total","sum_items","zip3_pdf","insurance_pdf"] if c in headers_df.columns]
        hd = headers_df[cols].copy()
        out = out.merge(hd, on="patient_id", how="left")
        if "pdf_total" in out.columns:
            out["pdf_total_missing"] = out["pdf_total"].isna().astype(int)
            out["pdf_total"] = pd.to_numeric(out["pdf_total"], errors="coerce").fillna(0.0)
        if "sum_items" in out.columns:
            out["sum_items"] = pd.to_numeric(out["sum_items"], errors="coerce").fillna(0.0)
        if "zip3_pdf" in out.columns:
            out["zip3_pdf"] = out["zip3_pdf"].apply(standardize_zip3)
            out["zip3_pdf"] = out["zip3_pdf"].astype("string").fillna("Unknown")
        if "insurance_pdf" in out.columns:
            out["insurance_pdf"] = out["insurance_pdf"].astype("string").fillna("Unknown")

    # final fill
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in out.columns:
        if c == "patient_id":
            continue
        if out[c].dtype.kind in "biufc":
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    print(f"[receipts] codes_keep={len(codes_keep)} | prefix2_keep={len(keep_p2)} | receipt_feat_cols={out.shape[1]-1}")
    return out

# -------------------------
# Derived features
# -------------------------
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["prior_cost"] = pd.to_numeric(df.get("prior_ed_cost_5y_usd", np.nan), errors="coerce").fillna(0.0)
    df["prior_visits"] = pd.to_numeric(df.get("prior_ed_visits_5y", np.nan), errors="coerce").fillna(0.0)
    df["log_prior_cost"] = np.log1p(df["prior_cost"])
    df["log_prior_visits"] = np.log1p(df["prior_visits"])
    df["cost_per_visit"] = df["prior_cost"] / np.maximum(1.0, df["prior_visits"])
    df["log_cost_per_visit"] = np.log1p(df["cost_per_visit"])
    df["cost_per_year"] = df["prior_cost"] / 5.0
    df["baseline_next3y"] = df["prior_cost"] * (3.0/5.0)
    df["prior_cost_cap20k"] = np.minimum(df["prior_cost"], 20000.0)
    df["log_prior_cost_cap20k"] = np.log1p(df["prior_cost_cap20k"])

    # key interactions (worked before)
    if "severe_proc_score" in df.columns:
        df["priorcost_x_severeproc"] = df["log_prior_cost"] * pd.to_numeric(df["severe_proc_score"], errors="coerce").fillna(0.0)
    if "pdf_cost_top1_share" in df.columns:
        df["logpriorcost_x_top1share"] = df["log_prior_cost"] * pd.to_numeric(df["pdf_cost_top1_share"], errors="coerce").fillna(0.0)
    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        df["age"] = df["age"].fillna(df["age"].median())
        df["age_x_logpriorcost"] = df["age"] * df["log_prior_cost"]

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

# -------------------------
# Load data
# -------------------------
train = safe_read_csv(TRAIN_PATH)
test  = safe_read_csv(TEST_PATH)
patients = safe_read_csv(PATIENTS_PATH)
adm_tr = safe_read_csv(ADM_TRAIN_PATH)
adm_te = safe_read_csv(ADM_TEST_PATH)

for df in [train, test, patients, adm_tr, adm_te]:
    if "patient_id" in df.columns:
        df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

if "zip3" in patients.columns:
    patients["zip3"] = patients["zip3"].apply(standardize_zip3)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)
log_shape("admissions_train", adm_tr)
log_shape("admissions_test", adm_te)

print("\n[target stats]")
print(train[TARGET_COL].describe().to_string())
print("\n[prior cost stats]")
print(train["prior_ed_cost_5y_usd"].describe().to_string())

# -------------------------
# Build admissions + receipts features
# -------------------------
adm_feat = build_admissions_features(adm_tr, adm_te)
print(f"\n[admissions] patient-level features: {adm_feat.shape}")

receipt_feat = None
joblib_path = Path(RECEIPTS_JOBLIB_PATH)
if joblib_path.exists():
    try:
        obj = joblib_load(joblib_path)
        headers_df, lineitems_df = extract_receipt_dfs(obj)
        # If already patient-level features and no lineitems, use directly
        if isinstance(obj, pd.DataFrame) and ("patient_id" in obj.columns) and ("code" not in obj.columns) and (obj["patient_id"].nunique() == len(obj)):
            receipt_feat = obj.copy()
            print(f"[receipts] joblib is patient-level features DF: {receipt_feat.shape}")
        else:
            receipt_feat = build_receipt_features(headers_df, lineitems_df, max_codes=120, min_patient_support=10, prefix2_topk=25, svd_dim=16)
            print(f"[receipts] built from joblib lineitems: {receipt_feat.shape}")
    except Exception as e:
        print(f"[receipts] ERROR loading/building from joblib: {e}")
        receipt_feat = pd.DataFrame({"patient_id": pd.concat([train["patient_id"], test["patient_id"]]).unique()})
else:
    print("[receipts] receipts_parsed.joblib not found; proceeding without receipt features (will likely be worse).")
    receipt_feat = pd.DataFrame({"patient_id": pd.concat([train["patient_id"], test["patient_id"]]).unique()})

# ensure unique pid
receipt_feat["patient_id"] = pd.to_numeric(receipt_feat["patient_id"], errors="coerce").astype("Int64")
receipt_feat = receipt_feat.dropna(subset=["patient_id"]).copy()
receipt_feat["patient_id"] = receipt_feat["patient_id"].astype(int)
receipt_feat = receipt_feat.drop_duplicates("patient_id", keep="last")

all_pids = pd.concat([train["patient_id"], test["patient_id"]]).astype(int).unique()
cov = receipt_feat["patient_id"].nunique() / len(all_pids)
print(f"[receipts] coverage: {receipt_feat['patient_id'].nunique()}/{len(all_pids)} = {cov:.3f}")

# -------------------------
# Merge
# -------------------------
def build_model_df(ed_df: pd.DataFrame, patients: pd.DataFrame, adm_feat: pd.DataFrame, receipt_feat: pd.DataFrame) -> pd.DataFrame:
    df = ed_df.merge(patients, on="patient_id", how="left")
    if adm_feat is not None and len(adm_feat):
        df = df.merge(adm_feat, on="patient_id", how="left")
    if receipt_feat is not None and len(receipt_feat):
        df = df.merge(receipt_feat, on="patient_id", how="left", suffixes=("", "_rcpt"))

    # fill numeric blocks
    fill_prefix_numeric_zero(df, prefixes=["adm_"], exclude=["adm_primary_dx_mode"])
    fill_prefix_numeric_zero(df, prefixes=["pdf_"], exclude=["pdf_topcode_cost1","pdf_topcode_cost2","zip3_pdf","insurance_pdf"])
    fill_prefix_numeric_zero(df, prefixes=["has_"])
    fill_prefix_numeric_zero(df, prefixes=["severe_"])

    # cats from receipts if any
    for c in ["zip3_pdf","insurance_pdf","pdf_topcode_cost1","pdf_topcode_cost2"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")
            if c == "zip3_pdf":
                df[c] = df[c].apply(standardize_zip3).astype("string").fillna("Unknown")

    if "adm_primary_dx_mode" in df.columns:
        df["adm_primary_dx_mode"] = df["adm_primary_dx_mode"].astype("string").fillna("Unknown")

    return df

train_df = build_model_df(train, patients, adm_feat, receipt_feat)
test_df  = build_model_df(test,  patients, adm_feat, receipt_feat)

train_df = add_derived_features(train_df)
test_df  = add_derived_features(test_df)

print("\n[merged data]")
log_shape("train_df", train_df)
log_shape("test_df", test_df)

# sanity: receipt totals equal prior cost (if available)
if "pdf_total_line_cost" in train_df.columns:
    diff = np.abs(pd.to_numeric(train_df["pdf_total_line_cost"], errors="coerce").fillna(0.0) - pd.to_numeric(train_df["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0))
    print(f"[sanity] |pdf_total_line_cost - prior_ed_cost_5y_usd| mean={diff.mean():.6e} max={diff.max():.6e}")

# -------------------------
# Prepare X/y
# -------------------------
y = pd.to_numeric(train_df[TARGET_COL], errors="coerce").values.astype(float)
test_patient_id = test_df["patient_id"].astype(int).values

X_train = train_df.drop(columns=[TARGET_COL, "patient_id"], errors="ignore").copy()
X_test  = test_df.drop(columns=["patient_id"], errors="ignore").copy()

cat_cols = [c for c in [
    "primary_chronic","sex","insurance","zip3",
    "adm_primary_dx_mode",
    "zip3_pdf","insurance_pdf",
    "pdf_topcode_cost1","pdf_topcode_cost2"
] if c in X_train.columns and c in X_test.columns]

ensure_cat_as_str(X_train, cat_cols)
ensure_cat_as_str(X_test,  cat_cols)

num_cols = [c for c in X_train.columns if c not in cat_cols]
median_fill_numeric(X_train, X_test, num_cols)

# quick missing checks
print("\n[features]")
print(f"Total features: {X_train.shape[1]} | Numeric: {len(num_cols)} | Cat: {len(cat_cols)}")
print("Categorical columns:", cat_cols)
print("[missing] train missing cells:", int(X_train.isna().sum().sum()), "| test missing cells:", int(X_test.isna().sum().sum()))

# -------------------------
# CV split (robust to small strata)
# -------------------------
# stratify by chronic + prior cost bins
try:
    bins = pd.qcut(train_df["prior_ed_cost_5y_usd"], q=8, duplicates="drop")
    strat = (train_df["primary_chronic"].astype(str) + "_" + bins.astype(str)).values
except Exception:
    strat = train_df["primary_chronic"].astype(str).values

vc = pd.Series(strat).value_counts()
min_class = int(vc.min())
desired_splits = 7
n_splits = min(desired_splits, max(3, min_class))
if min_class < 3:
    n_splits = 5
print(f"\n[CV] strat classes={vc.shape[0]} | min_class_count={min_class} | using n_splits={n_splits}")

N_REPEATS = 2  # bagging for robustness

use_gpu = detect_gpu()
print(f"[catboost] GPU available: {use_gpu}")

# -------------------------
# Model definitions (CPU for MAE/Quantile; logRMSE can use GPU safely)
# -------------------------
def cb_params(task_type: str, **kwargs):
    base = dict(
        iterations=12000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=12,
        random_strength=1.0,
        bagging_temperature=0.8,
        min_data_in_leaf=24,
        allow_writing_files=False,
        verbose=False,
        task_type=task_type,
        random_seed=SEED,
    )
    if task_type == "CPU":
        base["thread_count"] = -1
    base.update(kwargs)
    return base

models = [
    dict(name="cb_mae_deep", kind="raw", loss="MAE", params=cb_params("CPU", loss_function="MAE", eval_metric="MAE")),
    dict(name="cb_q50_deep", kind="raw", loss="Q50", params=cb_params("CPU", loss_function="Quantile:alpha=0.5", eval_metric="MAE")),
    dict(name="cb_q90_deep", kind="raw", loss="Q90", params=cb_params("CPU", loss_function="Quantile:alpha=0.9", eval_metric="MAE")),
    dict(name="cb_logrmse_d7", kind="log", loss="LOG_RMSE", params=cb_params("GPU" if use_gpu else "CPU",
                                                                              iterations=14000,
                                                                              depth=7,
                                                                              learning_rate=0.03,
                                                                              l2_leaf_reg=8,
                                                                              bagging_temperature=0.7,
                                                                              loss_function="RMSE",
                                                                              eval_metric="RMSE")),
]

# -------------------------
# CV training (bagged over repeats)
# -------------------------
oof_sum = {m["name"]: np.zeros(len(X_train), dtype=float) for m in models}
test_sum = {m["name"]: np.zeros(len(X_test), dtype=float) for m in models}
fold_rows = []
best_iters = {m["name"]: [] for m in models}

for rep in range(N_REPEATS):
    rs = SEED + 1000*rep
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rs)

    print(f"\n[CV] repeat {rep+1}/{N_REPEATS} (random_state={rs})")
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, strat), 1):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        row = {"rep": rep+1, "fold": fold}
        for m in models:
            name = m["name"]
            params = dict(m["params"])
            params["random_seed"] = rs + fold * 17 + (hash(name) % 1000)

            if m["kind"] == "log":
                y_tr_fit = np.log1p(y_tr)
                y_va_fit = np.log1p(y_va)
            else:
                y_tr_fit = y_tr
                y_va_fit = y_va

            tr_pool = Pool(X_tr, y_tr_fit, cat_features=cat_cols) if cat_cols else Pool(X_tr, y_tr_fit)
            va_pool = Pool(X_va, y_va_fit, cat_features=cat_cols) if cat_cols else Pool(X_va, y_va_fit)

            reg = CatBoostRegressor(**params)

            try:
                reg.fit(tr_pool, eval_set=va_pool, use_best_model=True, early_stopping_rounds=500)
            except Exception as e:
                # fallback to CPU if GPU fails
                if params.get("task_type") == "GPU":
                    params_cpu = dict(params)
                    params_cpu["task_type"] = "CPU"
                    params_cpu["thread_count"] = -1
                    reg = CatBoostRegressor(**params_cpu)
                    reg.fit(tr_pool, eval_set=va_pool, use_best_model=True, early_stopping_rounds=500)
                    print(f"[warn] GPU failed for {name} rep{rep+1} fold{fold}: {e} -> retrained on CPU")
                else:
                    raise

            try:
                best_iters[name].append(int(reg.get_best_iteration()))
            except Exception:
                pass

            if m["kind"] == "log":
                pred_va = np.expm1(reg.predict(X_va))
                pred_te = np.expm1(reg.predict(X_test))
            else:
                pred_va = reg.predict(X_va)
                pred_te = reg.predict(X_test)

            pred_va = np.clip(pred_va, 0.0, None)
            pred_te = np.clip(pred_te, 0.0, None)

            oof_sum[name][va_idx] += pred_va
            test_sum[name] += pred_te

            row[name] = mae(y_va, pred_va)

        fold_rows.append(row)
        fold_msg = " | ".join([f"{k}={row[k]:.2f}" for k in ["cb_mae_deep","cb_q50_deep","cb_q90_deep","cb_logrmse_d7"]])
        print(f"  fold {fold}/{n_splits}: {fold_msg}")

# average
oof = {k: v / N_REPEATS for k, v in oof_sum.items()}
test_pred = {k: v / (N_REPEATS * n_splits) for k, v in test_sum.items()}

# model summaries
print("\n[OOF summary]")
for m in models:
    name = m["name"]
    m_mae = mae(y, oof[name])
    med_res = float(np.median(y - oof[name]))
    bi = int(np.median(best_iters[name])) if best_iters[name] else None
    print(f"  {name:12s} MAE={m_mae:.4f} | median_resid={med_res:+.2f} | best_iter_med={bi}")
    print(f"    pred_q={quantiles(oof[name], qs=(0,0.01,0.05,0.5,0.95,0.99,1.0))}")

# bin diagnostics for main models
print("")
print_bin_mae(y, oof["cb_mae_deep"], name="cb_mae_deep")
print("")
print_bin_mae(y, oof["cb_logrmse_d7"], name="cb_logrmse_d7")

# -------------------------
# Ensemble: coarse weight grid + severity-gated uplift toward Q90
# -------------------------
p_mae = oof["cb_mae_deep"]
p_q50 = oof["cb_q50_deep"]
p_q90 = oof["cb_q90_deep"]
p_log = oof["cb_logrmse_d7"]

t_mae = test_pred["cb_mae_deep"]
t_q50 = test_pred["cb_q50_deep"]
t_q90 = test_pred["cb_q90_deep"]
t_log = test_pred["cb_logrmse_d7"]

# gate based on severity features (NO TARGET used)
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def build_gate(df: pd.DataFrame) -> np.ndarray:
    sev = pd.to_numeric(df.get("severe_proc_score", 0), errors="coerce").fillna(0.0).values
    cc  = pd.to_numeric(df.get("has_critical_care", 0), errors="coerce").fillna(0.0).values
    em4 = (pd.to_numeric(df.get("pdf_em_level_max", 0), errors="coerce").fillna(0.0).values >= 4).astype(float)
    lp  = pd.to_numeric(df.get("log_prior_cost", 0), errors="coerce").fillna(0.0).values
    ch  = pd.to_numeric(df.get("adm_charlson_band_mean", 0), errors="coerce").fillna(0.0).values

    lp_z = (lp - np.median(lp)) / (np.std(lp) + 1e-9)
    ch_z = (ch - np.median(ch)) / (np.std(ch) + 1e-9)

    score = 0.95*sev + 1.15*cc + 0.55*em4 + 0.30*lp_z + 0.18*ch_z
    g = sigmoid(score - 0.35)
    return g.astype(float)

gate_tr = build_gate(train_df)
gate_te = build_gate(test_df)

# Coarse grid search to avoid overfitting
w_vals = np.arange(0.0, 1.0 + 1e-9, 0.10)
a_vals = np.arange(0.0, 0.81 + 1e-9, 0.10)

best = {"mae": 1e18, "w_mae": None, "w_q50": None, "alpha": None, "pred": None}
cands = []

for w_mae in w_vals:
    for w_q50 in w_vals:
        if w_mae + w_q50 > 1.0:
            continue
        w_log = 1.0 - w_mae - w_q50
        base = w_mae*p_mae + w_q50*p_q50 + w_log*p_log

        for a in a_vals:
            pred = base + a * gate_tr * (p_q90 - base)  # uplift high-risk toward q90
            pred = np.clip(pred, 0.0, None)
            m = mae(y, pred)
            # soft penalty if floor too high (helps LB sometimes)
            q01 = float(np.quantile(pred, 0.01))
            pen = 0.0
            if q01 > 950:
                pen += (q01 - 950) * 0.01
            obj = m + pen

            if obj < best["mae"]:
                best = {"mae": obj, "w_mae": float(w_mae), "w_q50": float(w_q50), "alpha": float(a), "pred": pred}

            cands.append((m, float(w_mae), float(w_q50), float(a), q01))

# choose simplest among near-best MAE (not obj), to reduce LB overfit
best_mae_plain = float(mae(y, best["pred"]))
eps = 0.25
near = [x for x in cands if x[0] <= best_mae_plain + eps]
near.sort(key=lambda x: (x[0], abs(x[1]-0.5) + 0.7*abs(x[2]-0.2) + 0.5*abs(x[3]-0.3)))
chosen = near[0]
ens_w_mae, ens_w_q50, ens_a = chosen[1], chosen[2], chosen[3]
ens_w_log = 1.0 - ens_w_mae - ens_w_q50

ens_oof = ens_w_mae*p_mae + ens_w_q50*p_q50 + ens_w_log*p_log
ens_oof = np.clip(ens_oof + ens_a*gate_tr*(p_q90 - ens_oof), 0.0, None)
ens_mae = float(mae(y, ens_oof))

print("\n[ENS]")
print(f"  best_obj (with floor penalty)={best['mae']:.4f} at w_mae={best['w_mae']:.2f} w_q50={best['w_q50']:.2f} w_log={1-best['w_mae']-best['w_q50']:.2f} alpha={best['alpha']:.2f}")
print(f"  chosen (simplicity within eps) OOF_MAE={ens_mae:.4f} at w_mae={ens_w_mae:.2f} w_q50={ens_w_q50:.2f} w_log={ens_w_log:.2f} alpha={ens_a:.2f}")
print("  OOF pred quantiles:", quantiles(ens_oof, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("")
print_bin_mae(y, ens_oof, name=f"ENS(w_mae={ens_w_mae:.2f},w_q50={ens_w_q50:.2f},w_log={ens_w_log:.2f},a={ens_a:.2f})")

# ensemble test prediction
ens_test = ens_w_mae*t_mae + ens_w_q50*t_q50 + ens_w_log*t_log
ens_test = np.clip(ens_test + ens_a*gate_te*(t_q90 - ens_test), 0.0, None)

# -------------------------
# Train final model(s) on full train (requirement) + feature importance
# -------------------------
print("\n[Full-train] fitting cb_mae_deep for feature importance...")
full_params = dict(models[0]["params"])
# use median best iteration + buffer (avoid massive overfit)
if best_iters["cb_mae_deep"]:
    it = int(np.median(best_iters["cb_mae_deep"]))
    full_params["iterations"] = int(max(600, it + 200))
full_params["random_seed"] = SEED

full_pool = Pool(X_train, y, cat_features=cat_cols) if cat_cols else Pool(X_train, y)
full_model = CatBoostRegressor(**full_params)
full_model.fit(full_pool, verbose=False)

try:
    imp = full_model.get_feature_importance()
    imp_df = pd.DataFrame({"feature": X_train.columns.tolist(), "importance": imp}).sort_values("importance", ascending=False).head(10)
    print("\n[Top 10 feature importances] (cb_mae_deep full-train)")
    print(imp_df.to_string(index=False))
except Exception as e:
    print(f"[warn] Could not compute importances: {e}")

# -------------------------
# Write submission
# -------------------------
sub = pd.DataFrame({"patient_id": test_patient_id, "ed_cost_next3y_usd": ens_test.astype(float)})
sub = sub[["patient_id","ed_cost_next3y_usd"]]

# sanity checks
print("\n[FINAL sanity checks]")
print("overall CV MAE (ensemble OOF):", f"{ens_mae:.4f}")
print(f"ensemble used: w_mae={ens_w_mae:.2f}, w_q50={ens_w_q50:.2f}, w_log={ens_w_log:.2f}, alpha_uplift={ens_a:.2f}")
print("submission shape:", sub.shape)
print("submission columns exactly:", list(sub.columns))
print("any NaNs in preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(sub["ed_cost_next3y_usd"].min()), float(sub["ed_cost_next3y_usd"].median()), float(sub["ed_cost_next3y_usd"].max()))
print("pred quantiles:", quantiles(sub["ed_cost_next3y_usd"].values))

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)
print("\nSaved submission to:", str(out_path))

print("\nPaste back CV MAE and these logs for next iteration:")
print("  - OOF MAE per model + ensemble OOF MAE")
print("  - chosen ensemble weights/alpha")
print("  - pred quantiles + leaderboard MAE")


=== ITERATION 5: Back-to-success + Bagging + Gated Q90 uplift ===
Key decisions:
  1) Restore richer receipts features: per-code cnt/cost/share, prefix aggregates, top-code cats, TF-IDF+SVD (small).
  2) Drop covariate-shift weighting (AUC~0.52 = mostly noise).
  3) Use robust CatBoost set: MAE(deep), Q50, Q90, logRMSE; bagged over repeats.
  4) Simple coarse ensemble + severity-gated uplift toward Q90 for high-acuity (reduces underprediction).
  5) No global bias shift (it can overfit and hurt LB).

[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.66 MB
[admissions_train] shape=(5000, 9) | cols=9 | mem=0.57 MB
[admissions_test] shape=(5000, 8) | cols=8 | mem=0.53 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[prior cost stats]
count     2000.0

# Iteration 6
- 452.2421

In [14]:
# === APPLY SOLUTION: v3 Iteration 16 style (fast ~minutes, MAE-focused, no over-engineering) ===
# Goal: reproduce the practical recipe that got ~MAE452 quickly:
#   - Low-dimensional, clinically grouped receipt features (NOT huge per-code pivots)
#   - Simple ablation-validated admissions aggregates (charlson_max/mean, pct_emergent)
#   - 3 CatBoost RMSE variants + rsm regularization + pruned-feature model
#   - Seed-bagging + OOF-optimized ensemble weights (scipy if available, else random search)
# Output: D:\AgentDs\agent_ds_healthcare\submission.csv with correct columns.

import os, re, gc, sys, math, warnings, random
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Required Windows paths
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"  # last resort only
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
TARGET = "ed_cost_next3y_usd"

print("="*80)
print("APPLY PIPELINE (from v3 Iter15/16): low-dim receipt features + CatBoost(RMSE) rsm + pruned model + OOF weights")
print("Expected: fast iteration + avoids overfit from huge CPT pivots.")
print("="*80)

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# scipy optional
try:
    from scipy.optimize import minimize
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# -----------------------------
# Config (match v3 Iter16 spirit)
# -----------------------------
class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 5
    OUTPUT_PATH = OUT_SUB_PATH
    RECEIPTS_DIR = RECEIPTS_PDF_DIR
    RECEIPTS_JOBLIB = RECEIPTS_JOBLIB_PATH
    # PDF features to KEEP (>= ~1% importance in v15 per v3 notebook)
    PDF_KEEP_COLS = [
        "cost_per_em", "cost_hhi", "pct_cost_procedure",
        "n_high_acuity_total", "pct_cost_critical", "pct_cost_em",
    ]

# -----------------------------
# Utils
# -----------------------------
def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    # strip trailing .0
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    # remove spaces
    s = re.sub(r"\s+", "", s)
    return s

def _safe_num(x, default=0.0):
    try:
        if x is None: return default
        v = float(x)
        if np.isnan(v) or np.isinf(v): return default
        return v
    except Exception:
        return default

def build_pdf_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    """Vectorized reconstruction of v3 'extract_receipt_features' outputs using parsed lineitems."""
    li = li.copy()

    # Find columns
    if "patient_id" not in li.columns:
        raise ValueError("lineitems missing patient_id")

    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code"]:
        if c in li.columns:
            code_col = c
            break
    if code_col is None:
        raise ValueError("lineitems missing code/cpt column")

    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"]:
        if c in li.columns:
            total_col = c
            break
    if total_col is None:
        raise ValueError("lineitems missing line total column")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()

    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    # remove negative weirdness
    li.loc[li["amt"] < 0, "amt"] = 0.0

    # totals / basic
    total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")

    # cost_hhi (per LINE item shares)
    li = li.join(total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)
    share = (li["amt"] / denom).fillna(0.0)
    li["share_sq"] = share * share
    cost_hhi = li.groupby("patient_id")["share_sq"].sum().rename("cost_hhi")

    # flags by code
    code_series = li["code"]
    is_em = code_series.isin(["99281","99282","99283","99284","99285"])
    em_level = pd.Series(np.zeros(len(li), dtype=float), index=li.index)
    for k,v in {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}.items():
        em_level.loc[code_series == k] = float(v)

    is_crit = code_series.isin(["99291","99292"])
    is_obs = code_series.str.startswith("G037", na=False)
    is_high = code_series.isin(["31500","36556","32551","36620","92950"])

    # lab/imaging/procedure buckets (match v3 categorization)
    # lab: 80000-89999
    # imaging: 70000-79999
    # procedure: 10000-69999
    code_num = pd.to_numeric(code_series.where(code_series.str.fullmatch(r"\d+"), None), errors="coerce")
    is_lab = (code_series.str.startswith("8", na=False)) & (code_series.str.len() == 5) & (code_num.between(80000, 89999))
    is_imaging = (code_series.str.startswith("7", na=False)) & (code_series.str.len() == 5) & (code_num.between(70000, 79999))
    is_proc_general = (code_series.str[0].str.isdigit()) & (code_num.between(10000, 69999))
    is_proc_any = is_high | (is_proc_general & (~is_high))

    # category (mutually exclusive) for n_categories
    cat = np.select(
        [is_em, is_crit, is_obs, is_high, is_lab, is_imaging, (is_proc_general & (~is_high))],
        ["em","critical_care","observation","high_acuity_proc","lab","imaging","procedure"],
        default="other"
    )
    li["cat"] = cat

    # E&M stats
    li["is_em"] = is_em.astype(int)
    li["em_level"] = em_level.astype(float)
    li["is_high_em"] = ((li["em_level"] >= 4) & (li["is_em"] == 1)).astype(int)

    n_em_codes = li.groupby("patient_id")["is_em"].sum().rename("n_em_codes")
    max_em_level = li.groupby("patient_id")["em_level"].max().rename("max_em_level")
    avg_em_level = li.groupby("patient_id").apply(lambda g: float(g.loc[g["is_em"]==1, "em_level"].mean()) if (g["is_em"].sum()>0) else 0.0).rename("avg_em_level")
    n_high_em = li.groupby("patient_id")["is_high_em"].sum().rename("n_high_em")
    em_total = li.loc[li["is_em"]==1].groupby("patient_id")["amt"].sum().rename("em_total")

    # cost mix + procedure counts
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    n_procedures = li.loc[is_proc_any].groupby("patient_id")["amt"].size().rename("n_procedures")

    # flags any
    has_critical_care = li.groupby("patient_id").apply(lambda g: int((g["code"].isin(["99291","99292"])).any())).rename("has_critical_care")
    has_high_acuity = li.groupby("patient_id").apply(lambda g: int((g["code"].isin(["31500","36556","32551","36620","92950"])).any())).rename("has_high_acuity")
    has_observation = li.groupby("patient_id").apply(lambda g: int((g["code"].str.startswith("G037", na=False)).any())).rename("has_observation")
    has_imaging = li.groupby("patient_id").apply(lambda g: int(((g["cat"]=="imaging")).any())).rename("has_imaging")

    # top residual code indicators
    def has_code(code):
        return li.groupby("patient_id").apply(lambda g: int((g["code"]==code).any())).rename(f"has_{code}")
    # use exact names from v3 (important)
    has_intub_31500 = li.groupby("patient_id").apply(lambda g: int((g["code"]=="31500").any())).rename("has_intub_31500")
    has_cvc_36556   = li.groupby("patient_id").apply(lambda g: int((g["code"]=="36556").any())).rename("has_cvc_36556")
    has_cpr_92950   = li.groupby("patient_id").apply(lambda g: int((g["code"]=="92950").any())).rename("has_cpr_92950")
    has_artline_36620 = li.groupby("patient_id").apply(lambda g: int((g["code"]=="36620").any())).rename("has_artline_36620")
    has_ct_head_70450 = li.groupby("patient_id").apply(lambda g: int((g["code"]=="70450").any())).rename("has_ct_head_70450")
    has_99285 = li.groupby("patient_id").apply(lambda g: int((g["code"]=="99285").any())).rename("has_99285")
    has_ct_abdpel_74177 = li.groupby("patient_id").apply(lambda g: int((g["code"]=="74177").any())).rename("has_ct_abdpel_74177")
    has_troponin_84484 = li.groupby("patient_id").apply(lambda g: int((g["code"]=="84484").any())).rename("has_troponin_84484")
    has_obs_G0378 = li.groupby("patient_id").apply(lambda g: int((g["code"]=="G0378").any())).rename("has_obs_G0378")

    # n_categories
    n_categories = li.groupby("patient_id")["cat"].nunique().rename("n_categories")

    # assemble
    out = pd.concat([
        n_unique_codes, cost_hhi,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_categories, n_procedures,
        total
    ], axis=1).reset_index()

    # totals for pct
    out = out.merge(em_total.reset_index(), on="patient_id", how="left")
    out = out.merge(crit_total.reset_index(), on="patient_id", how="left")
    out = out.merge(proc_total.reset_index(), on="patient_id", how="left")
    for c in ["em_total","crit_total","proc_total"]:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    out["receipt_total"] = pd.to_numeric(out["receipt_total"], errors="coerce").fillna(0.0)
    denom2 = out["receipt_total"].replace(0.0, np.nan)

    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    # cost_per_em (v3 uses total / n_em_codes else total)
    out["n_em_codes"] = pd.to_numeric(out["n_em_codes"], errors="coerce").fillna(0.0)
    out["cost_per_em"] = np.where(out["n_em_codes"] > 0, out["receipt_total"] / out["n_em_codes"], out["receipt_total"])

    # composite: number of high-acuity indicators present
    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int)
        + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int)
        + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    # cleanup helper cols
    drop_cols = ["em_total","crit_total","proc_total","receipt_total"]
    out.drop(columns=[c for c in drop_cols if c in out.columns], inplace=True)

    # fill numeric
    for c in out.columns:
        if c == "patient_id": continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    return out

def load_receipts_joblib(joblib_path: str, patient_ids: list) -> pd.DataFrame:
    """Load receipts_parsed.joblib. If it contains lineitems, compute v3-style pdf features fast."""
    if not os.path.exists(joblib_path):
        print("  receipts_parsed.joblib NOT FOUND.")
        return None

    print("  Loading receipts_parsed.joblib ...")
    data = joblib_load(joblib_path)

    # Helper to normalize patient-level DF
    def _as_patient_df(df: pd.DataFrame) -> pd.DataFrame:
        d = df.copy()
        if "patient_id" not in d.columns:
            # try index
            if d.index.name == "patient_id":
                d = d.reset_index()
            else:
                # attempt first column
                d = d.reset_index().rename(columns={"index":"patient_id"})
        d["patient_id"] = pd.to_numeric(d["patient_id"], errors="coerce").astype("Int64")
        d = d.dropna(subset=["patient_id"]).copy()
        d["patient_id"] = d["patient_id"].astype(int)
        d = d.drop_duplicates("patient_id", keep="last")
        return d

    # 1) direct DataFrame
    if isinstance(data, pd.DataFrame):
        df = data
        # If looks like lineitems
        if any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]) and ("patient_id" in df.columns):
            print(f"  joblib DataFrame looks like lineitems: {df.shape} -> building v3-style pdf features...")
            pdf_df = build_pdf_features_from_lineitems(df)
            return pdf_df
        else:
            df = _as_patient_df(df)
            return df

    # 2) dict
    if isinstance(data, dict):
        # if it contains lineitems_df
        if any(k in data for k in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]):
            for k in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
                if k in data and isinstance(data[k], pd.DataFrame):
                    li = data[k]
                    print(f"  joblib dict contains lineitems_df: {li.shape} -> building v3-style pdf features...")
                    pdf_df = build_pdf_features_from_lineitems(li)
                    return pdf_df
        # else mapping patient_id -> feature dict
        try:
            df = pd.DataFrame.from_dict(data, orient="index")
            df.index.name = "patient_id"
            df = df.reset_index()
            df = _as_patient_df(df)
            return df
        except Exception as e:
            print(f"  Unknown dict joblib structure: {e}")
            return None

    # 3) tuple/list: search for DF
    if isinstance(data, (list, tuple)):
        dfs = [x for x in data if isinstance(x, pd.DataFrame)]
        # prioritize DF with code col
        for df in dfs:
            if ("patient_id" in df.columns) and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
                print(f"  joblib list/tuple contains lineitems DF: {df.shape} -> building v3-style pdf features...")
                pdf_df = build_pdf_features_from_lineitems(df)
                return pdf_df
        for df in dfs:
            if "patient_id" in df.columns:
                df = _as_patient_df(df)
                return df

    print(f"  Unknown joblib format: {type(data)}")
    return None

# -----------------------------
# Admissions features (ablation-validated from v3)
# -----------------------------
def load_admissions_features():
    dfs = []
    for path in [ADM_TRAIN_PATH, ADM_TEST_PATH]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None
    adm = pd.concat(dfs, ignore_index=True)
    for c in ["patient_id","charlson_band","acuity_emergent"]:
        if c not in adm.columns:
            print(f"  Admissions: missing column {c} -> skipping admissions features")
            return None
    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")
    result = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()
    print(f"  Admissions features: {len(result)} patients | cols={list(result.columns[1:])}")
    return result

# -----------------------------
# Feature engineering (match v3)
# -----------------------------
def build_features(df, pdf_df=None, patients_df=None, adm_df=None):
    feat = df.copy()

    chronic_map = {"Pneumonia":0, "DiabetesComp":1, "HF":2}
    feat["chronic_encoded"] = feat["primary_chronic"].map(chronic_map).fillna(-1).astype(float)
    feat["sqrt_prior_cost"] = np.sqrt(pd.to_numeric(feat["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0))
    feat["cost_per_visit"] = (pd.to_numeric(feat["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0) /
                              pd.to_numeric(feat["prior_ed_visits_5y"], errors="coerce").fillna(0.0).clip(lower=1.0))
    feat["log_visits"] = np.log1p(pd.to_numeric(feat["prior_ed_visits_5y"], errors="coerce").fillna(0.0))
    feat["log_prior_cost"] = np.log1p(pd.to_numeric(feat["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0))

    if patients_df is not None:
        p = patients_df.copy()
        p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
        p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)
        ins_map = {"private":2, "public":1, "self_pay":0}
        p["insurance_encoded"] = p["insurance"].map(ins_map).fillna(-1).astype(float)
        # zip_region = first digit of zip3
        zr = p["zip3"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(3).str[0]
        p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)
        feat = feat.merge(
            p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]],
            on="patient_id", how="left"
        )
        feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in adm_df.columns:
            if c != "patient_id" and feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != "patient_id"]
        feat = feat.merge(pdf_df, on="patient_id", how="left")
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    return feat

def get_feature_columns(df):
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3","charlson_band"}
    keep = []
    for c in df.columns:
        if c in exclude: 
            continue
        if df[c].dtype in ("int64","float64","int32","float32","bool","uint8"):
            keep.append(c)
    return keep

# -----------------------------
# Models (match v3 Iter16)
# -----------------------------
CAT_A_PARAMS = dict(
    loss_function="RMSE", eval_metric="MAE",
    iterations=3000, learning_rate=0.03,
    depth=5, l2_leaf_reg=5, min_data_in_leaf=25,
    rsm=0.8,
    verbose=0, early_stopping_rounds=100,
)

CAT_B_PARAMS = dict(
    loss_function="RMSE", eval_metric="MAE",
    iterations=3000, learning_rate=0.03,
    depth=4, l2_leaf_reg=3, min_data_in_leaf=20,
    rsm=0.8,
    verbose=0, early_stopping_rounds=100,
)

CAT_C_PARAMS = dict(
    loss_function="RMSE", eval_metric="MAE",
    iterations=3000, learning_rate=0.03,
    depth=5, l2_leaf_reg=5, min_data_in_leaf=25,
    random_strength=2,
    verbose=0, early_stopping_rounds=100,
)

MODEL_CONFIGS = {
    "cat_A_d5_rsm08": CAT_A_PARAMS,
    "cat_B_d4_rsm08_pruned": CAT_B_PARAMS,
    "cat_C_d5_rs2": CAT_C_PARAMS,
}
MODEL_NAMES = list(MODEL_CONFIGS.keys())

# -----------------------------
# Ensemble training + weight optimization
# -----------------------------
def optimize_weights(oof_dict, y):
    """Optimize convex weights for OOF MAE. Use scipy if available; else random dirichlet."""
    keys = list(oof_dict.keys())
    P = np.vstack([oof_dict[k] for k in keys]).T.astype(float)

    def mae_w(w):
        pred = P.dot(w)
        return float(mean_absolute_error(y, pred))

    n = len(keys)

    if _HAS_SCIPY:
        best = None
        for trial in range(60):
            if trial == 0:
                x0 = np.ones(n) / n
            else:
                x0 = np.random.dirichlet(np.ones(n))
            res = minimize(
                lambda w: mae_w(w),
                x0=x0,
                bounds=[(0,1)]*n,
                constraints={"type":"eq","fun": lambda w: np.sum(w)-1},
                method="SLSQP",
                options={"maxiter": 200, "ftol": 1e-9, "disp": False},
            )
            if best is None or res.fun < best.fun:
                best = res
        w = best.x
        w = np.clip(w, 0, 1)
        w = w / w.sum()
        return dict(zip(keys, w)), mae_w(w)
    else:
        # random search (fast enough)
        rng = np.random.default_rng(SEED)
        best_mae = 1e18
        best_w = None
        for _ in range(25000):
            w = rng.dirichlet(np.ones(n))
            m = mae_w(w)
            if m < best_mae:
                best_mae, best_w = m, w
        return dict(zip(keys, best_w)), best_mae

def run_ensemble(train_df, test_df, feat_cols_full, feat_cols_pruned):
    y = train_df[TARGET].values.astype(float)
    y_max = float(np.max(y))
    n_train, n_test = len(train_df), len(test_df)

    model_feat_cols = {
        "cat_A_d5_rsm08": feat_cols_full,
        "cat_B_d4_rsm08_pruned": feat_cols_pruned,
        "cat_C_d5_rs2": feat_cols_full,
    }

    # strat: chronic + target bin (as v3)
    tmp = train_df[["primary_chronic", TARGET]].copy()
    tmp["cost_bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop")
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"].values)

    all_oof = {k: [] for k in MODEL_NAMES}
    all_test = {k: [] for k in MODEL_NAMES}

    for seed_idx in range(Config.N_SEEDS):
        kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED + seed_idx * 7)
        oof = {k: np.zeros(n_train, dtype=float) for k in MODEL_NAMES}
        tst = {k: np.zeros(n_test, dtype=float) for k in MODEL_NAMES}

        for fold, (ti, vi) in enumerate(kf.split(train_df, strat)):
            s = seed_idx * 100 + fold

            for model_name in MODEL_NAMES:
                params = dict(MODEL_CONFIGS[model_name])
                params["random_seed"] = s
                cols = model_feat_cols[model_name]

                X_tr = train_df[cols].iloc[ti]
                y_tr = y[ti]
                X_vl = train_df[cols].iloc[vi]
                y_vl = y[vi]
                X_te = test_df[cols]

                m = CatBoostRegressor(**params)
                m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl), verbose=0)

                pred_v = m.predict(X_vl)
                pred_t = m.predict(X_te)

                oof[model_name][vi] = pred_v
                tst[model_name] += pred_t / Config.N_FOLDS

                del m
                gc.collect()

        # per-seed MAE
        seed_maes = {k: float(mean_absolute_error(y, oof[k])) for k in MODEL_NAMES}
        print(f"  Seed {seed_idx+1}/{Config.N_SEEDS} OOF MAE: " + " | ".join([f"{k}={seed_maes[k]:.2f}" for k in MODEL_NAMES]))

        for k in MODEL_NAMES:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

    avg_oof = {k: np.mean(np.vstack(all_oof[k]), axis=0) for k in MODEL_NAMES}
    avg_test = {k: np.mean(np.vstack(all_test[k]), axis=0) for k in MODEL_NAMES}

    print("\n" + "="*60)
    print("SEED-AVERAGED OOF MAE:")
    for k in MODEL_NAMES:
        print(f"  {k:24s}: {mean_absolute_error(y, avg_oof[k]):.2f}")

    # optimize ensemble weights
    w_opt, cv_mae = optimize_weights(avg_oof, y)
    print("\nOPTIMIZED ENSEMBLE WEIGHTS:")
    for k in MODEL_NAMES:
        print(f"  {k:24s}: {w_opt[k]:.4f}")
    print(f"\n  OPTIMIZED ENSEMBLE CV MAE: {cv_mae:.2f}")

    oof_final = np.zeros(n_train, dtype=float)
    test_final = np.zeros(n_test, dtype=float)
    for k in MODEL_NAMES:
        oof_final += w_opt[k] * avg_oof[k]
        test_final += w_opt[k] * avg_test[k]

    # clip to avoid outlier blowups (v3)
    test_final = np.clip(test_final, 0.0, y_max * 1.5)

    # feature importance (Model A)
    try:
        m_last = CatBoostRegressor(**CAT_A_PARAMS, random_seed=Config.SEED)
        m_last.fit(train_df[feat_cols_full], y, verbose=0)
        imp = pd.Series(m_last.get_feature_importance(), index=feat_cols_full).sort_values(ascending=False)
        print("\nTOP 15 FEATURE IMPORTANCES (Model A):")
        for fname, val in imp.head(15).items():
            print(f"  {fname:28s}: {val:.2f}")
        del m_last
    except Exception as e:
        print(f"  Feature importance error: {e}")

    # extra diagnostics
    med_resid = float(np.median(y - oof_final))
    print(f"\nEnsemble OOF median residual (y-pred): {med_resid:+.2f}")
    oof_q = {q: float(np.quantile(oof_final, q)) for q in [0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0]}
    print("Ensemble OOF pred quantiles:", oof_q)

    return test_final, float(mean_absolute_error(y, oof_final))

# -----------------------------
# Main
# -----------------------------
for p in [TRAIN_PATH, TEST_PATH, PATIENTS_PATH, ADM_TRAIN_PATH, ADM_TEST_PATH, RECEIPTS_JOBLIB_PATH]:
    if not os.path.exists(p):
        # receipts joblib may exist; if not, still run but warn
        if "receipts_parsed.joblib" in p:
            continue
        raise FileNotFoundError(f"Missing required file: {p}")

print("\n[1/5] Loading data...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)
adm_tr = pd.read_csv(ADM_TRAIN_PATH)
adm_te = pd.read_csv(ADM_TEST_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)
log_shape("admissions_train", adm_tr)
log_shape("admissions_test", adm_te)

print("\n[target stats]")
print(train[TARGET].describe().to_string())

print("\n[2/5] Loading admissions features...")
adm_df = load_admissions_features()

print("\n[3/5] Loading receipt features (joblib preferred)...")
all_pids = sorted(set(train["patient_id"].tolist() + test["patient_id"].tolist()))
pdf_df = load_receipts_joblib(Config.RECEIPTS_JOBLIB, all_pids)

# verify expected columns; if missing, DO NOT auto-parse PDFs unless joblib missing (last resort)
if pdf_df is not None:
    expected_cols = ["n_unique_codes", "has_critical_care", "has_intub_31500"]
    if not all(c in pdf_df.columns for c in expected_cols):
        print("  [warn] joblib-loaded receipt DF lacks expected v3 features.")
        print("         As per prompt, PDF parsing is last resort; will proceed WITHOUT receipts (likely worse).")
        pdf_df = None
else:
    print("  [warn] No receipt features available from joblib. (PDF parsing skipped)")

if pdf_df is not None:
    pdf_cols = [c for c in pdf_df.columns if c != "patient_id"]
    print(f"  Receipt/PDF features count: {len(pdf_cols)}")
    print(f"  Receipt features (first 25 cols): {pdf_cols[:25]}")
    print(f"  Coverage: {len(pdf_df)}/{len(all_pids)} = {100*len(pdf_df)/len(all_pids):.1f}%")

print("\n[4/5] Building features...")
train_feat = build_features(train, pdf_df, patients, adm_df)
test_feat  = build_features(test,  pdf_df, patients, adm_df)

feat_cols_full = get_feature_columns(train_feat)
print(f"  FULL feature count: {len(feat_cols_full)}")
print(f"  FULL features: {feat_cols_full}")

# PRUNED features: base (non-pdf) + 6 high-importance pdf cols
pdf_keep = [c for c in Config.PDF_KEEP_COLS if (c in train_feat.columns)]
pdf_cols_set = set(pdf_df.columns) if (pdf_df is not None) else set()
base_non_pdf = [c for c in feat_cols_full if c not in pdf_cols_set]
feat_cols_pruned = [c for c in (base_non_pdf + pdf_keep) if c in train_feat.columns]

print(f"  PRUNED feature count: {len(feat_cols_pruned)}")
print(f"  PRUNED features: {feat_cols_pruned}")

# Clean numeric features: median fill + numeric conversion (v3 style)
for c in feat_cols_full:
    med = train_feat[c].median() if (c in train_feat.columns and (not train_feat[c].isna().all())) else 0.0
    train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").fillna(med)
    test_feat[c]  = pd.to_numeric(test_feat[c],  errors="coerce").fillna(med)

# quick sanity
miss_train = int(train_feat[feat_cols_full].isna().sum().sum())
miss_test  = int(test_feat[feat_cols_full].isna().sum().sum())
print(f"  Missing cells after fill: train={miss_train} | test={miss_test}")

print("\n[5/5] Training 3 CatBoost RMSE variants (seed-bagged) + optimizing weights...")
test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols_full, feat_cols_pruned)

sub = pd.DataFrame({
    "patient_id": test["patient_id"].values,
    "ed_cost_next3y_usd": np.round(np.clip(test_pred, 0, None), 2)
})
sub = sub[["patient_id","ed_cost_next3y_usd"]]
sub.to_csv(Config.OUTPUT_PATH, index=False)

print("\n" + "="*70)
print("FINAL SANITY CHECKS:")
print(f"  overall CV MAE (ensemble OOF): {cv_mae:.2f}")
print(f"  submission shape:   {sub.shape}")
print(f"  submission columns: {list(sub.columns)}")
print(f"  NaN predictions:    {int(sub['ed_cost_next3y_usd'].isna().sum())}")
print(f"  pred min/median/mean/max: {sub['ed_cost_next3y_usd'].min():.2f} / {sub['ed_cost_next3y_usd'].median():.2f} / {sub['ed_cost_next3y_usd'].mean():.2f} / {sub['ed_cost_next3y_usd'].max():.2f}")
q = {q: float(np.quantile(sub["ed_cost_next3y_usd"].values, q)) for q in [0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0]}
print(f"  pred quantiles: {q}")
print(f"\nSaved submission to: {Config.OUTPUT_PATH}")
print("\nPaste back: (1) leaderboard MAE, (2) printed weights, (3) CV MAE, (4) pred quantiles.")


APPLY PIPELINE (from v3 Iter15/16): low-dim receipt features + CatBoost(RMSE) rsm + pruned model + OOF weights
Expected: fast iteration + avoids overfit from huge CPT pivots.

[1/5] Loading data...
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB
[admissions_train] shape=(5000, 9) | cols=9 | mem=0.57 MB
[admissions_test] shape=(5000, 8) | cols=8 | mem=0.53 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[2/5] Loading admissions features...
  Admissions features: 4000 patients | cols=['charlson_max', 'charlson_mean', 'pct_emergent']

[3/5] Loading receipt features (joblib preferred)...
  Loading receipts_parsed.joblib ...
  joblib dict contains lineitems_df: (27864, 6) -> building v3-style pdf features...
  Receipt/PDF features count: 26
  R

# Iteration 7
- 451.3810

In [16]:
# === ITERATION 7: Anti-overfit++ on the v3(Iter15/16) path (keep it practical, fast, and LB-oriented) ===
# Core ideas:
#   - Still LOW-DIM receipts features (buckets + shares + HHI + EM stats + key procedure flags)
#   - Shallow trees (depth 4-5), strong RSM (0.8), multi-seed bagging
#   - Stronger "generalization" ensemble selection: choose weights by stability across seeds (mean + std), not just best OOF
#   - Add ONE diverse model: MAE-loss on pruned features (metric-aligned but still regularized)
#   - Tiny baseline shrink option (toward baseline_next3y) considered in ensemble search (helps LB sometimes)
# Output: D:\AgentDs\agent_ds_healthcare\submission.csv with correct format.

import os, re, sys, gc, math, warnings, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Paths (must match prompt)
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"  # last resort only (we will NOT parse)
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
TARGET = "ed_cost_next3y_usd"

print("="*90)
print("ITERATION 7 | v3-spirit anti-overfit++: shallow trees + RSM + pruning + multi-seed + STABLE ensemble selection")
print("Goal: improve LB beyond ~452 by reducing generalization gap (avoid overfitting weights/features).")
print("="*90)

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Config (keep fast)
# -----------------------------
class CFG:
    N_FOLDS = 7
    N_SEEDS = 5
    # iterations are upper bounds; early stopping typically stops earlier
    ITERS = 3000
    ES_ROUNDS = 120
    LR = 0.03
    RSM = 0.80
    # ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.03, 0.05, 0.08, 0.10, 0.15]  # baseline blend (small)
    SHIFT_GRID = [0.0, 0.5, 1.0]  # global median shift multiplier
    # small complexity penalties (LB-oriented)
    STD_PEN = 0.20    # penalize variance across seeds (stability)
    LAM_PEN = 8.0     # penalize baseline blend (discourage too large)
    SHIFT_PEN = 0.002 # penalize big shift slightly

# -----------------------------
# Utilities
# -----------------------------
def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    # strip .0
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def safe_num(s, default=0.0):
    try:
        v = float(s)
        if np.isnan(v) or np.isinf(v): return default
        return v
    except Exception:
        return default

# -----------------------------
# Admissions features (keep simple, robust)
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs:
        return None
    adm = pd.concat(dfs, ignore_index=True)
    need = ["patient_id","charlson_band","acuity_emergent"]
    if not all(c in adm.columns for c in need):
        return None
    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")
    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()
    # fill numeric
    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    return out

# -----------------------------
# Low-dim receipts features from parsed lineitems (v3-style, but slightly expanded buckets)
# -----------------------------
def build_pdf_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()

    # locate columns
    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code"]:
        if c in li.columns:
            code_col = c
            break
    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"]:
        if c in li.columns:
            total_col = c
            break
    if code_col is None or total_col is None or "patient_id" not in li.columns:
        raise ValueError("Lineitems DF missing required columns.")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()

    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    li.loc[li["amt"] < 0, "amt"] = 0.0

    # totals
    total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)

    # code numeric where possible
    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")

    # buckets
    is_em = li["code"].isin(["99281","99282","99283","99284","99285"])
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)

    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])  # airway/lines/chest tube/cpr

    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    # per-line shares
    share = (li["amt"] / denom).fillna(0.0)
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")

    # basic counts
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_categories = li.groupby("patient_id").apply(
        lambda g: int(len(set(
            np.select(
                [
                    g["code"].isin(["99281","99282","99283","99284","99285"]),
                    g["code"].isin(["99291","99292"]),
                    g["code"].str.startswith("G037", na=False),
                    g["code"].isin(["31500","36556","32551","36620","92950"]),
                    pd.to_numeric(g["code"].where(g["code"].str.fullmatch(r"\d+"), None), errors="coerce").between(80000, 89999),
                    pd.to_numeric(g["code"].where(g["code"].str.fullmatch(r"\d+"), None), errors="coerce").between(70000, 79999),
                    pd.to_numeric(g["code"].where(g["code"].str.fullmatch(r"\d+"), None), errors="coerce").between(10000, 69999),
                ],
                ["em","critical","obs","high","lab","imaging","procedure"],
                default="other"
            )
        )))
    ).rename("n_categories")

    # EM stats
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    sum_em_level = (em_level * is_em.astype(int)).groupby(li["patient_id"]).sum().rename("sum_em_level")
    avg_em_level = (sum_em_level / n_em_codes.replace(0, np.nan)).fillna(0.0).rename("avg_em_level")
    n_high_em = ((em_level >= 4) & is_em).astype(int).groupby(li["patient_id"]).sum().rename("n_high_em")

    # totals by bucket
    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    img_total = li.loc[is_imaging].groupby("patient_id")["amt"].sum().rename("img_total")
    lab_total = li.loc[is_lab].groupby("patient_id")["amt"].sum().rename("lab_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    # counts by bucket
    n_procedures = is_proc_any.astype(int).groupby(li["patient_id"]).sum().rename("n_procedures")
    n_imaging = is_imaging.astype(int).groupby(li["patient_id"]).sum().rename("n_imaging")
    n_lab = is_lab.astype(int).groupby(li["patient_id"]).sum().rename("n_lab")

    # flags
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_observation = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")

    def has_code(code: str, name: str):
        return (li["code"].eq(code).astype(int).groupby(li["patient_id"]).max()).rename(name)

    has_intub_31500 = has_code("31500","has_intub_31500")
    has_cvc_36556 = has_code("36556","has_cvc_36556")
    has_cpr_92950 = has_code("92950","has_cpr_92950")
    has_artline_36620 = has_code("36620","has_artline_36620")
    has_ct_head_70450 = has_code("70450","has_ct_head_70450")
    has_99285 = has_code("99285","has_99285")
    has_ct_abdpel_74177 = has_code("74177","has_ct_abdpel_74177")
    has_troponin_84484 = has_code("84484","has_troponin_84484")
    has_obs_G0378 = has_code("G0378","has_obs_G0378")

    out = pd.concat([
        n_unique_codes, cost_hhi,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_categories, n_procedures, n_imaging, n_lab,
        total
    ], axis=1).reset_index()

    # merge totals (may be missing)
    for s in [em_total, crit_total, proc_total, img_total, lab_total, high_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")
    for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"]:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    denom2 = out["receipt_total"].replace(0.0, np.nan)

    # pct shares
    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_imaging"] = (out["img_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_lab"] = (out["lab_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    # cost_per_em uses total/ n_em (if none -> total)
    out["cost_per_em"] = np.where(out["n_em_codes"] > 0, out["receipt_total"] / out["n_em_codes"].clip(lower=1), out["receipt_total"])

    # composite high acuity count
    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int)
        + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int)
        + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    # cleanup helper totals
    out.drop(columns=[c for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"] if c in out.columns], inplace=True)

    # fill numeric
    for c in out.columns:
        if c == "patient_id":
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    return out

def load_receipts_joblib(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    data = joblib_load(joblib_path)

    # if dict contains lineitems_df
    if isinstance(data, dict):
        for k in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if k in data and isinstance(data[k], pd.DataFrame):
                li = data[k]
                return build_pdf_features_from_lineitems(li)

        # else: maybe patient_id->dict
        try:
            df = pd.DataFrame.from_dict(data, orient="index")
            df.index.name = "patient_id"
            df = df.reset_index()
            return df
        except Exception:
            return None

    # if direct df
    if isinstance(data, pd.DataFrame):
        df = data
        if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
            return build_pdf_features_from_lineitems(df)
        return df

    # if list/tuple
    if isinstance(data, (list, tuple)):
        dfs = [x for x in data if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
                return build_pdf_features_from_lineitems(df)
        for df in dfs:
            if "patient_id" in df.columns:
                return df

    return None

# -----------------------------
# Feature engineering (numeric-only, v3 style)
# -----------------------------
def build_features(ed_df: pd.DataFrame,
                   patients_df: pd.DataFrame,
                   adm_df: Optional[pd.DataFrame],
                   pdf_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    feat = ed_df.copy()

    # chronic encoding
    chronic_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    # base priors
    feat["prior_ed_visits_5y"] = pd.to_numeric(feat["prior_ed_visits_5y"], errors="coerce").fillna(0.0)
    feat["prior_ed_cost_5y_usd"] = pd.to_numeric(feat["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0)

    # transformations (anti-tail)
    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost_cap20k"] = np.log1p(feat["prior_cost_cap20k"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)

    # baseline for LB-safe blending
    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    # patients encodings
    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)

    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)

    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    feat = feat.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    # admissions aggregates
    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in feat.columns:
                feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(feat[c].median())

    # receipts features
    if pdf_df is not None:
        feat = feat.merge(pdf_df, on="patient_id", how="left")
        for c in pdf_df.columns:
            if c == "patient_id": continue
            feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(feat[c].median())

    # a couple of LIGHT interactions (still low-risk)
    if "pct_cost_critical" in feat.columns:
        feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * feat["pct_cost_critical"]
    if "n_high_acuity_total" in feat.columns:
        feat["logprior_x_highacu"] = feat["log_prior_cost"] * feat["n_high_acuity_total"]

    # derived stable ratios
    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)

    # numeric safety
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in ["patient_id", "primary_chronic", TARGET]:
            continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median())

    return feat

def get_numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3"}
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

# -----------------------------
# Training (multi-seed + fold bagging)
# -----------------------------
def train_models(train_feat: pd.DataFrame, test_feat: pd.DataFrame,
                 feat_full: List[str], feat_pruned: List[str]) -> Tuple[Dict[str, List[np.ndarray]], Dict[str, List[np.ndarray]], Dict[str, List[int]]]:
    y = train_feat[TARGET].values.astype(float)

    # stratify: chronic + target bin (v3)
    tmp = train_feat[["primary_chronic", TARGET]].copy()
    tmp["cost_bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop")
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"].values)

    # 3 models (2 RMSE + 1 MAE)
    # NOTE: keep shallow + RSM + regularization to preserve generalization.
    model_specs = {
        "A_RMSE_full_d5": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.0,
        ),
        "B_RMSE_pruned_d4": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=2.0,
        ),
        "C_MAE_pruned_d4": dict(
            loss_function="MAE", eval_metric="MAE",
            depth=4, l2_leaf_reg=7, min_data_in_leaf=36,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.5,
        ),
    }
    model_featcols = {
        "A_RMSE_full_d5": feat_full,
        "B_RMSE_pruned_d4": feat_pruned,
        "C_MAE_pruned_d4": feat_pruned,
    }

    oof_by_seed = {m: [] for m in model_specs.keys()}
    test_by_seed = {m: [] for m in model_specs.keys()}
    best_iters = {m: [] for m in model_specs.keys()}

    print("\n[training] CatBoost CPU | shallow trees | rsm=0.8 | multi-seed bagging")
    print("Models:", list(model_specs.keys()))
    print(f"Seeds={CFG.N_SEEDS}, Folds={CFG.N_FOLDS}\n")

    for seed_idx in range(CFG.N_SEEDS):
        rs = SEED + seed_idx * 7
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs.keys()}
        test_seed = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs.keys()}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat), 1):
            for mname, params in model_specs.items():
                cols = model_featcols[mname]
                X_tr = train_feat[cols].iloc[ti]
                y_tr = y[ti]
                X_va = train_feat[cols].iloc[vi]
                y_va = y[vi]
                X_te = test_feat[cols]

                cb = CatBoostRegressor(
                    **params,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=rs + fold * 31 + (hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )
                cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
                try:
                    best_iters[mname].append(int(cb.get_best_iteration()))
                except Exception:
                    pass

                pred_va = cb.predict(X_va)
                pred_te = cb.predict(X_te)

                oof_seed[mname][vi] = pred_va
                test_seed[mname] += pred_te / CFG.N_FOLDS

                del cb
                gc.collect()

        # per-seed MAE
        seed_maes = {m: float(mean_absolute_error(y, oof_seed[m])) for m in model_specs.keys()}
        print(f"  Seed {seed_idx+1}/{CFG.N_SEEDS} OOF MAE: " + " | ".join([f"{m}={seed_maes[m]:.2f}" for m in model_specs.keys()]))

        for m in model_specs.keys():
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed[m])

    print("\n[seed-averaged OOF MAE per model]")
    for m in oof_by_seed.keys():
        avg_oof = np.mean(np.vstack(oof_by_seed[m]), axis=0)
        print(f"  {m:18s}: {mean_absolute_error(y, avg_oof):.2f}")

    # print median best iters
    print("\n[median best_iteration per model] (for reference)")
    for m in best_iters.keys():
        if best_iters[m]:
            print(f"  {m:18s}: {int(np.median(best_iters[m]))}")
        else:
            print(f"  {m:18s}: (n/a)")

    return oof_by_seed, test_by_seed, best_iters

# -----------------------------
# Ensemble selection (stability across seeds)
# -----------------------------
def stable_ensemble_search(train_feat: pd.DataFrame,
                           oof_by_seed: Dict[str, List[np.ndarray]],
                           test_by_seed: Dict[str, List[np.ndarray]],
                           baseline_vec: np.ndarray) -> Tuple[np.ndarray, np.ndarray, Dict]:
    y = train_feat[TARGET].values.astype(float)
    model_names = list(oof_by_seed.keys())
    assert len(model_names) == 3, "This search expects exactly 3 models."

    # precompute seed-averaged oof/test per model
    oof_avg = {m: np.mean(np.vstack(oof_by_seed[m]), axis=0) for m in model_names}
    test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in model_names}

    # weights grid
    step = CFG.W_STEP
    grid = np.round(np.arange(0.0, 1.0 + 1e-9, step), 10)

    best = None
    top_list = []

    def eval_combo(wA, wB, wC, lam, shift_mult):
        # per-seed MAE for stability
        maes = []
        for s in range(CFG.N_SEEDS):
            pred = wA*oof_by_seed[model_names[0]][s] + wB*oof_by_seed[model_names[1]][s] + wC*oof_by_seed[model_names[2]][s]
            pred = (1.0-lam)*pred + lam*baseline_vec
            # global shift derived from avg oof (not per seed), scaled
            pred_avg = wA*oof_avg[model_names[0]] + wB*oof_avg[model_names[1]] + wC*oof_avg[model_names[2]]
            pred_avg = (1.0-lam)*pred_avg + lam*baseline_vec
            shift = float(np.median(y - pred_avg)) * shift_mult
            pred = pred + shift
            maes.append(float(mean_absolute_error(y, pred)))
        mean_m = float(np.mean(maes))
        std_m = float(np.std(maes, ddof=0))
        # objective prefers stability + simplicity
        obj = mean_m + CFG.STD_PEN*std_m + CFG.LAM_PEN*lam + CFG.SHIFT_PEN*abs(shift_mult)
        return obj, mean_m, std_m

    # enumerate
    for wA in grid:
        for wB in grid:
            wC = 1.0 - wA - wB
            if wC < -1e-9:
                continue
            wC = float(max(0.0, wC))
            if abs((wA+wB+wC) - 1.0) > 1e-6:
                continue
            for lam in CFG.LAM_GRID:
                for sm in CFG.SHIFT_GRID:
                    obj, mean_m, std_m = eval_combo(wA, wB, wC, lam, sm)
                    rec = (obj, mean_m, std_m, wA, wB, wC, lam, sm)
                    # keep top 12
                    top_list.append(rec)
                    if best is None or obj < best[0]:
                        best = rec

    top_list.sort(key=lambda x: x[0])
    print("\n[ensemble search] Top candidates by robust objective (mean + std + simplicity penalties):")
    for i, rec in enumerate(top_list[:10], 1):
        obj, mean_m, std_m, wA, wB, wC, lam, sm = rec
        print(f"  #{i:02d} obj={obj:.3f} meanMAE={mean_m:.3f} std={std_m:.3f} | w=({wA:.2f},{wB:.2f},{wC:.2f}) | lam={lam:.2f} | shift_mult={sm:.1f}")

    # finalize with best config
    _, mean_m, std_m, wA, wB, wC, lam, sm = best
    mA, mB, mC = model_names

    oof_final = wA*oof_avg[mA] + wB*oof_avg[mB] + wC*oof_avg[mC]
    test_final = wA*test_avg[mA] + wB*test_avg[mB] + wC*test_avg[mC]

    oof_final = (1.0-lam)*oof_final + lam*baseline_vec
    test_final = (1.0-lam)*test_final + lam*(baseline_vec[:len(test_final)]*0 + (baseline_vec.mean()))  # placeholder; will be overwritten outside

    # we need baseline for test separately; will handle outside
    # compute shift from OOF (avg)
    shift = float(np.median(y - oof_final)) * sm
    oof_final = oof_final + shift

    meta = {
        "models_order": model_names,
        "weights": (float(wA), float(wB), float(wC)),
        "lam_baseline": float(lam),
        "shift_mult": float(sm),
        "shift_value": float(shift),
        "oof_mean_mae_across_seeds": float(mean_m),
        "oof_std_mae_across_seeds": float(std_m),
    }
    return oof_final, test_final, meta

# -----------------------------
# Optional small group shift (very conservative)
# -----------------------------
def apply_chronic_group_shift(train_feat: pd.DataFrame, pred_oof: np.ndarray, shrink: float = 0.3) -> Tuple[np.ndarray, Dict]:
    y = train_feat[TARGET].values.astype(float)
    chronic = train_feat["primary_chronic"].astype(str).values
    resid = y - pred_oof
    shifts = {}
    for g in np.unique(chronic):
        med = float(np.median(resid[chronic == g]))
        shifts[g] = shrink * med
    pred2 = pred_oof.copy()
    for g, s in shifts.items():
        pred2[chronic == g] += s
    return pred2, shifts

# -----------------------------
# Main load
# -----------------------------
def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

must_exist(TRAIN_PATH, "TRAIN")
must_exist(TEST_PATH, "TEST")
must_exist(PATIENTS_PATH, "patients")
must_exist(ADM_TRAIN_PATH, "admissions_train")
must_exist(ADM_TEST_PATH, "admissions_test")
# receipts joblib may exist; if missing we will run but warn
if not os.path.exists(RECEIPTS_JOBLIB_PATH):
    print("[warn] receipts_parsed.joblib missing -> receipts features will be empty (likely worse).")

print("\n[load] reading CSVs...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)
adm_tr = pd.read_csv(ADM_TRAIN_PATH)
adm_te = pd.read_csv(ADM_TEST_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)
log_shape("admissions_train", adm_tr)
log_shape("admissions_test", adm_te)

print("\n[target stats]")
print(train[TARGET].describe().to_string())

# ids
for df in [train, test, patients]:
    df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

# admissions
print("\n[admissions] building robust aggregates...")
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
if adm_df is None:
    print("  admissions features: None")
else:
    print(f"  admissions features: {adm_df.shape} | cols={list(adm_df.columns)}")

# receipts
print("\n[receipts] loading receipts_parsed.joblib and building low-dim receipt features...")
pdf_df = None
if os.path.exists(RECEIPTS_JOBLIB_PATH):
    try:
        pdf_df = load_receipts_joblib(RECEIPTS_JOBLIB_PATH)
        if pdf_df is not None:
            # normalize patient_id
            pdf_df["patient_id"] = pd.to_numeric(pdf_df["patient_id"], errors="coerce").astype("Int64")
            pdf_df = pdf_df.dropna(subset=["patient_id"]).copy()
            pdf_df["patient_id"] = pdf_df["patient_id"].astype(int)
            pdf_df = pdf_df.drop_duplicates("patient_id", keep="last")
            print(f"  receipt_feat shape: {pdf_df.shape}")
            print(f"  receipt_feat cols ({len(pdf_df.columns)-1}): {[c for c in pdf_df.columns if c!='patient_id']}")
        else:
            print("  [warn] could not build receipt features from joblib structure.")
    except Exception as e:
        print(f"  [warn] receipts joblib load/build failed: {e}")
        pdf_df = None
else:
    print("  [warn] receipts joblib missing; skipping receipts features.")

# sanity invariant: sum(line totals) == prior_ed_cost_5y_usd (should match); check using available receipt feature proxy if possible
# Our low-dim features do not include sum_items; use n/a.

# build features
print("\n[features] building train/test feature frames...")
train_feat = build_features(train, patients, adm_df, pdf_df)
test_feat  = build_features(test,  patients, adm_df, pdf_df)

# choose features
feat_full = get_numeric_feature_cols(train_feat)
# remove any accidental target leakage
feat_full = [c for c in feat_full if c != TARGET]

# PRUNED set: hand-picked stable features (from your v3/v16 + latest importances)
pruned_candidates = [
    # priors + transforms
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k","cost_per_visit","log_visits",
    "baseline_next3y",
    # demographics
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    # admissions
    "charlson_max","charlson_mean","pct_emergent",
    # receipt robust
    "cost_per_em","cost_hhi","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","n_unique_codes",
    # light interactions
    "logprior_x_pctcritical","logprior_x_highacu",
    # stable ratio
    "cost_per_code",
]
feat_pruned = [c for c in pruned_candidates if c in train_feat.columns]

# median fill + numeric conversion safety for selected columns
for c in set(feat_full + feat_pruned):
    if c in train_feat.columns:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0.0
        train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").fillna(med)
        test_feat[c]  = pd.to_numeric(test_feat[c], errors="coerce").fillna(med)

# drop constant columns (anti-overfit)
def drop_constants(cols: List[str], df: pd.DataFrame) -> List[str]:
    out = []
    for c in cols:
        if c not in df.columns:
            continue
        if df[c].nunique(dropna=False) <= 1:
            continue
        out.append(c)
    return out

feat_full = drop_constants(feat_full, train_feat)
feat_pruned = drop_constants(feat_pruned, train_feat)

print(f"  FULL feature count:   {len(feat_full)}")
print(f"  PRUNED feature count: {len(feat_pruned)}")
print(f"  PRUNED features: {feat_pruned}")

miss_train = int(train_feat[feat_full].isna().sum().sum())
miss_test  = int(test_feat[feat_full].isna().sum().sum())
print(f"  Missing cells after fill (full): train={miss_train} test={miss_test}")

# train
oof_by_seed, test_by_seed, best_iters = train_models(train_feat, test_feat, feat_full, feat_pruned)

# baseline vectors for blending
baseline_oof = train_feat["baseline_next3y"].values.astype(float)
baseline_test = test_feat["baseline_next3y"].values.astype(float)

# ensemble search (stable)
# NOTE: stable_ensemble_search expects baseline_vec length == train length; test baseline handled outside.
oof_ens, test_ens_placeholder, meta = stable_ensemble_search(train_feat, oof_by_seed, test_by_seed, baseline_oof)

# build final test ensemble using chosen weights
mA, mB, mC = meta["models_order"]
wA, wB, wC = meta["weights"]
lam = meta["lam_baseline"]
shift = meta["shift_value"]

test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in meta["models_order"]}
test_ens = wA*test_avg[mA] + wB*test_avg[mB] + wC*test_avg[mC]
test_ens = (1.0-lam)*test_ens + lam*baseline_test
test_ens = test_ens + shift

# optional chronic shift (very conservative; only apply if meaningful OOF gain)
y = train_feat[TARGET].values.astype(float)
base_mae = float(mean_absolute_error(y, oof_ens))
best_oof = oof_ens
best_shift = {"type": "none"}

for shrink in [0.0, 0.3, 0.5]:
    if shrink <= 0:
        continue
    oof2, shifts = apply_chronic_group_shift(train_feat, oof_ens, shrink=shrink)
    m2 = float(mean_absolute_error(y, oof2))
    # require noticeable improvement to accept (avoid LB overfit)
    if m2 < base_mae - 0.15:
        base_mae = m2
        best_oof = oof2
        best_shift = {"type": "chronic_group", "shrink": shrink, "shifts": shifts}

# apply chosen chronic shift to test
if best_shift["type"] == "chronic_group":
    test_chronic = test_feat["primary_chronic"].astype(str).values
    for g, s in best_shift["shifts"].items():
        test_ens[test_chronic == g] += s

# clip predictions to reasonable range (LB-safe; small)
y_max = float(np.max(y))
test_ens = np.clip(test_ens, 0.0, y_max * 1.5)

# feature importance from a full-fit Model A (compliance + insight)
print("\n[full-train] fitting Model A on full train for feature importance...")
A_params = dict(
    loss_function="RMSE", eval_metric="MAE",
    depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
    learning_rate=CFG.LR, iterations=CFG.ITERS,
    rsm=CFG.RSM, random_strength=1.0,
    task_type="CPU", thread_count=-1,
    verbose=0, allow_writing_files=False,
    random_seed=SEED,
)
mA_full = CatBoostRegressor(**A_params)
mA_full.fit(train_feat[feat_full], y, verbose=0)
try:
    imp = mA_full.get_feature_importance()
    imp_df = pd.DataFrame({"feature": feat_full, "importance": imp}).sort_values("importance", ascending=False).head(10)
    print("\n[Top 10 feature importances] (Model A full)")
    print(imp_df.to_string(index=False))
except Exception as e:
    print(f"[warn] feature importance failed: {e}")
del mA_full
gc.collect()

# final logs
final_oof_mae = float(mean_absolute_error(y, best_oof))
print("\n" + "="*70)
print("[FINAL OOF]")
print(f"  ensemble OOF MAE (stable search + optional chronic shift): {final_oof_mae:.2f}")
print("  ensemble meta:", meta)
print("  extra shift:", best_shift["type"], ("shrink="+str(best_shift.get("shrink")) if best_shift["type"]!="none" else ""))
print("  OOF pred quantiles:", qdict(best_oof, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("="*70)

# write submission
sub = pd.DataFrame({
    "patient_id": test["patient_id"].values.astype(int),
    "ed_cost_next3y_usd": np.round(test_ens.astype(float), 2)
})
sub = sub[["patient_id", "ed_cost_next3y_usd"]]

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

# sanity checks
print("\n[SUBMISSION sanity checks]")
print("submission shape:", sub.shape)
print("submission columns exactly:", list(sub.columns))
print("any NaNs in preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(sub["ed_cost_next3y_usd"].min()), float(sub["ed_cost_next3y_usd"].median()), float(sub["ed_cost_next3y_usd"].max()))
print("pred quantiles:", qdict(sub["ed_cost_next3y_usd"].values))
print("\nSaved submission to:", str(out_path))
print("\nPaste back: (1) leaderboard MAE, (2) FINAL OOF MAE, (3) ensemble meta + extra shift, (4) pred quantiles.")


ITERATION 7 | v3-spirit anti-overfit++: shallow trees + RSM + pruning + multi-seed + STABLE ensemble selection
Goal: improve LB beyond ~452 by reducing generalization gap (avoid overfitting weights/features).

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB
[admissions_train] shape=(5000, 9) | cols=9 | mem=0.57 MB
[admissions_test] shape=(5000, 8) | cols=8 | mem=0.53 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[admissions] building robust aggregates...
  admissions features: (4000, 4) | cols=['patient_id', 'charlson_max', 'charlson_mean', 'pct_emergent']

[receipts] loading receipts_parsed.joblib and building low-dim receipt features...
  receipt_feat shape: (4000, 32)
  receipt_feat cols (31): ['n_unique_codes',

# Iteration 8
- 483.7891

In [19]:
# === ITERATION 8: v3-spirit++ (keep low-dim + stronger anti-overfit + add LOG-target model + add line-distribution stats) ===
# What changes vs Iteration 7:
#   1) Receipts: add *line distribution* features (n_line_items, line_max/median/std, top1/top2 share) — low-dim but informative.
#      (You previously saw pdf_line_median/mean/max dominate in earlier attempts; now we add them safely in the low-dim setup.)
#   2) Models: keep 2 RMSE models (full/pruned) + replace the unused MAE model with a LOG-RMSE pruned model (tail-robust).
#   3) Ensemble: still stability-aware across seeds, but also penalize *tail underperformance* via (Top20MAE - OverallMAE).
#   4) Optional severity-bin residual correction (0/1/2) with small shrink, only if it improves OOF.
#
# Output: D:\AgentDs\agent_ds_healthcare\submission.csv (patient_id, ed_cost_next3y_usd)

import os, re, sys, gc, warnings, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Paths (MUST match prompt)
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"  # last resort only (we will NOT parse)
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
TARGET = "ed_cost_next3y_usd"

print("="*100)
print("ITERATION 8 | v3-spirit++: low-dim + line-distribution stats + add LOG-RMSE model + tail-aware stable ensemble")
print("Goal: push LB below ~451 by improving generalization & tail handling without overfitting.")
print("="*100)

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Config
# -----------------------------
class CFG:
    N_FOLDS = 7
    N_SEEDS = 5
    ITERS = 3500
    ES_ROUNDS = 130
    LR = 0.03
    RSM = 0.80

    # ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.03, 0.05, 0.08, 0.10, 0.15]   # baseline shrink/blend
    SHIFT_GRID = [0.0, 0.5, 1.0]                       # global median shift multiplier

    # robust objective weights (LB-oriented)
    STD_PEN = 0.20
    TAIL_PEN = 0.15        # penalize (Top20MAE - OverallMAE)
    LAM_PEN = 2.5          # discourage too much baseline blending
    SHIFT_ABS_PEN = 0.0005 # discourage huge global shift (usually tiny anyway)

# -----------------------------
# Utilities
# -----------------------------
def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

# -----------------------------
# Admissions features
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs:
        return None
    adm = pd.concat(dfs, ignore_index=True)
    need = ["patient_id","charlson_band","acuity_emergent"]
    if not all(c in adm.columns for c in need):
        return None
    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")
    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()
    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    return out

# -----------------------------
# Receipts: low-dim features + NEW line distribution stats (vectorized)
# -----------------------------
def build_pdf_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()

    # locate columns
    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code"]:
        if c in li.columns:
            code_col = c
            break
    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"]:
        if c in li.columns:
            total_col = c
            break
    if code_col is None or total_col is None or "patient_id" not in li.columns:
        raise ValueError("Lineitems DF missing required columns.")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()

    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    li.loc[li["amt"] < 0, "amt"] = 0.0

    # per-patient total + line stats
    g = li.groupby("patient_id")
    receipt_total = g["amt"].sum().rename("receipt_total")
    n_line_items = g.size().rename("n_line_items")
    line_mean = g["amt"].mean().rename("line_mean")
    line_median = g["amt"].median().rename("line_median")
    line_max = g["amt"].max().rename("line_max")
    line_std = g["amt"].std(ddof=0).fillna(0.0).rename("line_std")

    # top1/top2 shares (sorted within patient)
    li_sorted = li.sort_values(["patient_id","amt"], ascending=[True, False])
    top1_amt = li_sorted.groupby("patient_id")["amt"].nth(0).fillna(0.0).rename("line_top1_amt")
    top2_amt = li_sorted.groupby("patient_id")["amt"].nth(1).fillna(0.0).rename("line_top2_amt")

    tmp = pd.concat([receipt_total, n_line_items, line_mean, line_median, line_max, line_std, top1_amt, top2_amt], axis=1).reset_index()
    denom = tmp["receipt_total"].replace(0.0, np.nan)
    tmp["line_top1_share"] = (tmp["line_top1_amt"] / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    tmp["line_top2_share"] = ((tmp["line_top1_amt"] + tmp["line_top2_amt"]) / denom).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    tmp.drop(columns=["line_top1_amt","line_top2_amt","receipt_total"], inplace=True)

    # bucket flags
    code = li["code"]
    code_num = pd.to_numeric(code.where(code.str.fullmatch(r"\d+"), None), errors="coerce")

    is_em = code.isin(["99281","99282","99283","99284","99285"])
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = code.map(em_map).fillna(0).astype(float)

    is_crit = code.isin(["99291","99292"])
    is_obs = code.str.startswith("G037", na=False)
    is_high = code.isin(["31500","36556","32551","36620","92950"])

    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    # per-line shares for HHI
    li = li.join(receipt_total, on="patient_id")
    share = (li["amt"] / li["receipt_total"].replace(0.0, np.nan)).fillna(0.0)
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")

    # counts and EM stats
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    sum_em_level = (em_level * is_em.astype(int)).groupby(li["patient_id"]).sum().rename("sum_em_level")
    avg_em_level = (sum_em_level / n_em_codes.replace(0, np.nan)).fillna(0.0).rename("avg_em_level")
    n_high_em = ((em_level >= 4) & is_em).astype(int).groupby(li["patient_id"]).sum().rename("n_high_em")

    # totals by bucket (for pct shares)
    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    img_total = li.loc[is_imaging].groupby("patient_id")["amt"].sum().rename("img_total")
    lab_total = li.loc[is_lab].groupby("patient_id")["amt"].sum().rename("lab_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    # counts by bucket
    n_procedures = is_proc_any.astype(int).groupby(li["patient_id"]).sum().rename("n_procedures")
    n_imaging = is_imaging.astype(int).groupby(li["patient_id"]).sum().rename("n_imaging")
    n_lab = is_lab.astype(int).groupby(li["patient_id"]).sum().rename("n_lab")

    # flags
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_observation = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")

    def has_code(code_str: str, name: str):
        return (li["code"].eq(code_str).astype(int).groupby(li["patient_id"]).max()).rename(name)

    has_intub_31500 = has_code("31500","has_intub_31500")
    has_cvc_36556 = has_code("36556","has_cvc_36556")
    has_cpr_92950 = has_code("92950","has_cpr_92950")
    has_artline_36620 = has_code("36620","has_artline_36620")
    has_ct_head_70450 = has_code("70450","has_ct_head_70450")
    has_99285 = has_code("99285","has_99285")
    has_ct_abdpel_74177 = has_code("74177","has_ct_abdpel_74177")
    has_troponin_84484 = has_code("84484","has_troponin_84484")
    has_obs_G0378 = has_code("G0378","has_obs_G0378")

    # category count (vectorized)
    cat = np.select(
        [is_em, is_crit, is_obs, is_high, is_lab, is_imaging, (is_proc_general & (~is_high))],
        ["em","critical","obs","high","lab","imaging","procedure"],
        default="other"
    )
    n_categories = pd.Series(cat, index=li.index).groupby(li["patient_id"]).nunique().rename("n_categories")

    base = pd.concat([
        n_unique_codes, cost_hhi,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_categories, n_procedures, n_imaging, n_lab,
    ], axis=1).reset_index()

    # merge totals and compute pct shares
    out = base
    for s in [em_total, crit_total, proc_total, img_total, lab_total, high_total, receipt_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")

    for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    denom2 = out["receipt_total"].replace(0.0, np.nan)
    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_imaging"] = (out["img_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_lab"] = (out["lab_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    out["n_em_codes"] = pd.to_numeric(out["n_em_codes"], errors="coerce").fillna(0.0)
    out["cost_per_em"] = np.where(out["n_em_codes"] > 0, out["receipt_total"] / out["n_em_codes"].clip(lower=1), out["receipt_total"])

    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int)
        + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int)
        + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    # drop helper totals
    out.drop(columns=["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"], inplace=True)

    # merge NEW line distribution stats
    out = out.merge(tmp, on="patient_id", how="left")
    for c in tmp.columns:
        if c == "patient_id": continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(out[c].median())

    # numeric fill
    for c in out.columns:
        if c == "patient_id": continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    return out

def load_receipts_joblib(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    data = joblib_load(joblib_path)

    if isinstance(data, dict):
        for k in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if k in data and isinstance(data[k], pd.DataFrame):
                return build_pdf_features_from_lineitems(data[k])
        # else: try patient_id->dict
        try:
            df = pd.DataFrame.from_dict(data, orient="index")
            df.index.name = "patient_id"
            return df.reset_index()
        except Exception:
            return None

    if isinstance(data, pd.DataFrame):
        df = data
        if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
            return build_pdf_features_from_lineitems(df)
        return df

    if isinstance(data, (list, tuple)):
        dfs = [x for x in data if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
                return build_pdf_features_from_lineitems(df)
        for df in dfs:
            if "patient_id" in df.columns:
                return df

    return None

# -----------------------------
# Feature engineering
# -----------------------------
def build_features(ed_df: pd.DataFrame,
                   patients_df: pd.DataFrame,
                   adm_df: Optional[pd.DataFrame],
                   pdf_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    feat = ed_df.copy()

    # chronic encoding (keep v3-style)
    chronic_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    # priors
    feat["prior_ed_visits_5y"] = pd.to_numeric(feat["prior_ed_visits_5y"], errors="coerce").fillna(0.0)
    feat["prior_ed_cost_5y_usd"] = pd.to_numeric(feat["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0)

    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost_cap20k"] = np.log1p(feat["prior_cost_cap20k"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)

    # baseline anchor
    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    # patient encodings
    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)
    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)
    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    feat = feat.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    # admissions
    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in feat.columns:
                feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(feat[c].median())

    # receipts
    if pdf_df is not None:
        feat = feat.merge(pdf_df, on="patient_id", how="left")
        for c in pdf_df.columns:
            if c == "patient_id": continue
            feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(feat[c].median())

    # light interactions (low-risk)
    if "pct_cost_critical" in feat.columns:
        feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * feat["pct_cost_critical"]
    if "n_high_acuity_total" in feat.columns:
        feat["logprior_x_highacu"] = feat["log_prior_cost"] * feat["n_high_acuity_total"]

    # NEW robust ratios (low-dim)
    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)
        feat["highacu_per_code"] = feat.get("n_high_acuity_total", 0.0) / (feat["n_unique_codes"].clip(lower=1.0))
        feat["em_per_code"] = feat.get("n_em_codes", 0.0) / (feat["n_unique_codes"].clip(lower=1.0))
    if "n_line_items" in feat.columns:
        feat["highacu_per_line"] = feat.get("n_high_acuity_total", 0.0) / feat["n_line_items"].clip(lower=1.0)

    # numeric safety fill
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in ["patient_id","primary_chronic",TARGET,"sex","insurance","zip3"]:
            continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median())

    return feat

def get_numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3"}
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def drop_constants(cols: List[str], df: pd.DataFrame) -> List[str]:
    out = []
    for c in cols:
        if c not in df.columns:
            continue
        if df[c].nunique(dropna=False) <= 1:
            continue
        out.append(c)
    return out

# -----------------------------
# Training: 3 models (2 RMSE + 1 LOG-RMSE)
# -----------------------------
def train_models(train_feat: pd.DataFrame, test_feat: pd.DataFrame,
                 feat_full: List[str], feat_pruned: List[str]) -> Tuple[Dict[str, List[np.ndarray]], Dict[str, List[np.ndarray]], Dict[str, List[int]]]:
    y = train_feat[TARGET].values.astype(float)
    y_log = np.log1p(y)

    # stratify: chronic + target bins
    tmp = train_feat[["primary_chronic", TARGET]].copy()
    tmp["cost_bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop")
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"].values)

    model_specs = {
        # slightly more regularized than Iter7 (LB-oriented)
        "A_RMSE_full_d5": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=5, l2_leaf_reg=6, min_data_in_leaf=34,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.2,
            bootstrap_type="Bayesian", bagging_temperature=1.2,
        ),
        "B_RMSE_pruned_d4": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=4, l2_leaf_reg=6, min_data_in_leaf=42,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=2.0,
            bootstrap_type="Bayesian", bagging_temperature=1.5,
        ),
        # NEW: log-target RMSE (tail robust)
        "C_LOGRMSE_pruned_d4": dict(
            loss_function="RMSE", eval_metric="RMSE",
            depth=4, l2_leaf_reg=5, min_data_in_leaf=42,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.0,
            bootstrap_type="Bayesian", bagging_temperature=1.0,
        ),
    }
    model_featcols = {
        "A_RMSE_full_d5": feat_full,
        "B_RMSE_pruned_d4": feat_pruned,
        "C_LOGRMSE_pruned_d4": feat_pruned,
    }

    oof_by_seed = {m: [] for m in model_specs.keys()}
    test_by_seed = {m: [] for m in model_specs.keys()}
    best_iters = {m: [] for m in model_specs.keys()}

    y_q80 = float(np.quantile(y, 0.80))
    top_mask = (y >= y_q80)

    print("\n[training] CatBoost CPU | shallow trees | strong RSM | 5 seeds x 7 folds")
    print("Models:", list(model_specs.keys()))
    print(f"Top20 threshold (y>=q80): {y_q80:.2f}\n")

    for seed_idx in range(CFG.N_SEEDS):
        rs = SEED + seed_idx * 7
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs.keys()}
        test_seed = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs.keys()}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat), 1):
            for mname, params in model_specs.items():
                cols = model_featcols[mname]

                X_tr = train_feat[cols].iloc[ti]
                X_va = train_feat[cols].iloc[vi]
                X_te = test_feat[cols]

                if mname.startswith("C_LOG"):
                    y_tr = y_log[ti]
                    y_va = y_log[vi]
                else:
                    y_tr = y[ti]
                    y_va = y[vi]

                cb = CatBoostRegressor(
                    **params,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=rs + fold * 31 + (hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )
                cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)

                try:
                    best_iters[mname].append(int(cb.get_best_iteration()))
                except Exception:
                    pass

                pred_va = cb.predict(X_va)
                pred_te = cb.predict(X_te)

                if mname.startswith("C_LOG"):
                    pred_va = np.expm1(pred_va)
                    pred_te = np.expm1(pred_te)

                pred_va = np.clip(pred_va, 0.0, None)
                pred_te = np.clip(pred_te, 0.0, None)

                oof_seed[mname][vi] = pred_va
                test_seed[mname] += pred_te / CFG.N_FOLDS

                del cb
                gc.collect()

        # per-seed logs: overall + top20
        parts = []
        for m in model_specs.keys():
            mae_all = float(mean_absolute_error(y, oof_seed[m]))
            mae_top = float(mean_absolute_error(y[top_mask], oof_seed[m][top_mask]))
            parts.append(f"{m}=all{mae_all:.2f}/top{mae_top:.2f}")
        print(f"  Seed {seed_idx+1}/{CFG.N_SEEDS} OOF: " + " | ".join(parts))

        for m in model_specs.keys():
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed[m])

    print("\n[seed-averaged OOF MAE per model] (overall / top20)")
    for m in model_specs.keys():
        oof_avg = np.mean(np.vstack(oof_by_seed[m]), axis=0)
        mae_all = float(mean_absolute_error(y, oof_avg))
        mae_top = float(mean_absolute_error(y[top_mask], oof_avg[top_mask]))
        print(f"  {m:20s}: all={mae_all:.2f} | top20={mae_top:.2f}")

    print("\n[median best_iteration per model] (reference)")
    for m in model_specs.keys():
        med = int(np.median(best_iters[m])) if best_iters[m] else None
        print(f"  {m:20s}: {med}")

    return oof_by_seed, test_by_seed, best_iters

# -----------------------------
# Ensemble selection (stable + tail-aware)
# -----------------------------
def ensemble_search(train_feat: pd.DataFrame,
                    oof_by_seed: Dict[str, List[np.ndarray]],
                    test_by_seed: Dict[str, List[np.ndarray]],
                    baseline_oof: np.ndarray,
                    baseline_test: np.ndarray) -> Tuple[np.ndarray, np.ndarray, Dict]:

    y = train_feat[TARGET].values.astype(float)
    y_q80 = float(np.quantile(y, 0.80))
    top_mask = (y >= y_q80)

    models = list(oof_by_seed.keys())
    assert len(models) == 3, "This ensemble search assumes 3 models."

    oof_avg = {m: np.mean(np.vstack(oof_by_seed[m]), axis=0) for m in models}
    test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in models}

    grid = np.round(np.arange(0.0, 1.0 + 1e-9, CFG.W_STEP), 10)

    best = None
    records = []

    def eval_combo(wA, wB, wC, lam, shift_mult):
        # avg pred for shift estimation (stable)
        pred_avg = wA*oof_avg[models[0]] + wB*oof_avg[models[1]] + wC*oof_avg[models[2]]
        pred_avg = (1.0-lam)*pred_avg + lam*baseline_oof
        shift_val = float(np.median(y - pred_avg)) * shift_mult

        maes = []
        maes_top = []
        for s in range(CFG.N_SEEDS):
            pred = wA*oof_by_seed[models[0]][s] + wB*oof_by_seed[models[1]][s] + wC*oof_by_seed[models[2]][s]
            pred = (1.0-lam)*pred + lam*baseline_oof
            pred = pred + shift_val
            maes.append(float(mean_absolute_error(y, pred)))
            maes_top.append(float(mean_absolute_error(y[top_mask], pred[top_mask])))

        mean_all = float(np.mean(maes))
        std_all = float(np.std(maes, ddof=0))
        mean_top = float(np.mean(maes_top))

        obj = (
            mean_all
            + CFG.STD_PEN*std_all
            + CFG.TAIL_PEN*max(0.0, (mean_top - mean_all))
            + CFG.LAM_PEN*lam
            + CFG.SHIFT_ABS_PEN*abs(shift_val)
        )
        return obj, mean_all, std_all, mean_top, shift_val

    for wA in grid:
        for wB in grid:
            wC = 1.0 - wA - wB
            if wC < -1e-9:
                continue
            wC = float(max(0.0, wC))
            if abs((wA+wB+wC) - 1.0) > 1e-6:
                continue
            for lam in CFG.LAM_GRID:
                for sm in CFG.SHIFT_GRID:
                    obj, mean_all, std_all, mean_top, shift_val = eval_combo(wA, wB, wC, lam, sm)
                    rec = (obj, mean_all, std_all, mean_top, wA, wB, wC, lam, sm, shift_val)
                    records.append(rec)
                    if best is None or obj < best[0]:
                        best = rec

    records.sort(key=lambda x: x[0])

    print("\n[ensemble search] Top 12 candidates (robust objective):")
    for i, rec in enumerate(records[:12], 1):
        obj, mean_all, std_all, mean_top, wA, wB, wC, lam, sm, shift_val = rec
        print(f"  #{i:02d} obj={obj:.3f} | all={mean_all:.3f} std={std_all:.3f} top20={mean_top:.3f} | "
              f"w=({wA:.2f},{wB:.2f},{wC:.2f}) lam={lam:.2f} shift_mult={sm:.1f} shift={shift_val:+.2f}")

    obj, mean_all, std_all, mean_top, wA, wB, wC, lam, sm, shift_val = best
    meta = {
        "models_order": models,
        "weights": (float(wA), float(wB), float(wC)),
        "lam_baseline": float(lam),
        "shift_mult": float(sm),
        "shift_value": float(shift_val),
        "oof_mean_mae_across_seeds": float(mean_all),
        "oof_std_mae_across_seeds": float(std_all),
        "oof_top20_mae_across_seeds": float(mean_top),
        "top20_threshold": float(y_q80),
    }

    # final OOF avg
    oof_final = wA*oof_avg[models[0]] + wB*oof_avg[models[1]] + wC*oof_avg[models[2]]
    oof_final = (1.0-lam)*oof_final + lam*baseline_oof
    oof_final = oof_final + shift_val

    # final TEST avg
    test_final = wA*test_avg[models[0]] + wB*test_avg[models[1]] + wC*test_avg[models[2]]
    test_final = (1.0-lam)*test_final + lam*baseline_test
    test_final = test_final + shift_val

    return oof_final, test_final, meta

# -----------------------------
# Optional severity-bin correction (0/1/2) using OOF residual medians (shrunk)
# -----------------------------
def severity_bin(train_feat: pd.DataFrame) -> np.ndarray:
    # Safely get 'n_high_acuity_total', defaulting to zeros if missing
    if "n_high_acuity_total" in train_feat.columns:
        hi = pd.to_numeric(train_feat["n_high_acuity_total"], errors="coerce").fillna(0.0).values
    else:
        hi = np.zeros(len(train_feat))

    # Safely get 'has_critical_care', defaulting to zeros if missing
    if "has_critical_care" in train_feat.columns:
        crit = pd.to_numeric(train_feat["has_critical_care"], errors="coerce").fillna(0.0).values
    else:
        crit = np.zeros(len(train_feat))

    # 0: no high acuity and no critical
    # 1: mild (exactly 1 high acuity, no critical)
    # 2: severe (critical OR >=2 high acuity)
    sev = np.where((crit >= 1) | (hi >= 2), 2, np.where(hi >= 1, 1, 0)).astype(int)
    return sev

def apply_bin_shift(y: np.ndarray, pred: np.ndarray, bins: np.ndarray, shrink: float) -> Tuple[np.ndarray, Dict[int, float]]:
    resid = y - pred
    shifts = {}
    for b in [0,1,2]:
        r = resid[bins == b]
        shifts[b] = float(np.median(r)) if len(r) else 0.0
    pred2 = pred.copy()
    for b in [0,1,2]:
        pred2[bins == b] += shrink * shifts[b]
    return pred2, {b: float(shrink*shifts[b]) for b in shifts}

# -----------------------------
# Main
# -----------------------------
def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

must_exist(TRAIN_PATH, "TRAIN")
must_exist(TEST_PATH, "TEST")
must_exist(PATIENTS_PATH, "patients")
must_exist(ADM_TRAIN_PATH, "admissions_train")
must_exist(ADM_TEST_PATH, "admissions_test")
if not os.path.exists(RECEIPTS_JOBLIB_PATH):
    print("[warn] receipts_parsed.joblib missing -> receipts features will be absent (likely worse).")

print("\n[load] reading CSVs...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)
adm_tr = pd.read_csv(ADM_TRAIN_PATH)
adm_te = pd.read_csv(ADM_TEST_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)
log_shape("admissions_train", adm_tr)
log_shape("admissions_test", adm_te)

print("\n[target stats]")
print(train[TARGET].describe().to_string())

# ids
for df in [train, test, patients]:
    df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

# admissions
print("\n[admissions] building aggregates...")
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
print("  admissions features:", None if adm_df is None else adm_df.shape)

# receipts
print("\n[receipts] loading receipts_parsed.joblib -> low-dim + line distribution features...")
pdf_df = None
if os.path.exists(RECEIPTS_JOBLIB_PATH):
    try:
        pdf_df = load_receipts_joblib(RECEIPTS_JOBLIB_PATH)
        if pdf_df is not None:
            pdf_df["patient_id"] = pd.to_numeric(pdf_df["patient_id"], errors="coerce").astype("Int64")
            pdf_df = pdf_df.dropna(subset=["patient_id"]).copy()
            pdf_df["patient_id"] = pdf_df["patient_id"].astype(int)
            pdf_df = pdf_df.drop_duplicates("patient_id", keep="last")
            print(f"  receipt_feat shape: {pdf_df.shape}")
            feat_cols = [c for c in pdf_df.columns if c != "patient_id"]
            print(f"  receipt_feat cols ({len(feat_cols)}): {feat_cols}")
        else:
            print("  [warn] Could not build receipt features from joblib structure.")
    except Exception as e:
        print(f"  [warn] receipts joblib failed: {e}")
        pdf_df = None
else:
    print("  [warn] receipts joblib missing; skipping receipts features.")

# build features
print("\n[features] building model frames...")
train_feat = build_features(train, patients, adm_df, pdf_df)
test_feat  = build_features(test,  patients, adm_df, pdf_df)

feat_full = drop_constants(get_numeric_feature_cols(train_feat), train_feat)

# PRUNED set: stable hand-picked features (plus new line stats)
pruned_candidates = [
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","prior_cost_cap20k","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k","cost_per_visit","log_visits",
    "baseline_next3y",
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    "charlson_max","charlson_mean","pct_emergent",
    "cost_per_em","cost_hhi","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","n_unique_codes",
    "logprior_x_pctcritical","logprior_x_highacu",
    "cost_per_code","highacu_per_code","em_per_code",
    # NEW line distribution stats
    "n_line_items","line_median","line_max","line_std","line_top1_share","line_top2_share",
]
feat_pruned = drop_constants([c for c in pruned_candidates if c in train_feat.columns], train_feat)

# fill numeric for selected cols
for c in set(feat_full + feat_pruned + [TARGET]):
    if c in train_feat.columns:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0.0
        train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").fillna(med)
    if c in test_feat.columns:
        med = train_feat[c].median() if (c in train_feat.columns and not train_feat[c].isna().all()) else 0.0
        test_feat[c] = pd.to_numeric(test_feat[c], errors="coerce").fillna(med)

print(f"  FULL feature count:   {len(feat_full)}")
print(f"  PRUNED feature count: {len(feat_pruned)}")
print(f"  PRUNED features: {feat_pruned}")
print(f"  Missing cells after fill: train={int(train_feat[feat_full].isna().sum().sum())} test={int(test_feat[feat_full].isna().sum().sum())}")

# train CV bagging
oof_by_seed, test_by_seed, best_iters = train_models(train_feat, test_feat, feat_full, feat_pruned)

# baseline vectors
baseline_oof = train_feat["baseline_next3y"].values.astype(float)
baseline_test = test_feat["baseline_next3y"].values.astype(float)

# ensemble
oof_ens, test_ens, meta = ensemble_search(train_feat, oof_by_seed, test_by_seed, baseline_oof, baseline_test)

y = train_feat[TARGET].values.astype(float)
base_mae = float(mean_absolute_error(y, oof_ens))
y_q80 = meta["top20_threshold"]
top_mask = (y >= y_q80)
base_top = float(mean_absolute_error(y[top_mask], oof_ens[top_mask]))

print("\n[ENS result before severity correction]")
print("  meta:", meta)
print(f"  OOF MAE: {base_mae:.3f} | Top20 MAE: {base_top:.3f}")
print("  OOF pred quantiles:", qdict(oof_ens, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))

# severity-bin correction (optional)
bins_tr = severity_bin(train_feat)
bins_te = severity_bin(test_feat)  # same function works (uses available cols)

best_oof = oof_ens.copy()
best_test = test_ens.copy()
best_corr = {"type":"none"}

for shrink in [0.0, 0.20, 0.30, 0.40]:
    if shrink <= 0:
        continue
    oof2, shifts = apply_bin_shift(y, oof_ens, bins_tr, shrink=shrink)
    mae2 = float(mean_absolute_error(y, oof2))
    top2 = float(mean_absolute_error(y[top_mask], oof2[top_mask]))
    # accept only if improves overall by a meaningful margin (avoid LB overfit)
    if mae2 < base_mae - 0.08:
        base_mae, base_top = mae2, top2
        best_oof = oof2
        # apply same shifts to test
        test2 = test_ens.copy()
        for b, s in shifts.items():
            test2[bins_te == b] += s
        best_test = test2
        best_corr = {"type":"severity_bin_shift", "shrink": shrink, "shifts": shifts}

print("\n[FINAL OOF after optional severity correction]")
print(f"  OOF MAE: {base_mae:.3f} | Top20 MAE: {base_top:.3f}")
print("  correction:", best_corr)
print("  OOF pred quantiles:", qdict(best_oof, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))

# clip test predictions
y_max = float(np.max(y))
best_test = np.clip(best_test, 0.0, y_max*1.5)

# -----------------------------
# Full-train fits (compliance) + feature importance
# -----------------------------
print("\n[full-train] fitting final models on FULL train (compliance) ...")
y_log = np.log1p(y)

def fit_full_model(name: str, params: dict, cols: List[str], y_fit: np.ndarray) -> CatBoostRegressor:
    it_med = int(np.median(best_iters[name])) if best_iters.get(name) else 800
    it_use = int(max(300, it_med + 80))
    p = dict(params)
    p.pop("early_stopping_rounds", None)
    p["iterations"] = it_use
    cb = CatBoostRegressor(
        **p,
        task_type="CPU",
        thread_count=-1,
        verbose=0,
        allow_writing_files=False,
        random_seed=SEED + 2025 + (hash(name) % 1000),
    )
    cb.fit(train_feat[cols], y_fit, verbose=0)
    print(f"  fitted {name:20s} | iterations={it_use} | features={len(cols)}")
    return cb

# reconstruct model params used (same as in train_models)
PARAMS = {
    "A_RMSE_full_d5": dict(
        loss_function="RMSE", eval_metric="MAE",
        depth=5, l2_leaf_reg=6, min_data_in_leaf=34,
        learning_rate=CFG.LR, iterations=CFG.ITERS,
        rsm=CFG.RSM, random_strength=1.2,
        bootstrap_type="Bayesian", bagging_temperature=1.2,
    ),
    "B_RMSE_pruned_d4": dict(
        loss_function="RMSE", eval_metric="MAE",
        depth=4, l2_leaf_reg=6, min_data_in_leaf=42,
        learning_rate=CFG.LR, iterations=CFG.ITERS,
        rsm=CFG.RSM, random_strength=2.0,
        bootstrap_type="Bayesian", bagging_temperature=1.5,
    ),
    "C_LOGRMSE_pruned_d4": dict(
        loss_function="RMSE", eval_metric="RMSE",
        depth=4, l2_leaf_reg=5, min_data_in_leaf=42,
        learning_rate=CFG.LR, iterations=CFG.ITERS,
        rsm=CFG.RSM, random_strength=1.0,
        bootstrap_type="Bayesian", bagging_temperature=1.0,
    ),
}
COLS = {"A_RMSE_full_d5": feat_full, "B_RMSE_pruned_d4": feat_pruned, "C_LOGRMSE_pruned_d4": feat_pruned}
YFIT = {"A_RMSE_full_d5": y, "B_RMSE_pruned_d4": y, "C_LOGRMSE_pruned_d4": y_log}

full_models = {}
for m in ["A_RMSE_full_d5", "B_RMSE_pruned_d4", "C_LOGRMSE_pruned_d4"]:
    full_models[m] = fit_full_model(m, PARAMS[m], COLS[m], YFIT[m])

# feature importance from B (pruned) tends to be more stable
try:
    imp = full_models["B_RMSE_pruned_d4"].get_feature_importance()
    imp_df = pd.DataFrame({"feature": feat_pruned, "importance": imp}).sort_values("importance", ascending=False).head(10)
    print("\n[Top 10 feature importances] (B_RMSE_pruned_d4 full)")
    print(imp_df.to_string(index=False))
except Exception as e:
    print(f"[warn] feature importance failed: {e}")

# -----------------------------
# Write submission
# -----------------------------
sub = pd.DataFrame({
    "patient_id": test["patient_id"].values.astype(int),
    "ed_cost_next3y_usd": np.round(best_test.astype(float), 2),
})[["patient_id","ed_cost_next3y_usd"]]

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

# sanity
print("\n[SUBMISSION sanity checks]")
print("overall CV MAE (ensemble OOF, after correction):", f"{base_mae:.3f}")
print("ensemble meta:", meta)
print("correction used:", best_corr)
print("submission shape:", sub.shape)
print("submission columns exactly:", list(sub.columns))
print("any NaNs in preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(sub["ed_cost_next3y_usd"].min()), float(sub["ed_cost_next3y_usd"].median()), float(sub["ed_cost_next3y_usd"].max()))
print("pred quantiles:", qdict(sub["ed_cost_next3y_usd"].values))
print("\nSaved submission to:", str(out_path))
print("\nPaste back: (1) leaderboard MAE, (2) OOF MAE+Top20 MAE, (3) ensemble meta, (4) correction used, (5) pred quantiles.")


ITERATION 8 | v3-spirit++: low-dim + line-distribution stats + add LOG-RMSE model + tail-aware stable ensemble
Goal: push LB below ~451 by improving generalization & tail handling without overfitting.

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB
[admissions_train] shape=(5000, 9) | cols=9 | mem=0.57 MB
[admissions_test] shape=(5000, 8) | cols=8 | mem=0.53 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[admissions] building aggregates...
  admissions features: (4000, 4)

[receipts] loading receipts_parsed.joblib -> low-dim + line distribution features...
  [warn] receipts joblib failed: 'patient_id'

[features] building model frames...
  FULL feature count:   18
  PRUNED feature count: 18
  PRUNED features: ['pri

# Iteration 9 
- 482.6532

In [21]:
# === ITERATION 9: SAFE RECOVER + INCREMENTAL PUSH (stick to Iter7 winning path, add ONLY low-risk receipt line-stats + smoother variant + baseline-binned calibration) ===
# Philosophy: inherit v3 Iter15/16 anti-overfit spirit:
#   - shallow trees (d4-5), strong RSM=0.8, pruning, multi-seed bagging
#   - DO NOT explode feature space; keep receipt features low-dim
#   - add ONLY proven/low-risk signals: line distribution stats (median/max/std/top-share) + a smoother model variant
#   - optional baseline-binned residual calibration (very conservative; only if OOF improves)
#
# Output: D:\AgentDs\agent_ds_healthcare\submission.csv with columns [patient_id, ed_cost_next3y_usd]

import os, re, sys, gc, math, warnings, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Required Windows paths
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"  # last resort only (we will NOT parse)
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
TARGET = "ed_cost_next3y_usd"

print("="*110)
print("ITERATION 9 | SAFE RECOVER + INCREMENTAL PUSH")
print("  - Start from Iter7 (LB~451.38) which is currently best on this path.")
print("  - Add ONLY low-risk receipt line-distribution stats + one smoother model variant.")
print("  - Optional baseline-binned residual calibration (conservative, OOF-validated).")
print("  - Keep: shallow trees, RSM=0.8, pruning, multi-seed bagging, stable ensemble selection.")
print("="*110)

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Config
# -----------------------------
class CFG:
    N_FOLDS = 7
    N_SEEDS = 5

    # CatBoost core (anti-overfit)
    ITERS = 3200
    ES_ROUNDS = 120
    LR = 0.03
    RSM = 0.80

    # ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.03, 0.05, 0.08]        # tiny baseline blend (helps if LB has lower tail)
    SHIFT_GRID = [0.0, 0.5, 1.0]               # global median shift multiplier

    # objective: prefer stability (mean + std)
    STD_PEN = 0.20
    LAM_PEN = 1.00

    # final blend between fold-bag and full-train-bag (for compliance)
    FINAL_ALPHA_FULL = 0.15
    FINAL_FULL_SEEDS = [SEED, SEED + 1000, SEED + 2000]

# -----------------------------
# Helpers
# -----------------------------
def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

# -----------------------------
# Admissions features (stable, low-dim)
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs:
        return None
    adm = pd.concat(dfs, ignore_index=True)
    need = ["patient_id","charlson_band","acuity_emergent"]
    if not all(c in adm.columns for c in need):
        return None

    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")

    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()

    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    out["patient_id"] = pd.to_numeric(out["patient_id"], errors="coerce").astype("Int64")
    out = out.dropna(subset=["patient_id"]).copy()
    out["patient_id"] = out["patient_id"].astype(int)
    return out

# -----------------------------
# Receipts loader (robust) + features (Iter7 + NEW line distribution stats)
# -----------------------------
def _find_lineitems_df(obj) -> Optional[pd.DataFrame]:
    code_cols = ["code","cpt","cpt_code","hcpcs","proc_code"]
    amt_cols = ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"]

    def _looks_like_lineitems(df: pd.DataFrame) -> bool:
        cols = set(df.columns)
        has_code = any(c in cols for c in code_cols)
        has_amt = any(c in cols for c in amt_cols)
        has_pid = ("patient_id" in cols) or (df.index.name == "patient_id")
        return has_code and has_amt and has_pid

    if isinstance(obj, dict):
        # common keys first
        for k in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if k in obj and isinstance(obj[k], pd.DataFrame):
                df = obj[k]
                if _looks_like_lineitems(df):
                    return df
        # any df values
        for v in obj.values():
            if isinstance(v, pd.DataFrame) and _looks_like_lineitems(v):
                return v

    if isinstance(obj, pd.DataFrame):
        if _looks_like_lineitems(obj):
            return obj

    if isinstance(obj, (list, tuple)):
        for v in obj:
            if isinstance(v, pd.DataFrame) and _looks_like_lineitems(v):
                return v

    return None

def build_pdf_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()

    # ensure patient_id is a column
    if "patient_id" not in li.columns and li.index.name == "patient_id":
        li = li.reset_index()

    # locate columns
    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code"]:
        if c in li.columns:
            code_col = c
            break
    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"]:
        if c in li.columns:
            total_col = c
            break
    if code_col is None or total_col is None or "patient_id" not in li.columns:
        raise ValueError("Lineitems DF missing required columns (patient_id/code/amount).")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()

    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    li.loc[li["amt"] < 0, "amt"] = 0.0

    # totals (this should match prior_ed_cost_5y_usd perfectly by your invariant)
    g = li.groupby("patient_id")
    total = g["amt"].sum().rename("pdf_total_line_cost")

    # NEW: line distribution stats (low-dim, historically informative)
    n_line_items = g.size().rename("n_line_items")
    line_mean = g["amt"].mean().rename("line_mean")
    line_median = g["amt"].median().rename("line_median")
    line_max = g["amt"].max().rename("line_max")
    line_std = g["amt"].std(ddof=0).fillna(0.0).rename("line_std")

    li_sorted = li.sort_values(["patient_id","amt"], ascending=[True, False])
    top1_amt = li_sorted.groupby("patient_id")["amt"].nth(0).fillna(0.0).rename("line_top1_amt")
    top2_amt = li_sorted.groupby("patient_id")["amt"].nth(1).fillna(0.0).rename("line_top2_amt")

    # base: HHI on line items
    li = li.join(total, on="patient_id")
    denom = li["pdf_total_line_cost"].replace(0.0, np.nan)
    share = (li["amt"] / denom).fillna(0.0)
    cost_hhi = (share*share).groupby(li["patient_id"]).sum().rename("cost_hhi")

    # buckets/flags
    code_series = li["code"]
    code_num = pd.to_numeric(code_series.where(code_series.str.fullmatch(r"\d+"), None), errors="coerce")

    is_em = code_series.isin(["99281","99282","99283","99284","99285"])
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = code_series.map(em_map).fillna(0).astype(float)

    is_crit = code_series.isin(["99291","99292"])
    is_obs = code_series.str.startswith("G037", na=False)
    is_high = code_series.isin(["31500","36556","32551","36620","92950"])

    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    # category count
    cat = np.select(
        [is_em, is_crit, is_obs, is_high, is_lab, is_imaging, (is_proc_general & (~is_high))],
        ["em","critical","obs","high","lab","imaging","procedure"],
        default="other"
    )
    n_categories = pd.Series(cat, index=li.index).groupby(li["patient_id"]).nunique().rename("n_categories")

    # counts & EM stats
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    sum_em_level = (em_level * is_em.astype(int)).groupby(li["patient_id"]).sum().rename("sum_em_level")
    avg_em_level = (sum_em_level / n_em_codes.replace(0, np.nan)).fillna(0.0).rename("avg_em_level")
    n_high_em = ((em_level >= 4) & is_em).astype(int).groupby(li["patient_id"]).sum().rename("n_high_em")

    # totals by bucket for pct shares
    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    img_total = li.loc[is_imaging].groupby("patient_id")["amt"].sum().rename("img_total")
    lab_total = li.loc[is_lab].groupby("patient_id")["amt"].sum().rename("lab_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    # counts by bucket
    n_procedures = is_proc_any.astype(int).groupby(li["patient_id"]).sum().rename("n_procedures")
    n_imaging = is_imaging.astype(int).groupby(li["patient_id"]).sum().rename("n_imaging")
    n_lab = is_lab.astype(int).groupby(li["patient_id"]).sum().rename("n_lab")

    # flags
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_observation = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")

    def has_code(code: str, name: str):
        return (li["code"].eq(code).astype(int).groupby(li["patient_id"]).max()).rename(name)

    has_intub_31500 = has_code("31500","has_intub_31500")
    has_cvc_36556 = has_code("36556","has_cvc_36556")
    has_cpr_92950 = has_code("92950","has_cpr_92950")
    has_artline_36620 = has_code("36620","has_artline_36620")
    has_ct_head_70450 = has_code("70450","has_ct_head_70450")
    has_99285 = has_code("99285","has_99285")
    has_ct_abdpel_74177 = has_code("74177","has_ct_abdpel_74177")
    has_troponin_84484 = has_code("84484","has_troponin_84484")
    has_obs_G0378 = has_code("G0378","has_obs_G0378")

    base = pd.concat([
        n_unique_codes, cost_hhi,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_categories, n_procedures, n_imaging, n_lab,
        total
    ], axis=1).reset_index()

    # merge bucket totals for pct shares
    out = base
    for s in [em_total, crit_total, proc_total, img_total, lab_total, high_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")

    for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","pdf_total_line_cost"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    denom2 = out["pdf_total_line_cost"].replace(0.0, np.nan)
    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_imaging"] = (out["img_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_lab"] = (out["lab_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    out["cost_per_em"] = np.where(out["n_em_codes"] > 0, out["pdf_total_line_cost"] / out["n_em_codes"].clip(lower=1), out["pdf_total_line_cost"])

    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int)
        + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int)
        + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    # drop helper totals
    out.drop(columns=["em_total","crit_total","proc_total","img_total","lab_total","high_total"], inplace=True)

    # NEW: merge line stats and compute top shares
    line_df = pd.concat([n_line_items, line_mean, line_median, line_max, line_std, top1_amt, top2_amt, total], axis=1).reset_index()
    denom3 = line_df["pdf_total_line_cost"].replace(0.0, np.nan)
    line_df["line_top1_share"] = (line_df["line_top1_amt"] / denom3).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    line_df["line_top2_share"] = ((line_df["line_top1_amt"] + line_df["line_top2_amt"]) / denom3).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    line_df.drop(columns=["line_top1_amt","line_top2_amt","pdf_total_line_cost"], inplace=True)

    out = out.merge(line_df, on="patient_id", how="left")

    # fill numeric
    for c in out.columns:
        if c == "patient_id":
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    return out

def load_receipts_features(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    obj = joblib_load(joblib_path)

    # lineitems path
    li = _find_lineitems_df(obj)
    if li is not None:
        pdf = build_pdf_features_from_lineitems(li)
        pdf["patient_id"] = pd.to_numeric(pdf["patient_id"], errors="coerce").astype("Int64")
        pdf = pdf.dropna(subset=["patient_id"]).copy()
        pdf["patient_id"] = pdf["patient_id"].astype(int)
        pdf = pdf.drop_duplicates("patient_id", keep="last")
        return pdf

    # fallback: maybe patient-level dict
    if isinstance(obj, dict):
        try:
            df = pd.DataFrame.from_dict(obj, orient="index")
            df.index.name = "patient_id"
            df = df.reset_index()
            return df
        except Exception:
            return None

    # fallback: df
    if isinstance(obj, pd.DataFrame):
        df = obj
        if "patient_id" not in df.columns and df.index.name == "patient_id":
            df = df.reset_index()
        return df

    return None

# -----------------------------
# Feature engineering (v3-ish)
# -----------------------------
def build_features(ed_df: pd.DataFrame,
                   patients_df: pd.DataFrame,
                   adm_df: Optional[pd.DataFrame],
                   pdf_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    feat = ed_df.copy()

    # chronic encoding
    chronic_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    # priors
    feat["prior_ed_visits_5y"] = pd.to_numeric(feat["prior_ed_visits_5y"], errors="coerce").fillna(0.0)
    feat["prior_ed_cost_5y_usd"] = pd.to_numeric(feat["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0)

    # transforms
    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)

    # baseline
    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    # patients (encodings)
    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)

    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)

    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    feat = feat.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    # admissions
    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in feat.columns:
                feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(feat[c].median())

    # receipts
    if pdf_df is not None:
        feat = feat.merge(pdf_df, on="patient_id", how="left")
        for c in pdf_df.columns:
            if c == "patient_id": continue
            feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(feat[c].median())

    # interactions (kept small)
    if "pct_cost_critical" in feat.columns:
        feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * feat["pct_cost_critical"]
    if "n_high_acuity_total" in feat.columns:
        feat["logprior_x_highacu"] = feat["log_prior_cost"] * feat["n_high_acuity_total"]
    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)

    # numeric safety
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in ["patient_id","primary_chronic",TARGET,"sex","insurance","zip3"]:
            continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median())

    return feat

def get_numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3"}
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def drop_constants(cols: List[str], df: pd.DataFrame) -> List[str]:
    out = []
    for c in cols:
        if c not in df.columns:
            continue
        if df[c].nunique(dropna=False) <= 1:
            continue
        out.append(c)
    return out

# -----------------------------
# CV training (3 models)
# -----------------------------
def train_cv_models(train_feat: pd.DataFrame, test_feat: pd.DataFrame,
                    feat_full: List[str],
                    feat_pruned_base: List[str],
                    feat_pruned_line: List[str]) -> Tuple[Dict[str, List[np.ndarray]], Dict[str, List[np.ndarray]], Dict[str, List[int]]]:

    y = train_feat[TARGET].values.astype(float)

    # stratify: chronic + target bin
    tmp = train_feat[["primary_chronic", TARGET]].copy()
    tmp["cost_bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop")
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"].values)

    # model specs
    # A: full features (slightly richer)
    # B: pruned base (strong baseline)
    # C: pruned+line, smoother Bayesian bagging (stability)
    model_specs = {
        "A_full_d5": dict(
            loss_function="RMSE", eval_metric="MAE",
            iterations=CFG.ITERS, learning_rate=CFG.LR,
            depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
            rsm=CFG.RSM,
        ),
        "B_pruned_d4": dict(
            loss_function="RMSE", eval_metric="MAE",
            iterations=CFG.ITERS, learning_rate=CFG.LR,
            depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
            rsm=CFG.RSM, random_strength=2.0,
        ),
        "C_pruned_line_smooth": dict(
            loss_function="RMSE", eval_metric="MAE",
            iterations=CFG.ITERS, learning_rate=CFG.LR,
            depth=4, l2_leaf_reg=7, min_data_in_leaf=44,
            rsm=CFG.RSM, random_strength=3.0,
            bootstrap_type="Bayesian", bagging_temperature=1.5,
        ),
    }
    model_cols = {
        "A_full_d5": feat_full,
        "B_pruned_d4": feat_pruned_base,
        "C_pruned_line_smooth": feat_pruned_line,
    }

    oof_by_seed = {m: [] for m in model_specs.keys()}
    test_by_seed = {m: [] for m in model_specs.keys()}
    best_iters = {m: [] for m in model_specs.keys()}

    print("\n[CV TRAINING]")
    print(f"  folds={CFG.N_FOLDS} | seeds={CFG.N_SEEDS} | models={list(model_specs.keys())}")
    print(f"  NOTE: CPU only (CatBoost rsm not supported on GPU for regression).")
    print("-"*110)

    for seed_idx in range(CFG.N_SEEDS):
        rs = SEED + seed_idx * 7
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs.keys()}
        test_seed = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs.keys()}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat), 1):
            fold_maes = {}
            for mname, params in model_specs.items():
                cols = model_cols[mname]
                X_tr = train_feat[cols].iloc[ti]
                y_tr = y[ti]
                X_va = train_feat[cols].iloc[vi]
                y_va = y[vi]
                X_te = test_feat[cols]

                cb = CatBoostRegressor(
                    **params,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=rs + fold * 31 + (hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )
                cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)

                try:
                    best_iters[mname].append(int(cb.get_best_iteration()))
                except Exception:
                    pass

                pred_va = cb.predict(X_va)
                pred_te = cb.predict(X_te)

                pred_va = np.clip(pred_va, 0.0, None)
                pred_te = np.clip(pred_te, 0.0, None)

                oof_seed[mname][vi] = pred_va
                test_seed[mname] += pred_te / CFG.N_FOLDS

                fold_maes[mname] = float(mean_absolute_error(y_va, pred_va))

                del cb
                gc.collect()

            print(f"  seed{seed_idx+1}/{CFG.N_SEEDS} fold{fold}/{CFG.N_FOLDS}: " +
                  " | ".join([f"{m}={fold_maes[m]:.2f}" for m in model_specs.keys()]))

        # seed-level MAE
        seed_maes = {m: float(mean_absolute_error(y, oof_seed[m])) for m in model_specs.keys()}
        print("  --> seed summary MAE: " + " | ".join([f"{m}={seed_maes[m]:.2f}" for m in model_specs.keys()]))
        print("-"*110)

        for m in model_specs.keys():
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed[m])

    print("\n[OOF SUMMARY | seed-averaged]")
    for m in model_specs.keys():
        oof_avg = np.mean(np.vstack(oof_by_seed[m]), axis=0)
        print(f"  {m:22s} MAE={mean_absolute_error(y, oof_avg):.3f} | pred_q={qdict(oof_avg, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0))}")

    print("\n[median best_iteration] (reference)")
    for m in model_specs.keys():
        if best_iters[m]:
            print(f"  {m:22s}: {int(np.median(best_iters[m]))}")
        else:
            print(f"  {m:22s}: (n/a)")

    return oof_by_seed, test_by_seed, best_iters

# -----------------------------
# Stable ensemble weight search (mean + std across seeds)
# -----------------------------
def stable_weight_search(train_feat: pd.DataFrame,
                         oof_by_seed: Dict[str, List[np.ndarray]],
                         test_by_seed: Dict[str, List[np.ndarray]],
                         baseline_oof: np.ndarray,
                         baseline_test: np.ndarray) -> Tuple[np.ndarray, np.ndarray, Dict]:

    y = train_feat[TARGET].values.astype(float)
    model_names = list(oof_by_seed.keys())
    assert len(model_names) == 3, "Expecting exactly 3 models."

    # seed-avg preds
    oof_avg = {m: np.mean(np.vstack(oof_by_seed[m]), axis=0) for m in model_names}
    test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in model_names}

    grid = np.round(np.arange(0.0, 1.0 + 1e-9, CFG.W_STEP), 10)
    best = None
    records = []

    def eval_combo(wA, wB, wC, lam, shift_mult):
        # compute stable shift from avg pred
        pred_avg = wA*oof_avg[model_names[0]] + wB*oof_avg[model_names[1]] + wC*oof_avg[model_names[2]]
        pred_avg = (1.0-lam)*pred_avg + lam*baseline_oof
        shift_val = float(np.median(y - pred_avg)) * shift_mult

        maes = []
        for s in range(CFG.N_SEEDS):
            pred = wA*oof_by_seed[model_names[0]][s] + wB*oof_by_seed[model_names[1]][s] + wC*oof_by_seed[model_names[2]][s]
            pred = (1.0-lam)*pred + lam*baseline_oof
            pred = pred + shift_val
            maes.append(float(mean_absolute_error(y, pred)))

        mean_m = float(np.mean(maes))
        std_m = float(np.std(maes, ddof=0))
        obj = mean_m + CFG.STD_PEN*std_m + CFG.LAM_PEN*lam
        return obj, mean_m, std_m, shift_val

    for wA in grid:
        for wB in grid:
            wC = 1.0 - wA - wB
            if wC < -1e-9:
                continue
            wC = float(max(0.0, wC))
            if abs((wA+wB+wC) - 1.0) > 1e-6:
                continue
            for lam in CFG.LAM_GRID:
                for sm in CFG.SHIFT_GRID:
                    obj, mean_m, std_m, shift_val = eval_combo(wA, wB, wC, lam, sm)
                    rec = (obj, mean_m, std_m, wA, wB, wC, lam, sm, shift_val)
                    records.append(rec)
                    if best is None or obj < best[0]:
                        best = rec

    records.sort(key=lambda x: x[0])

    print("\n[ENSEMBLE SEARCH] Top 12 (robust objective = meanMAE + 0.2*std + lam_pen*lam)")
    for i, rec in enumerate(records[:12], 1):
        obj, mean_m, std_m, wA, wB, wC, lam, sm, shift_val = rec
        print(f"  #{i:02d} obj={obj:.3f} mean={mean_m:.3f} std={std_m:.3f} | w=({wA:.2f},{wB:.2f},{wC:.2f}) lam={lam:.2f} shift_mult={sm:.1f} shift={shift_val:+.2f}")

    obj, mean_m, std_m, wA, wB, wC, lam, sm, shift_val = best
    meta = {
        "models_order": model_names,
        "weights": (float(wA), float(wB), float(wC)),
        "lam_baseline": float(lam),
        "shift_mult": float(sm),
        "shift_value": float(shift_val),
        "oof_mean_mae_across_seeds": float(mean_m),
        "oof_std_mae_across_seeds": float(std_m),
    }

    # final OOF avg
    oof_final = wA*oof_avg[model_names[0]] + wB*oof_avg[model_names[1]] + wC*oof_avg[model_names[2]]
    oof_final = (1.0-lam)*oof_final + lam*baseline_oof
    oof_final = oof_final + shift_val

    # final TEST avg
    test_final = wA*test_avg[model_names[0]] + wB*test_avg[model_names[1]] + wC*test_avg[model_names[2]]
    test_final = (1.0-lam)*test_final + lam*baseline_test
    test_final = test_final + shift_val

    return oof_final, test_final, meta

# -----------------------------
# Baseline-binned residual calibration (conservative)
# -----------------------------
def baseline_binned_calibration(y: np.ndarray,
                                pred_oof: np.ndarray,
                                baseline_oof: np.ndarray,
                                pred_test: np.ndarray,
                                baseline_test: np.ndarray,
                                n_bins: int = 10,
                                shrink_grid = (0.0, 0.15, 0.25, 0.35)) -> Tuple[np.ndarray, np.ndarray, Dict]:

    # bin edges on baseline (stable across train/test)
    qs = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(baseline_oof, qs)
    edges = np.unique(edges)
    if len(edges) <= 2:
        # cannot bin
        meta = {"used": False, "reason": "baseline edges not unique"}
        return pred_oof, pred_test, meta

    # digitize bins
    def assign_bins(x):
        b = np.digitize(x, edges[1:-1], right=True)
        return b  # 0..(nbins-1)

    bins_tr = assign_bins(baseline_oof)
    bins_te = assign_bins(baseline_test)

    resid = y - pred_oof
    bin_shift = {}
    for b in range(int(bins_tr.max()) + 1):
        r = resid[bins_tr == b]
        bin_shift[b] = float(np.median(r)) if len(r) else 0.0

    # evaluate shrink
    best = None
    for shrink in shrink_grid:
        adj_oof = pred_oof.copy()
        for b, s in bin_shift.items():
            adj_oof[bins_tr == b] += shrink * s
        mae = float(mean_absolute_error(y, adj_oof))
        rec = (mae, shrink)
        if best is None or mae < best[0]:
            best = rec

    best_mae, best_shrink = best

    # only apply if improves meaningfully
    base_mae = float(mean_absolute_error(y, pred_oof))
    improve = base_mae - best_mae

    meta = {
        "used": bool(improve > 0.08 and best_shrink > 0),
        "base_mae": float(base_mae),
        "best_mae": float(best_mae),
        "improve": float(improve),
        "best_shrink": float(best_shrink),
        "n_bins": int(len(edges) - 1),
        "edges_head": [float(x) for x in edges[:min(5, len(edges))]],
        "edges_tail": [float(x) for x in edges[max(0, len(edges)-5):]],
        "bin_shift_minmax": (float(min(bin_shift.values())), float(max(bin_shift.values()))),
    }

    if not meta["used"]:
        return pred_oof, pred_test, meta

    # apply
    adj_oof = pred_oof.copy()
    for b, s in bin_shift.items():
        adj_oof[bins_tr == b] += best_shrink * s

    adj_test = pred_test.copy()
    for b, s in bin_shift.items():
        adj_test[bins_te == b] += best_shrink * s

    # clip to >=0
    adj_oof = np.clip(adj_oof, 0.0, None)
    adj_test = np.clip(adj_test, 0.0, None)

    meta["applied_bin_shift_sample"] = {int(k): float(best_shrink*v) for k,v in list(bin_shift.items())[:min(5, len(bin_shift))]}
    return adj_oof, adj_test, meta

# -----------------------------
# Full-train fits (for compliance) and prediction bagging
# -----------------------------
def fit_full_bag_predictions(train_feat: pd.DataFrame, test_feat: pd.DataFrame,
                             feat_full: List[str], feat_pruned_base: List[str], feat_pruned_line: List[str],
                             best_iters: Dict[str, List[int]]) -> Dict[str, np.ndarray]:

    y = train_feat[TARGET].values.astype(float)

    # model specs must match CV
    model_specs = {
        "A_full_d5": dict(
            loss_function="RMSE", eval_metric="MAE",
            learning_rate=CFG.LR, depth=5, l2_leaf_reg=5, min_data_in_leaf=28, rsm=CFG.RSM,
        ),
        "B_pruned_d4": dict(
            loss_function="RMSE", eval_metric="MAE",
            learning_rate=CFG.LR, depth=4, l2_leaf_reg=4, min_data_in_leaf=32, rsm=CFG.RSM, random_strength=2.0,
        ),
        "C_pruned_line_smooth": dict(
            loss_function="RMSE", eval_metric="MAE",
            learning_rate=CFG.LR, depth=4, l2_leaf_reg=7, min_data_in_leaf=44, rsm=CFG.RSM,
            random_strength=3.0, bootstrap_type="Bayesian", bagging_temperature=1.5,
        ),
    }
    model_cols = {
        "A_full_d5": feat_full,
        "B_pruned_d4": feat_pruned_base,
        "C_pruned_line_smooth": feat_pruned_line,
    }

    preds = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs.keys()}

    print("\n[FULL-TRAIN BAGGING] (for compliance + small blend)")
    print(f"  full_seeds={CFG.FINAL_FULL_SEEDS} | alpha_full={CFG.FINAL_ALPHA_FULL}")
    for m in model_specs.keys():
        med_iter = int(np.median(best_iters[m])) if best_iters.get(m) and best_iters[m] else 700
        it_use = int(max(350, med_iter + 60))
        print(f"  {m:22s}: median_best_iter={med_iter} -> full_fit_iters={it_use}")

        for s in CFG.FINAL_FULL_SEEDS:
            cb = CatBoostRegressor(
                **model_specs[m],
                iterations=it_use,
                task_type="CPU",
                thread_count=-1,
                verbose=0,
                allow_writing_files=False,
                random_seed=int(s + (hash(m) % 997)),
            )
            cb.fit(train_feat[model_cols[m]], y, verbose=0)
            preds[m] += np.clip(cb.predict(test_feat[model_cols[m]]), 0.0, None) / len(CFG.FINAL_FULL_SEEDS)
            del cb
            gc.collect()

    return preds

# -----------------------------
# MAIN
# -----------------------------
must_exist(TRAIN_PATH, "ed_cost_train.csv")
must_exist(TEST_PATH, "ed_cost_test.csv")
must_exist(PATIENTS_PATH, "patients.csv")
must_exist(ADM_TRAIN_PATH, "admissions_train.csv")
must_exist(ADM_TEST_PATH, "admissions_test.csv")
if not os.path.exists(RECEIPTS_JOBLIB_PATH):
    print("[warn] receipts_parsed.joblib not found -> receipts features missing -> performance will degrade.")

print("\n[LOAD DATA]")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)
adm_tr = pd.read_csv(ADM_TRAIN_PATH)
adm_te = pd.read_csv(ADM_TEST_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)
log_shape("admissions_train", adm_tr)
log_shape("admissions_test", adm_te)

print("\n[target stats]")
print(train[TARGET].describe().to_string())

# ensure patient_id int
for df in [train, test, patients]:
    df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

# admissions agg
print("\n[ADMISSIONS FEATURES]")
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
print("  admissions_feat:", None if adm_df is None else (adm_df.shape, list(adm_df.columns)))

# receipts features
print("\n[RECEIPTS FEATURES]")
pdf_df = None
if os.path.exists(RECEIPTS_JOBLIB_PATH):
    try:
        pdf_df = load_receipts_features(RECEIPTS_JOBLIB_PATH)
        if pdf_df is None:
            print("  [warn] receipts_parsed.joblib loaded but could not extract lineitems/patient features.")
        else:
            feat_cols = [c for c in pdf_df.columns if c != "patient_id"]
            print(f"  receipt_feat shape: {pdf_df.shape} | n_feat={len(feat_cols)}")
            print(f"  receipt_feat cols: {feat_cols}")
            # coverage sanity
            all_pids = set(train["patient_id"].tolist() + test["patient_id"].tolist())
            cov = pdf_df["patient_id"].nunique()
            print(f"  coverage: {cov}/{len(all_pids)} = {cov/len(all_pids):.3f}")
    except Exception as e:
        print(f"  [warn] receipts feature build failed: {repr(e)}")
        pdf_df = None
else:
    print("  [warn] receipts joblib missing; skipping receipts.")

# build features
print("\n[BUILD FEATURES]")
train_feat = build_features(train, patients, adm_df, pdf_df)
test_feat  = build_features(test,  patients, adm_df, pdf_df)

# invariant check (if receipts present)
if pdf_df is not None and "pdf_total_line_cost" in train_feat.columns:
    diff = (train_feat["pdf_total_line_cost"] - train_feat["prior_ed_cost_5y_usd"]).abs()
    print(f"[sanity] |pdf_total_line_cost - prior_ed_cost_5y_usd| mean={diff.mean():.3e} max={diff.max():.3e}")

# feature sets
feat_full = drop_constants(get_numeric_feature_cols(train_feat), train_feat)

# Iter7 pruned base (unchanged, stable)
pruned_base = [
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","chronic_encoded","sqrt_prior_cost","cost_per_visit","log_visits","log_prior_cost",
    "age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    "charlson_max","charlson_mean","pct_emergent",
    # receipts stable
    "cost_per_em","cost_hhi","pct_cost_procedure","n_high_acuity_total","pct_cost_critical","pct_cost_em",
    # a bit more (Iter7)
    "pct_cost_high_acuity","pct_cost_imaging","pct_cost_lab","n_unique_codes","max_em_level","n_em_codes",
    # interactions/ratios
    "logprior_x_pctcritical","logprior_x_highacu","cost_per_code",
    # duplicated strong cost feature (helps under rsm)
    "pdf_total_line_cost",
    # baseline anchor
    "baseline_next3y",
]
feat_pruned_base = drop_constants([c for c in pruned_base if c in train_feat.columns], train_feat)

# NEW: pruned_line = base + low-risk line stats (only if receipts present)
line_stats = ["n_line_items","line_median","line_max","line_std","line_top1_share","line_top2_share"]
feat_pruned_line = drop_constants([c for c in (feat_pruned_base + [c for c in line_stats if c in train_feat.columns]) if c in train_feat.columns], train_feat)

print(f"  FULL feature count:   {len(feat_full)}")
print(f"  PRUNED_BASE count:    {len(feat_pruned_base)}")
print(f"  PRUNED_LINE count:    {len(feat_pruned_line)}")
print(f"  PRUNED_LINE added:    {[c for c in line_stats if c in feat_pruned_line and c not in feat_pruned_base]}")

# fill numeric safety
for c in set(feat_full + feat_pruned_line + feat_pruned_base + [TARGET]):
    if c in train_feat.columns:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0.0
        train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").fillna(med)
    if c in test_feat.columns:
        med = train_feat[c].median() if (c in train_feat.columns and not train_feat[c].isna().all()) else 0.0
        test_feat[c] = pd.to_numeric(test_feat[c], errors="coerce").fillna(med)

print(f"  Missing cells after fill (full): train={int(train_feat[feat_full].isna().sum().sum())} test={int(test_feat[feat_full].isna().sum().sum())}")

# CV train
oof_by_seed, test_by_seed, best_iters = train_cv_models(train_feat, test_feat, feat_full, feat_pruned_base, feat_pruned_line)

# stable ensemble
y = train_feat[TARGET].values.astype(float)
baseline_oof = train_feat["baseline_next3y"].values.astype(float)
baseline_test = test_feat["baseline_next3y"].values.astype(float)

oof_ens, test_ens, ens_meta = stable_weight_search(train_feat, oof_by_seed, test_by_seed, baseline_oof, baseline_test)

oof_mae = float(mean_absolute_error(y, oof_ens))
print("\n[ENSEMBLE RESULT] (before calibration)")
print("  meta:", ens_meta)
print(f"  OOF MAE: {oof_mae:.3f}")
print("  OOF pred quantiles:", qdict(oof_ens, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))

# baseline-binned calibration (conservative)
oof_cal, test_cal, cal_meta = baseline_binned_calibration(
    y=y,
    pred_oof=oof_ens,
    baseline_oof=baseline_oof,
    pred_test=test_ens,
    baseline_test=baseline_test,
    n_bins=10,
    shrink_grid=(0.0, 0.15, 0.25, 0.35)
)
oof_cal_mae = float(mean_absolute_error(y, oof_cal))

print("\n[CALIBRATION] baseline-binned residual correction")
print("  cal_meta:", cal_meta)
print(f"  OOF MAE before={oof_mae:.3f} after={oof_cal_mae:.3f}")
print("  OOF pred quantiles (after):", qdict(oof_cal, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))

# choose calibrated if used, else original
use_cal = bool(cal_meta.get("used", False))
final_oof = oof_cal if use_cal else oof_ens
final_test_foldbag = test_cal if use_cal else test_ens
final_oof_mae = float(mean_absolute_error(y, final_oof))

# full-train fits for compliance + small blend
fullbag_preds = fit_full_bag_predictions(train_feat, test_feat, feat_full, feat_pruned_base, feat_pruned_line, best_iters)

# ensemble fullbag using same weights/meta
mA, mB, mC = ens_meta["models_order"]
wA, wB, wC = ens_meta["weights"]
lam = ens_meta["lam_baseline"]
shift_val = ens_meta["shift_value"]

test_fullbag = wA*fullbag_preds[mA] + wB*fullbag_preds[mB] + wC*fullbag_preds[mC]
test_fullbag = (1.0-lam)*test_fullbag + lam*baseline_test
test_fullbag = test_fullbag + shift_val

# apply same calibration to fullbag if used
if use_cal:
    # re-apply calibration using same baseline-binned shifts (recompute using OOF bin shifts from cal_meta is hard; simplest: redo using OOF mapping with chosen shrink)
    # To keep consistent, we reuse the same function but FORCE use (since it already found best_shrink and used=True)
    _, test_fullbag_cal, _ = baseline_binned_calibration(
        y=y,
        pred_oof=oof_ens,  # only used to derive shifts
        baseline_oof=baseline_oof,
        pred_test=test_fullbag,
        baseline_test=baseline_test,
        n_bins=10,
        shrink_grid=(cal_meta.get("best_shrink", 0.0),)
    )
    test_fullbag = test_fullbag_cal

# final blend (fold-bag is primary; full-bag is small alpha)
alpha = float(CFG.FINAL_ALPHA_FULL)
final_test = (1.0 - alpha) * final_test_foldbag + alpha * test_fullbag

# clip to sane max
y_max = float(np.max(y))
final_test = np.clip(final_test, 0.0, y_max * 1.5)

# feature importance (fit B on full train for interpretability)
print("\n[FEATURE IMPORTANCE] fitting B_pruned_d4 on full train for top-10 importances...")
B_params = dict(
    loss_function="RMSE", eval_metric="MAE",
    iterations=900, learning_rate=CFG.LR,
    depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
    rsm=CFG.RSM, random_strength=2.0,
)
try:
    cb_imp = CatBoostRegressor(
        **B_params, task_type="CPU", thread_count=-1, verbose=0, allow_writing_files=False, random_seed=SEED+999
    )
    cb_imp.fit(train_feat[feat_pruned_base], y, verbose=0)
    imp = cb_imp.get_feature_importance()
    imp_df = pd.DataFrame({"feature": feat_pruned_base, "importance": imp}).sort_values("importance", ascending=False).head(10)
    print(imp_df.to_string(index=False))
    del cb_imp
except Exception as e:
    print("  [warn] importance failed:", repr(e))
gc.collect()

# write submission
sub = pd.DataFrame({
    "patient_id": test["patient_id"].values.astype(int),
    "ed_cost_next3y_usd": np.round(final_test.astype(float), 2),
})[["patient_id","ed_cost_next3y_usd"]]

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

# final logs / sanity
print("\n" + "="*80)
print("[FINAL SUMMARY]")
print(f"  OOF MAE (final used for calibration selection): {final_oof_mae:.3f}")
print(f"  ensemble meta: {ens_meta}")
print(f"  calibration_used: {use_cal} | cal_meta: {cal_meta}")
print(f"  final blend: foldbag*(1-{alpha:.2f}) + fullbag*({alpha:.2f})")
print(f"  submission shape: {sub.shape}")
print(f"  submission columns exactly: {list(sub.columns)}")
print(f"  any NaNs in preds: {bool(np.isnan(sub['ed_cost_next3y_usd']).any())}")
print(f"  pred min/median/max: {float(sub['ed_cost_next3y_usd'].min()):.2f} / {float(sub['ed_cost_next3y_usd'].median()):.2f} / {float(sub['ed_cost_next3y_usd'].max()):.2f}")
print(f"  pred quantiles: {qdict(sub['ed_cost_next3y_usd'].values)}")
print("\nSaved submission to:", str(out_path))
print("\nPaste back: (1) leaderboard MAE, (2) FINAL OOF MAE, (3) ensemble meta, (4) calibration_used + cal_meta, (5) pred quantiles.")


ITERATION 9 | SAFE RECOVER + INCREMENTAL PUSH
  - Start from Iter7 (LB~451.38) which is currently best on this path.
  - Add ONLY low-risk receipt line-distribution stats + one smoother model variant.
  - Optional baseline-binned residual calibration (conservative, OOF-validated).
  - Keep: shallow trees, RSM=0.8, pruning, multi-seed bagging, stable ensemble selection.

[LOAD DATA]
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB
[admissions_train] shape=(5000, 9) | cols=9 | mem=0.57 MB
[admissions_test] shape=(5000, 8) | cols=8 | mem=0.53 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[ADMISSIONS FEATURES]
  admissions_feat: ((4000, 4), ['patient_id', 'charlson_max', 'charlson_mean', 'pct_emergent'])

[RECEIPTS FEATURES]
  [warn] receipts

# New Iter

# Submission

In [22]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 2, "D:/AgentDs/agent_ds_healthcare/submission.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 3!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 482.6532 (MAE)
✅ Validation passed
✅ Submission successful!
   📊 Score: 482.6532
   📏 Metric: MAE
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. Move on to Healthcare Challenge 3!
